In [1]:
# ============================================================
# STEP 1: ENVIRONMENT + HAM10000 DATASET VERIFICATION
# ============================================================

import os
import sys
import glob
import random
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("Python version :", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print("CUDA version   :", torch.version.cuda)

print("\n" + "=" * 70)
print("KAGGLE INPUT DATASETS")
print("=" * 70)

for path in glob.glob("/kaggle/input/*"):
    print(path)

print("\n" + "=" * 70)
print("SEARCHING FOR HAM10000")
print("=" * 70)

ham_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            ham_candidates.append(os.path.join(root, file))

for f in ham_candidates:
    print(f)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

ENVIRONMENT CHECK
Python version : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch version: 2.11.0+cu128
CUDA available : True
GPU            : Tesla T4
CUDA version   : 12.8

KAGGLE INPUT DATASETS
/kaggle/input/datasets

SEARCHING FOR HAM10000
/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/hmnist_8_8_RGB.csv
/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/hmnist_28_28_RGB.csv
/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/hmnist_8_8_L.csv
/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/hmnist_28_28_L.csv
/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv

DONE


In [2]:
# ============================================================
# STEP 2: LOAD + AUDIT HAM10000
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

HAM_ROOT = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

METADATA_PATH = os.path.join(
    HAM_ROOT,
    "HAM10000_metadata.csv"
)

print("=" * 70)
print("HAM10000 PATH CHECK")
print("=" * 70)

print("HAM root exists :", os.path.exists(HAM_ROOT))
print("Metadata exists :", os.path.exists(METADATA_PATH))
print("Metadata path   :", METADATA_PATH)


# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

df = pd.read_csv(METADATA_PATH)

print("\n" + "=" * 70)
print("METADATA")
print("=" * 70)

print("Number of rows :", len(df))
print("Number of cols :", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ------------------------------------------------------------
# 3. Class distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

class_counts = df["dx"].value_counts()

print(class_counts)

print("\nClass proportions:")
print((class_counts / len(df)).round(4))


# ------------------------------------------------------------
# 4. Lesion statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LESION STATISTICS")
print("=" * 70)

print("Unique image IDs :", df["image_id"].nunique())
print("Unique lesions  :", df["lesion_id"].nunique())

images_per_lesion = df.groupby("lesion_id").size()

print("\nImages per lesion:")
print(images_per_lesion.value_counts().sort_index())

print("\nSingle-image lesions:",
      (images_per_lesion == 1).sum())

print("Multi-image lesions:",
      (images_per_lesion > 1).sum())


# ------------------------------------------------------------
# 5. Check whether a lesion has multiple diagnosis labels
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LESION → DIAGNOSIS CONSISTENCY")
print("=" * 70)

dx_per_lesion = df.groupby("lesion_id")["dx"].nunique()

inconsistent_lesions = (dx_per_lesion > 1).sum()

print("Lesions with exactly one diagnosis:",
      (dx_per_lesion == 1).sum())

print("Lesions with multiple diagnoses:",
      inconsistent_lesions)

if inconsistent_lesions > 0:
    print("\nWARNING: Some lesions have multiple diagnosis labels.")
    print(
        df[df["lesion_id"].isin(
            dx_per_lesion[dx_per_lesion > 1].index
        )][["lesion_id", "dx"]].drop_duplicates().head(20)
    )
else:
    print("✓ Diagnosis is consistent within every lesion.")


# ------------------------------------------------------------
# 6. Age / sensitive attribute audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGE / SENSITIVE ATTRIBUTE")
print("=" * 70)

print("Missing age values:", df["age"].isna().sum())

print("\nAge statistics:")
print(df["age"].describe())

# Do NOT drop anything yet.
# First inspect the missing values.

df["age_group"] = np.where(
    df["age"].isna(),
    "missing",
    np.where(df["age"] >= 60, "age>=60", "age<60")
)

print("\nAge groups:")
print(df["age_group"].value_counts())


# ------------------------------------------------------------
# 7. Find actual image files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE FILE AUDIT")
print("=" * 70)

jpg_files = []

for root, dirs, files in os.walk(HAM_ROOT):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            jpg_files.append(os.path.join(root, file))

print("Image files found:", len(jpg_files))

# Create image-id lookup
image_paths = {}

for path in jpg_files:
    image_name = os.path.splitext(os.path.basename(path))[0]
    image_paths[image_name] = path

print("Unique image filenames:", len(image_paths))


# ------------------------------------------------------------
# 8. Check metadata ↔ image correspondence
# ------------------------------------------------------------

metadata_ids = set(df["image_id"].astype(str))
file_ids = set(image_paths.keys())

missing_images = metadata_ids - file_ids
extra_images = file_ids - metadata_ids

print("\nMetadata images :", len(metadata_ids))
print("Physical images :", len(file_ids))

print("Metadata images without file:", len(missing_images))
print("Files without metadata      :", len(extra_images))

if len(missing_images) > 0:
    print("\nFirst missing images:")
    print(list(missing_images)[:20])

if len(extra_images) > 0:
    print("\nFirst extra images:")
    print(list(extra_images)[:20])


# ------------------------------------------------------------
# 9. Final audit summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL AUDIT SUMMARY")
print("=" * 70)

print("Total metadata rows       :", len(df))
print("Unique images             :", df["image_id"].nunique())
print("Unique lesions            :", df["lesion_id"].nunique())
print("Unique classes            :", df["dx"].nunique())
print("Missing ages              :", df["age"].isna().sum())
print("Image files               :", len(image_paths))
print("Missing image files       :", len(missing_images))
print("Extra image files         :", len(extra_images))
print("Inconsistent lesion labels:", inconsistent_lesions)

print("\nClass counts:")
print(class_counts)

print("\n✓ STEP 2 COMPLETE")

HAM10000 PATH CHECK
HAM root exists : True
Metadata exists : True
Metadata path   : /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv

METADATA
Number of rows : 10015
Number of cols : 7

Columns:
['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']

First 5 rows:


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear



CLASS DISTRIBUTION
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Class proportions:
dx
nv       0.6695
mel      0.1111
bkl      0.1097
bcc      0.0513
akiec    0.0327
vasc     0.0142
df       0.0115
Name: count, dtype: float64

LESION STATISTICS
Unique image IDs : 10015
Unique lesions  : 7470

Images per lesion:
1    5514
2    1423
3     490
4      34
5       5
6       4
Name: count, dtype: int64

Single-image lesions: 5514
Multi-image lesions: 1956

LESION → DIAGNOSIS CONSISTENCY
Lesions with exactly one diagnosis: 7470
Lesions with multiple diagnoses: 0
✓ Diagnosis is consistent within every lesion.

AGE / SENSITIVE ATTRIBUTE
Missing age values: 57

Age statistics:
count    9958.000000
mean       51.863828
std        16.968614
min         0.000000
25%        40.000000
50%        50.000000
75%        65.000000
max        85.000000
Name: age, dtype: float64

Age groups:
age_group
age<60     6356
age>=60 

In [3]:
# ============================================================
# STEP 3: LESION-LEVEL 70/20/10 SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

np.random.seed(SEED)

# ------------------------------------------------------------
# Checkpoint directories
# ------------------------------------------------------------

CHECKPOINT_ROOT = "/kaggle/working/biaspruner_checkpoints"

os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(os.path.join(CHECKPOINT_ROOT, "task1"), exist_ok=True)
os.makedirs(os.path.join(CHECKPOINT_ROOT, "task2"), exist_ok=True)
os.makedirs(os.path.join(CHECKPOINT_ROOT, "splits"), exist_ok=True)

print("Checkpoint directory:")
print(CHECKPOINT_ROOT)


# ------------------------------------------------------------
# Create one row per lesion
# ------------------------------------------------------------

lesion_df = (
    df.groupby("lesion_id", as_index=False)
      .agg({
          "dx": "first"
      })
)

print("\n" + "=" * 70)
print("LESION-LEVEL DATA")
print("=" * 70)

print("Total lesions:", len(lesion_df))
print("\nLesion class distribution:")
print(lesion_df["dx"].value_counts())


# ------------------------------------------------------------
# 70% train / 30% temporary
# ------------------------------------------------------------

train_lesions, temp_lesions = train_test_split(
    lesion_df,
    test_size=0.30,
    random_state=SEED,
    stratify=lesion_df["dx"]
)


# ------------------------------------------------------------
# 20% validation / 10% test
#
# temp = 30%
# validation should be 20/30 = 2/3 of temp
# test should be 10/30 = 1/3 of temp
# ------------------------------------------------------------

val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=1/3,
    random_state=SEED,
    stratify=temp_lesions["dx"]
)


# ------------------------------------------------------------
# Assign split labels to the original image-level dataframe
# ------------------------------------------------------------

train_ids = set(train_lesions["lesion_id"])
val_ids = set(val_lesions["lesion_id"])
test_ids = set(test_lesions["lesion_id"])

df["split"] = df["lesion_id"].apply(
    lambda x:
        "train" if x in train_ids
        else "val" if x in val_ids
        else "test"
)


# ------------------------------------------------------------
# Verify split sizes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE-LEVEL SPLIT")
print("=" * 70)

print(df["split"].value_counts())

print("\nPercentages:")
print(
    (df["split"].value_counts(normalize=True) * 100)
    .round(2)
)


# ------------------------------------------------------------
# Verify lesion-level sizes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LESION-LEVEL SPLIT")
print("=" * 70)

print("Train lesions:", len(train_lesions))
print("Val lesions  :", len(val_lesions))
print("Test lesions :", len(test_lesions))

print("\nPercentages:")

total_lesions = len(lesion_df)

print(
    "Train:",
    round(len(train_lesions) / total_lesions * 100, 2),
    "%"
)

print(
    "Val  :",
    round(len(val_lesions) / total_lesions * 100, 2),
    "%"
)

print(
    "Test :",
    round(len(test_lesions) / total_lesions * 100, 2),
    "%"
)


# ------------------------------------------------------------
# Class distribution in each split
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION BY SPLIT")
print("=" * 70)

split_class_table = pd.crosstab(
    df["dx"],
    df["split"]
)

# Ensure column order
for col in ["train", "val", "test"]:
    if col not in split_class_table.columns:
        split_class_table[col] = 0

split_class_table = split_class_table[
    ["train", "val", "test"]
]

print(split_class_table)

print("\nClass percentages within each split:")

for split_name in ["train", "val", "test"]:
    print(f"\n--- {split_name.upper()} ---")
    
    counts = df.loc[
        df["split"] == split_name,
        "dx"
    ].value_counts()
    
    percentages = (
        counts / counts.sum() * 100
    ).round(2)
    
    print(
        pd.DataFrame({
            "count": counts,
            "percentage": percentages
        })
    )


# ------------------------------------------------------------
# Verify NO lesion leakage
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LESION LEAKAGE CHECK")
print("=" * 70)

train_set = set(df.loc[df["split"] == "train", "lesion_id"])
val_set = set(df.loc[df["split"] == "val", "lesion_id"])
test_set = set(df.loc[df["split"] == "test", "lesion_id"])

train_val_overlap = train_set & val_set
train_test_overlap = train_set & test_set
val_test_overlap = val_set & test_set

print("Train ∩ Val :", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Val ∩ Test  :", len(val_test_overlap))

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\n✓ NO LESION LEAKAGE")


# ------------------------------------------------------------
# Verify every image has exactly one split
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT INTEGRITY")
print("=" * 70)

print(
    "Images with missing split:",
    df["split"].isna().sum()
)

print(
    "Unique images:",
    df["image_id"].nunique()
)

print(
    "Unique images after split:",
    df.groupby("split")["image_id"].nunique().sum()
)


# ------------------------------------------------------------
# Save permanent split
# ------------------------------------------------------------

SPLIT_PATH = os.path.join(
    CHECKPOINT_ROOT,
    "splits",
    "ham10000_lesion_split_seed42.csv"
)

df.to_csv(
    SPLIT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("SPLIT SAVED")
print("=" * 70)

print(SPLIT_PATH)
print("File exists:", os.path.exists(SPLIT_PATH))


# ------------------------------------------------------------
# Save a small configuration file
# ------------------------------------------------------------

CONFIG_PATH = os.path.join(
    CHECKPOINT_ROOT,
    "splits",
    "split_config.txt"
)

with open(CONFIG_PATH, "w") as f:
    f.write("Dataset: HAM10000\n")
    f.write("Seed: 42\n")
    f.write("Split: lesion-level 70/20/10\n")
    f.write("Total images: 10015\n")
    f.write(f"Total lesions: {len(lesion_df)}\n")
    f.write(f"Train lesions: {len(train_lesions)}\n")
    f.write(f"Validation lesions: {len(val_lesions)}\n")
    f.write(f"Test lesions: {len(test_lesions)}\n")

print(CONFIG_PATH)

print("\n✓ STEP 3 COMPLETE")

Checkpoint directory:
/kaggle/working/biaspruner_checkpoints

LESION-LEVEL DATA
Total lesions: 7470

Lesion class distribution:
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64

IMAGE-LEVEL SPLIT
split
train    6981
val      2033
test     1001
Name: count, dtype: int64

Percentages:
split
train    69.71
val      20.30
test     10.00
Name: proportion, dtype: float64

LESION-LEVEL SPLIT
Train lesions: 5229
Val lesions  : 1494
Test lesions : 747

Percentages:
Train: 70.0 %
Val  : 20.0 %
Test : 10.0 %

CLASS DISTRIBUTION BY SPLIT
split  train   val  test
dx                      
akiec    222    72    33
bcc      361   106    47
bkl      772   215   112
df        71    32    12
mel      773   226   114
nv      4683  1355   667
vasc      99    27    16

Class percentages within each split:

--- TRAIN ---
       count  percentage
dx                      
nv      4683       67.08
mel      773       11.07
bkl      772

In [4]:
# ============================================================
# STEP 4: DEFINE TASK 1 AND TASK 2
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Make sure the permanent split exists
# ------------------------------------------------------------

SPLIT_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "splits/ham10000_lesion_split_seed42.csv"
)

assert os.path.exists(SPLIT_PATH), \
    "Permanent split file was not found!"

# Load the permanent split
df = pd.read_csv(SPLIT_PATH)

print("=" * 70)
print("PERMANENT SPLIT LOADED")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


# ------------------------------------------------------------
# TASK DEFINITIONS
# ------------------------------------------------------------

TASKS = {
    1: ["df", "bkl"],
    2: ["vasc", "mel"],
    3: ["bcc", "nv", "akiec"]
}

# We are only implementing these today
ACTIVE_TASKS = [1, 2]


print("\n" + "=" * 70)
print("TASK DEFINITIONS")
print("=" * 70)

for task_id, classes in TASKS.items():
    status = "ACTIVE TODAY" if task_id in ACTIVE_TASKS else "NOT USED TODAY"
    
    print(
        f"Task {task_id}: {classes}"
        f"    [{status}]"
    )


# ------------------------------------------------------------
# Verify every class belongs to exactly one task
# ------------------------------------------------------------

all_task_classes = [
    cls
    for task_classes in TASKS.values()
    for cls in task_classes
]

print("\n" + "=" * 70)
print("TASK CLASS INTEGRITY")
print("=" * 70)

print("Total task classes:", len(all_task_classes))
print("Unique task classes:", len(set(all_task_classes)))

assert len(all_task_classes) == len(set(all_task_classes)), \
    "ERROR: A class appears in more than one task!"

assert set(all_task_classes) == set(df["dx"].unique()), \
    "ERROR: Task definitions do not cover all dataset classes!"

print("✓ Every HAM class belongs to exactly one task.")


# ------------------------------------------------------------
# Create task label
# ------------------------------------------------------------

class_to_task = {}

for task_id, classes in TASKS.items():
    for cls in classes:
        class_to_task[cls] = task_id

df["task"] = df["dx"].map(class_to_task)

print("\nTask assignment:")
print(df["task"].value_counts().sort_index())


# ------------------------------------------------------------
# TASK DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE DISTRIBUTION BY TASK")
print("=" * 70)

task_distribution = pd.crosstab(
    df["dx"],
    df["task"]
)

print(task_distribution)


print("\nTotal images per task:")

for task_id in [1, 2, 3]:
    count = (df["task"] == task_id).sum()
    print(f"Task {task_id}: {count}")


# ------------------------------------------------------------
# TASK DISTRIBUTION BY SPLIT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK DISTRIBUTION BY TRAIN / VAL / TEST")
print("=" * 70)

task_split_table = pd.crosstab(
    df["task"],
    df["split"]
)

for col in ["train", "val", "test"]:
    if col not in task_split_table.columns:
        task_split_table[col] = 0

task_split_table = task_split_table[
    ["train", "val", "test"]
]

print(task_split_table)


# ------------------------------------------------------------
# CLASS DISTRIBUTION FOR TASK 1
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK 1: df + bkl")
print("=" * 70)

task1_df = df[df["task"] == 1].copy()

print("Total images:", len(task1_df))

print("\nClass distribution:")
print(task1_df["dx"].value_counts())

print("\nDistribution by split:")
print(
    pd.crosstab(
        task1_df["dx"],
        task1_df["split"]
    )
)


# ------------------------------------------------------------
# CLASS DISTRIBUTION FOR TASK 2
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK 2: vasc + mel")
print("=" * 70)

task2_df = df[df["task"] == 2].copy()

print("Total images:", len(task2_df))

print("\nClass distribution:")
print(task2_df["dx"].value_counts())

print("\nDistribution by split:")
print(
    pd.crosstab(
        task2_df["dx"],
        task2_df["split"]
    )
)


# ------------------------------------------------------------
# AGE / FAIRNESS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGE GROUP DISTRIBUTION")
print("=" * 70)

print(
    pd.crosstab(
        df["task"],
        df["age_group"]
    )
)


# ------------------------------------------------------------
# AGE DISTRIBUTION FOR ACTIVE TASKS
# ------------------------------------------------------------

for task_id in ACTIVE_TASKS:

    task_data = df[df["task"] == task_id]

    print("\n" + "-" * 70)
    print(f"TASK {task_id} AGE GROUPS")
    print("-" * 70)

    print(
        task_data["age_group"].value_counts()
    )


# ------------------------------------------------------------
# CHECK TASK OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK OVERLAP CHECK")
print("=" * 70)

task1_ids = set(task1_df["image_id"])
task2_ids = set(task2_df["image_id"])

print(
    "Task 1 ∩ Task 2:",
    len(task1_ids & task2_ids)
)

assert len(task1_ids & task2_ids) == 0

print("✓ Task 1 and Task 2 contain completely different classes/images.")


# ------------------------------------------------------------
# SAVE TASK DEFINITIONS
# ------------------------------------------------------------

TASK_CONFIG_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "splits/task_definitions.txt"
)

with open(TASK_CONFIG_PATH, "w") as f:

    f.write("BiasPruner HAM10000 experiment\n")
    f.write("================================\n\n")

    f.write("Task 1: df, bkl\n")
    f.write("Task 2: vasc, mel\n")
    f.write("Task 3: bcc, nv, akiec\n\n")

    f.write("Active tasks for current experiment: Task 1, Task 2\n")

print("\nTask definition saved:")
print(TASK_CONFIG_PATH)

print("\n" + "=" * 70)
print("✓ STEP 4 COMPLETE")
print("=" * 70)

PERMANENT SPLIT LOADED
Rows: 10015
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'age_group', 'split']

TASK DEFINITIONS
Task 1: ['df', 'bkl']    [ACTIVE TODAY]
Task 2: ['vasc', 'mel']    [ACTIVE TODAY]
Task 3: ['bcc', 'nv', 'akiec']    [NOT USED TODAY]

TASK CLASS INTEGRITY
Total task classes: 7
Unique task classes: 7
✓ Every HAM class belongs to exactly one task.

Task assignment:
task
1    1214
2    1255
3    7546
Name: count, dtype: int64

IMAGE DISTRIBUTION BY TASK
task      1     2     3
dx                     
akiec     0     0   327
bcc       0     0   514
bkl    1099     0     0
df      115     0     0
mel       0  1113     0
nv        0     0  6705
vasc      0   142     0

Total images per task:
Task 1: 1214
Task 2: 1255
Task 3: 7546

TASK DISTRIBUTION BY TRAIN / VAL / TEST
split  train   val  test
task                    
1        843   247   124
2        872   253   130
3       5266  1533   747

TASK 1: df + bkl
Total images: 1214

Class 

In [5]:
# ============================================================
# STEP 5 — RESNET-50 + TASK 1 DATALOADERS
# ============================================================

import os
import glob
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

BASE_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
METADATA_PATH = os.path.join(BASE_DIR, "HAM10000_metadata.csv")

SPLIT_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "splits/ham10000_lesion_split_seed42.csv"
)

# ------------------------------------------------------------
# 3. Load metadata + permanent split
# ------------------------------------------------------------

df = pd.read_csv(METADATA_PATH)
split_df = pd.read_csv(SPLIT_PATH)

print("=" * 70)
print("STEP 5 — DATA + MODEL SETUP")
print("=" * 70)

print("Metadata rows:", len(df))
print("Split rows:", len(split_df))

# Merge split information if necessary
if "split" not in df.columns:
    df = df.merge(
        split_df[["image_id", "split"]],
        on="image_id",
        how="left"
    )

# ------------------------------------------------------------
# 4. Build image lookup
# ------------------------------------------------------------

image_files = glob.glob(
    os.path.join(BASE_DIR, "**", "*.jpg"),
    recursive=True
)

image_lookup = {}

for path in image_files:
    image_id = os.path.splitext(os.path.basename(path))[0]

    # Keep one file per unique image_id
    if image_id not in image_lookup:
        image_lookup[image_id] = path

df["image_path"] = df["image_id"].map(image_lookup)

missing_images = df["image_path"].isna().sum()

print("\nImage lookup:")
print("Unique image files indexed:", len(image_lookup))
print("Missing images:", missing_images)

assert missing_images == 0, "Some HAM images could not be located."

# ------------------------------------------------------------
# 5. Task 1 only
# ------------------------------------------------------------

TASK1_CLASSES = ["df", "bkl"]

TASK1_LABEL_MAP = {
    "df": 0,
    "bkl": 1
}

task1_df = df[df["dx"].isin(TASK1_CLASSES)].copy()

task1_df["label"] = task1_df["dx"].map(TASK1_LABEL_MAP)

print("\n" + "=" * 70)
print("TASK 1")
print("=" * 70)

print("Classes:", TASK1_CLASSES)
print("Total images:", len(task1_df))

print("\nClass distribution:")
print(task1_df["dx"].value_counts())

print("\nSplit distribution:")
print(
    pd.crosstab(
        task1_df["dx"],
        task1_df["split"]
    )
)

# ------------------------------------------------------------
# 6. Image transformations
# ------------------------------------------------------------
#
# Paper:
# - resize
# - dataset-specific normalization
# - random horizontal flipping
#
# We use ImageNet normalization because ResNet-50 is the
# standard torchvision implementation and we are not using
# pretrained ImageNet weights for the paper-style experiment.
#
# IMPORTANT:
# We will keep this explicit and consistent across all tasks.
# ------------------------------------------------------------

IMG_SIZE = 224

# Standard ImageNet statistics
# Used only for normalization; NOT ImageNet pretrained weights.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ------------------------------------------------------------
# 7. Dataset class
# ------------------------------------------------------------

class HAMDiseaseDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["label"])

        # Keep age-group information for fairness evaluation
        age_group = row["age_group"]

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "age_group": age_group,
            "image_id": row["image_id"]
        }


# ------------------------------------------------------------
# 8. Create Task-1 splits
# ------------------------------------------------------------

task1_train_df = task1_df[task1_df["split"] == "train"].copy()
task1_val_df   = task1_df[task1_df["split"] == "val"].copy()
task1_test_df  = task1_df[task1_df["split"] == "test"].copy()

print("\n" + "=" * 70)
print("TASK 1 SPLITS")
print("=" * 70)

print("Train:", len(task1_train_df))
print("Val:  ", len(task1_val_df))
print("Test: ", len(task1_test_df))

# ------------------------------------------------------------
# 9. Datasets
# ------------------------------------------------------------

train_dataset = HAMDiseaseDataset(
    task1_train_df,
    transform=train_transform
)

val_dataset = HAMDiseaseDataset(
    task1_val_df,
    transform=eval_transform
)

test_dataset = HAMDiseaseDataset(
    task1_test_df,
    transform=eval_transform
)

# ------------------------------------------------------------
# 10. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 11. Check one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\n" + "=" * 70)
print("BATCH CHECK")
print("=" * 70)

print("Image tensor shape:", batch["image"].shape)
print("Label shape:", batch["label"].shape)
print("Labels in batch:", batch["label"][:10].tolist())
print("First image ID:", batch["image_id"][0])
print("First age group:", batch["age_group"][0])

# ------------------------------------------------------------
# 12. Build ResNet-50
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\n" + "=" * 70)
print("RESNET-50")
print("=" * 70)

print("Device:", device)

# IMPORTANT:
# Paper uses ResNet-50 backbone.
# We start from random initialization rather than ImageNet
# pretrained weights because the paper's HAM configuration
# used pretrained=False in the implementation setup we are
# reproducing.

model = models.resnet50(weights=None)

# Replace ImageNet 1000-class classifier with Task-1's 2 classes
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(TASK1_CLASSES)
)

model = model.to(device)

print("Backbone: ResNet-50")
print("Pretrained weights: None")
print("Classifier outputs:", len(TASK1_CLASSES))

# ------------------------------------------------------------
# 13. Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\n" + "=" * 70)
print("✓ STEP 5 READY")
print("=" * 70)

STEP 5 — DATA + MODEL SETUP
Metadata rows: 10015
Split rows: 10015

Image lookup:
Unique image files indexed: 10015
Missing images: 0

TASK 1
Classes: ['df', 'bkl']
Total images: 1214

Class distribution:
dx
bkl    1099
df      115
Name: count, dtype: int64

Split distribution:
split  test  train  val
dx                     
bkl     112    772  215
df       12     71   32

TASK 1 SPLITS
Train: 843
Val:   247
Test:  124


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'age_group'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File

In [6]:
# ============================================================
# STEP 5 — RESNET-50 + TASK 1 DATALOADERS
# CORRECTED VERSION
# ============================================================

import os
import glob
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

BASE_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

METADATA_PATH = os.path.join(
    BASE_DIR,
    "HAM10000_metadata.csv"
)

SPLIT_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "splits/ham10000_lesion_split_seed42.csv"
)

# ------------------------------------------------------------
# 3. Load metadata + permanent split
# ------------------------------------------------------------

df = pd.read_csv(METADATA_PATH)
split_df = pd.read_csv(SPLIT_PATH)

print("=" * 70)
print("STEP 5 — DATA + MODEL SETUP")
print("=" * 70)

print("Metadata rows:", len(df))
print("Split rows:", len(split_df))

# ------------------------------------------------------------
# 4. Attach permanent split + age_group
# ------------------------------------------------------------

# Remove these columns first if they already exist
for col in ["split", "age_group"]:
    if col in df.columns:
        df = df.drop(columns=[col])

df = df.merge(
    split_df[["image_id", "split", "age_group"]],
    on="image_id",
    how="left"
)

print("\nColumns after merge:")
print(df.columns.tolist())

print("\nMissing split:", df["split"].isna().sum())
print("Missing age_group:", df["age_group"].isna().sum())

assert df["split"].isna().sum() == 0

# ------------------------------------------------------------
# 5. Build image lookup
# ------------------------------------------------------------

image_files = glob.glob(
    os.path.join(BASE_DIR, "**", "*.jpg"),
    recursive=True
)

image_lookup = {}

for path in image_files:
    image_id = os.path.splitext(
        os.path.basename(path)
    )[0]

    if image_id not in image_lookup:
        image_lookup[image_id] = path

df["image_path"] = df["image_id"].map(image_lookup)

missing_images = df["image_path"].isna().sum()

print("\n" + "=" * 70)
print("IMAGE LOOKUP")
print("=" * 70)

print("Unique image files indexed:", len(image_lookup))
print("Missing images:", missing_images)

assert missing_images == 0

# ------------------------------------------------------------
# 6. Task 1
# ------------------------------------------------------------

TASK1_CLASSES = ["df", "bkl"]

TASK1_LABEL_MAP = {
    "df": 0,
    "bkl": 1
}

task1_df = df[
    df["dx"].isin(TASK1_CLASSES)
].copy()

task1_df["label"] = task1_df["dx"].map(
    TASK1_LABEL_MAP
)

print("\n" + "=" * 70)
print("TASK 1")
print("=" * 70)

print("Classes:", TASK1_CLASSES)
print("Total images:", len(task1_df))

print("\nClass distribution:")
print(task1_df["dx"].value_counts())

print("\nSplit distribution:")
print(
    pd.crosstab(
        task1_df["dx"],
        task1_df["split"]
    )
)

# ------------------------------------------------------------
# 7. Transforms
# ------------------------------------------------------------

IMG_SIZE = 224

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 8. Dataset
# ------------------------------------------------------------

class HAMDiseaseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(
            drop=True
        )
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["label"])

        age_group = row["age_group"]

        return {
            "image": image,
            "label": torch.tensor(
                label,
                dtype=torch.long
            ),
            "age_group": age_group,
            "image_id": row["image_id"]
        }


# ------------------------------------------------------------
# 9. Task 1 splits
# ------------------------------------------------------------

task1_train_df = task1_df[
    task1_df["split"] == "train"
].copy()

task1_val_df = task1_df[
    task1_df["split"] == "val"
].copy()

task1_test_df = task1_df[
    task1_df["split"] == "test"
].copy()

print("\n" + "=" * 70)
print("TASK 1 SPLITS")
print("=" * 70)

print("Train:", len(task1_train_df))
print("Val:  ", len(task1_val_df))
print("Test: ", len(task1_test_df))

# ------------------------------------------------------------
# 10. Datasets
# ------------------------------------------------------------

train_dataset = HAMDiseaseDataset(
    task1_train_df,
    transform=train_transform
)

val_dataset = HAMDiseaseDataset(
    task1_val_df,
    transform=eval_transform
)

test_dataset = HAMDiseaseDataset(
    task1_test_df,
    transform=eval_transform
)

# ------------------------------------------------------------
# 11. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 12. Check one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\n" + "=" * 70)
print("BATCH CHECK")
print("=" * 70)

print(
    "Image tensor shape:",
    batch["image"].shape
)

print(
    "Label shape:",
    batch["label"].shape
)

print(
    "Labels in batch:",
    batch["label"][:10].tolist()
)

print(
    "First image ID:",
    batch["image_id"][0]
)

print(
    "First age group:",
    batch["age_group"][0]
)

# ------------------------------------------------------------
# 13. ResNet-50
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\n" + "=" * 70)
print("RESNET-50")
print("=" * 70)

print("Device:", device)

model = models.resnet50(
    weights=None
)

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(TASK1_CLASSES)
)

model = model.to(device)

print("Backbone: ResNet-50")
print("Pretrained weights: None")
print(
    "Classifier outputs:",
    len(TASK1_CLASSES)
)

# ------------------------------------------------------------
# 14. Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print("\n" + "=" * 70)
print("✓ STEP 5 COMPLETE")
print("=" * 70)

STEP 5 — DATA + MODEL SETUP
Metadata rows: 10015
Split rows: 10015

Columns after merge:
['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'split', 'age_group']

Missing split: 0
Missing age_group: 0

IMAGE LOOKUP
Unique image files indexed: 10015
Missing images: 0

TASK 1
Classes: ['df', 'bkl']
Total images: 1214

Class distribution:
dx
bkl    1099
df      115
Name: count, dtype: int64

Split distribution:
split  test  train  val
dx                     
bkl     112    772  215
df       12     71   32

TASK 1 SPLITS
Train: 843
Val:   247
Test:  124

BATCH CHECK
Image tensor shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])
Labels in batch: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
First image ID: ISIC_0030231
First age group: age<60

RESNET-50
Device: cuda
Backbone: ResNet-50
Pretrained weights: None
Classifier outputs: 2
Total parameters: 23,512,130
Trainable parameters: 23,512,130

✓ STEP 5 COMPLETE


In [7]:
# ============================================================
# STEP 5A — COMPUTE HAM TRAINING-SET MEAN / STD
# ============================================================

from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# No normalization here
stats_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

stats_dataset = HAMDiseaseDataset(
    task1_train_df,
    transform=stats_transform
)

stats_loader = DataLoader(
    stats_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
num_pixels = 0

for batch in stats_loader:

    images = batch["image"]   # [B, 3, H, W]

    channel_sum += images.sum(
        dim=[0, 2, 3]
    )

    channel_squared_sum += (
        images ** 2
    ).sum(dim=[0, 2, 3])

    num_pixels += (
        images.shape[0]
        * images.shape[2]
        * images.shape[3]
    )

mean = channel_sum / num_pixels

variance = (
    channel_squared_sum / num_pixels
    - mean ** 2
)

std = torch.sqrt(variance)

print("=" * 70)
print("TASK 1 TRAINING-SET IMAGE STATISTICS")
print("=" * 70)

print("Mean:", mean.tolist())
print("Std: ", std.tolist())

print("\nNumber of pixels used:", num_pixels)

print("\n" + "=" * 70)
print("✓ STEP 5A COMPLETE")
print("=" * 70)

TASK 1 TRAINING-SET IMAGE STATISTICS
Mean: [0.7035894393920898, 0.5483366847038269, 0.578921377658844]
Std:  [0.13415099680423737, 0.13551630079746246, 0.15061676502227783]

Number of pixels used: 42298368

✓ STEP 5A COMPLETE


In [8]:
# ============================================================
# STEP 5B — FREEZE HAM PREPROCESSING
# ============================================================

HAM_MEAN = [
    0.70358944,
    0.54833668,
    0.57892138
]

HAM_STD = [
    0.13415100,
    0.13551630,
    0.15061677
]

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(HAM_MEAN, HAM_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(HAM_MEAN, HAM_STD)
])

# Rebuild datasets with FINAL transforms
train_dataset = HAMDiseaseDataset(
    task1_train_df,
    transform=train_transform
)

val_dataset = HAMDiseaseDataset(
    task1_val_df,
    transform=eval_transform
)

test_dataset = HAMDiseaseDataset(
    task1_test_df,
    transform=eval_transform
)

# Rebuild loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Check normalized batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

images = batch["image"]

print("=" * 70)
print("FINAL PREPROCESSING CHECK")
print("=" * 70)

print("Batch shape:", images.shape)

print(
    "Batch mean:",
    images.mean(dim=[0, 2, 3]).tolist()
)

print(
    "Batch std:",
    images.std(dim=[0, 2, 3]).tolist()
)

print(
    "Labels:",
    batch["label"][:10].tolist()
)

print(
    "Age groups:",
    list(batch["age_group"][:10])
)

print("\nFrozen HAM mean:", HAM_MEAN)
print("Frozen HAM std: ", HAM_STD)

print("\n" + "=" * 70)
print("✓ PREPROCESSING FROZEN")
print("=" * 70)

FINAL PREPROCESSING CHECK
Batch shape: torch.Size([32, 3, 224, 224])
Batch mean: [-0.036054834723472595, -0.02942693419754505, -0.0784621313214302]
Batch std: [0.8586270213127136, 0.8876276016235352, 0.9020814299583435]
Labels: [1, 1, 1, 1, 1, 0, 0, 1, 1, 1]
Age groups: ['age<60', 'age>=60', 'age<60', 'age>=60', 'age>=60', 'age<60', 'age<60', 'age>=60', 'age>=60', 'age<60']

Frozen HAM mean: [0.70358944, 0.54833668, 0.57892138]
Frozen HAM std:  [0.134151, 0.1355163, 0.15061677]

✓ PREPROCESSING FROZEN


In [9]:
import torch
import torch.nn.functional as F

# ============================================================
# STEP 6A: GENERALIZED CROSS ENTROPY (GCE) DIAGNOSTIC
# ============================================================

Q = 0.7

def generalized_cross_entropy(logits, targets, q=0.7):
    """
    GCE:
        L = (1 - p_y^q) / q

    logits  : [batch_size, num_classes]
    targets : [batch_size]
    """
    probs = F.softmax(logits, dim=1)

    # Probability assigned to the true class
    p_y = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

    # Numerical safety
    p_y = p_y.clamp(min=1e-7, max=1.0)

    loss = (1.0 - p_y.pow(q)) / q

    return loss.mean()


# ------------------------------------------------------------
# Put model in training mode
# ------------------------------------------------------------
model.train()

# Make sure gradients are clear
model.zero_grad(set_to_none=True)


# ------------------------------------------------------------
# Get ONE batch
# ------------------------------------------------------------
images, labels, image_ids, age_groups = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)


# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------
logits = model(images)

print("=" * 70)
print("STEP 6A — GCE DIAGNOSTIC")
print("=" * 70)

print("Input shape :", images.shape)
print("Logit shape :", logits.shape)


# ------------------------------------------------------------
# Check logits
# ------------------------------------------------------------
print("\nLogit statistics:")
print("  min :", logits.min().item())
print("  max :", logits.max().item())
print("  mean:", logits.mean().item())
print("  std :", logits.std().item())


# ------------------------------------------------------------
# Calculate probabilities
# ------------------------------------------------------------
probs = F.softmax(logits, dim=1)

print("\nProbability statistics:")
print("  min :", probs.min().item())
print("  max :", probs.max().item())
print("  mean:", probs.mean().item())


# ------------------------------------------------------------
# Calculate GCE
# ------------------------------------------------------------
gce_loss = generalized_cross_entropy(
    logits,
    labels,
    q=Q
)

print("\nGCE:")
print("  q    :", Q)
print("  loss :", gce_loss.item())


# ------------------------------------------------------------
# Compare with ordinary Cross Entropy
# ------------------------------------------------------------
ce_loss = F.cross_entropy(logits, labels)

print("\nCross Entropy:")
print("  loss :", ce_loss.item())


# ------------------------------------------------------------
# Backward pass
# ------------------------------------------------------------
gce_loss.backward()


# ------------------------------------------------------------
# Check gradients
# ------------------------------------------------------------
finite_gradients = True
total_grad_norm = 0.0
num_params_with_grad = 0

for name, param in model.named_parameters():

    if param.grad is not None:

        num_params_with_grad += 1

        if not torch.isfinite(param.grad).all():
            finite_gradients = False

        total_grad_norm += param.grad.detach().norm().item() ** 2

total_grad_norm = total_grad_norm ** 0.5


print("\nGradient check:")
print("  Parameters with gradients:", num_params_with_grad)
print("  Total gradient norm      :", total_grad_norm)
print("  Gradients finite?        :", finite_gradients)


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------
predictions = logits.argmax(dim=1)

batch_accuracy = (predictions == labels).float().mean().item()

print("\nBatch prediction check:")
print("  Accuracy:", batch_accuracy)

print("  True labels :", labels[:20].detach().cpu().tolist())
print("  Predictions :", predictions[:20].detach().cpu().tolist())


# ------------------------------------------------------------
# Class prediction distribution
# ------------------------------------------------------------
unique_pred, pred_counts = torch.unique(
    predictions,
    return_counts=True
)

print("\nPrediction distribution:")

for cls, count in zip(
    unique_pred.detach().cpu().tolist(),
    pred_counts.detach().cpu().tolist()
):
    print(f"  Class {cls}: {count}")


print("\n" + "=" * 70)
print("STEP 6A COMPLETE")
print("=" * 70)

AttributeError: 'str' object has no attribute 'to'

In [10]:
# ============================================================
# STEP 6A — CHECK EXACT DATALOADER OUTPUT
# ============================================================

batch = next(iter(train_loader))

print("=" * 70)
print("DATALOADER BATCH STRUCTURE")
print("=" * 70)

print("Batch type:", type(batch))
print("Number of returned items:", len(batch))

for i, item in enumerate(batch):
    print("\nItem", i)
    print("  Type:", type(item))

    if torch.is_tensor(item):
        print("  Shape:", item.shape)
        print("  Dtype:", item.dtype)

        if item.numel() > 0:
            print("  First values:",
                  item.flatten()[:10].detach().cpu().tolist())

    elif isinstance(item, (list, tuple)):
        print("  Length:", len(item))
        print("  First values:", item[:5])

    else:
        print("  First values:", str(item)[:200])

print("\n" + "=" * 70)
print("STEP 6A COMPLETE")
print("=" * 70)

DATALOADER BATCH STRUCTURE
Batch type: <class 'dict'>
Number of returned items: 4

Item 0
  Type: <class 'str'>
  First values: image

Item 1
  Type: <class 'str'>
  First values: label

Item 2
  Type: <class 'str'>
  First values: age_group

Item 3
  Type: <class 'str'>
  First values: image_id

STEP 6A COMPLETE


In [11]:
# ============================================================
# STEP 6A — GCE FORWARD/BACKWARD DIAGNOSTIC
# ============================================================

import torch
import torch.nn.functional as F

Q = 0.7

def generalized_cross_entropy(logits, targets, q=0.7):
    """
    Generalized Cross Entropy:
        L_GCE = (1 - p_y^q) / q

    p_y = probability assigned to the true class
    """

    probs = F.softmax(logits, dim=1)

    p_y = probs.gather(
        1,
        targets.unsqueeze(1)
    ).squeeze(1)

    # Numerical stability
    p_y = p_y.clamp(min=1e-7, max=1.0)

    loss = (1.0 - p_y.pow(q)) / q

    return loss.mean()


# ============================================================
# GET ONE BATCH
# ============================================================

batch = next(iter(train_loader))

images = batch["image"].to(device)
labels = batch["label"].to(device)


# ============================================================
# MODEL
# ============================================================

model.train()
model.zero_grad(set_to_none=True)


# ============================================================
# FORWARD
# ============================================================

logits = model(images)


print("=" * 70)
print("STEP 6A — GCE DIAGNOSTIC")
print("=" * 70)

print("\nInput:")
print("  Shape:", images.shape)
print("  Dtype:", images.dtype)

print("\nLabels:")
print("  Shape:", labels.shape)
print("  Dtype:", labels.dtype)
print("  Values:", labels[:20].detach().cpu().tolist())


# ============================================================
# LOGITS
# ============================================================

print("\nLogits:")
print("  Shape:", logits.shape)
print("  Min  :", logits.min().item())
print("  Max  :", logits.max().item())
print("  Mean :", logits.mean().item())
print("  Std  :", logits.std().item())


# ============================================================
# PROBABILITIES
# ============================================================

probs = F.softmax(logits, dim=1)

print("\nProbabilities:")
print("  Min  :", probs.min().item())
print("  Max  :", probs.max().item())
print("  Mean :", probs.mean().item())
print("  Row sums:",
      probs.sum(dim=1)[:5].detach().cpu().tolist())


# ============================================================
# GCE
# ============================================================

gce_loss = generalized_cross_entropy(
    logits,
    labels,
    q=Q
)

print("\nGCE:")
print("  q    :", Q)
print("  Loss :", gce_loss.item())


# ============================================================
# ORDINARY CE — FOR REFERENCE ONLY
# ============================================================

ce_loss = F.cross_entropy(logits, labels)

print("\nCross Entropy:")
print("  Loss :", ce_loss.item())


# ============================================================
# BACKWARD
# ============================================================

gce_loss.backward()


# ============================================================
# GRADIENT CHECK
# ============================================================

finite_gradients = True
total_grad_norm_squared = 0.0
num_params_with_grad = 0

for name, param in model.named_parameters():

    if param.grad is not None:

        num_params_with_grad += 1

        if not torch.isfinite(param.grad).all():
            finite_gradients = False

        total_grad_norm_squared += (
            param.grad.detach().norm().item() ** 2
        )

total_grad_norm = total_grad_norm_squared ** 0.5


print("\nGradient check:")
print("  Parameters with gradients:",
      num_params_with_grad)

print("  Total gradient norm:",
      total_grad_norm)

print("  Gradients finite?:",
      finite_gradients)


# ============================================================
# PREDICTIONS
# ============================================================

predictions = logits.argmax(dim=1)

batch_accuracy = (
    predictions == labels
).float().mean().item()

print("\nPredictions:")
print("  Accuracy:", batch_accuracy)

print("  True:",
      labels[:20].detach().cpu().tolist())

print("  Pred:",
      predictions[:20].detach().cpu().tolist())


# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

unique_pred, pred_counts = torch.unique(
    predictions,
    return_counts=True
)

print("\nPrediction distribution:")

for cls, count in zip(
    unique_pred.detach().cpu().tolist(),
    pred_counts.detach().cpu().tolist()
):
    print(f"  Class {cls}: {count}")


# ============================================================
# AGE GROUP CHECK
# ============================================================

print("\nAge groups:")
print(batch["age_group"][:10])


print("\n" + "=" * 70)
print("STEP 6A COMPLETE")
print("=" * 70)

STEP 6A — GCE DIAGNOSTIC

Input:
  Shape: torch.Size([32, 3, 224, 224])
  Dtype: torch.float32

Labels:
  Shape: torch.Size([32])
  Dtype: torch.int64
  Values: [1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Logits:
  Shape: torch.Size([32, 2])
  Min  : 0.10869001597166061
  Max  : 2.3578524589538574
  Mean : 0.4523259997367859
  Std  : 0.42027467489242554

Probabilities:
  Min  : 0.16079501807689667
  Max  : 0.8392049670219421
  Mean : 0.5
  Row sums: [1.0, 1.0, 1.0, 1.0, 1.0]

GCE:
  q    : 0.7
  Loss : 0.5021165609359741

Cross Entropy:
  Loss : 0.6267799139022827

Gradient check:
  Parameters with gradients: 161
  Total gradient norm: 37.28727035982192
  Gradients finite?: True

Predictions:
  Accuracy: 0.5
  True: [1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Pred: [1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0]

Prediction distribution:
  Class 0: 16
  Class 1: 16

Age groups:
['age>=60', 'age<60', 'age>=60', 'age>=60', 'age>=60', 'age

In [12]:
# ============================================================
# STEP 6B — 3-EPOCH GCE TRAINING SMOKE TEST
# ============================================================

import copy
import torch
import torch.nn.functional as F
from torchvision.models import resnet50


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

Q = 0.7
SMOKE_EPOCHS = 3
LR = 1e-4

print("=" * 70)
print("STEP 6B — GCE TRAINING SMOKE TEST")
print("=" * 70)

print("GCE q       :", Q)
print("Learning rate:", LR)
print("Epochs       :", SMOKE_EPOCHS)


# ------------------------------------------------------------
# Create a FRESH ResNet-50
# ------------------------------------------------------------

smoke_model = resnet50(weights=None)

num_features = smoke_model.fc.in_features

smoke_model.fc = torch.nn.Linear(
    num_features,
    2
)

smoke_model = smoke_model.to(device)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    smoke_model.parameters(),
    lr=LR
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

history = []

for epoch in range(1, SMOKE_EPOCHS + 1):

    smoke_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = smoke_model(images)

        loss = generalized_cross_entropy(
            logits,
            labels,
            q=Q
        )

        # Safety check
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite GCE loss at epoch {epoch}"
            )

        loss.backward()

        # Check gradients
        for param in smoke_model.parameters():
            if param.grad is not None:
                if not torch.isfinite(param.grad).all():
                    raise RuntimeError(
                        f"Non-finite gradient at epoch {epoch}"
                    )

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    history.append(
        {
            "epoch": epoch,
            "loss": epoch_loss,
            "accuracy": epoch_acc
        }
    )

    print(
        f"Epoch {epoch:02d}/{SMOKE_EPOCHS} | "
        f"GCE Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )


# ------------------------------------------------------------
# Final prediction distribution
# ------------------------------------------------------------

smoke_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in train_loader:

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = smoke_model(images)

        predictions = logits.argmax(dim=1)

        all_predictions.append(
            predictions.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

unique_pred, pred_counts = torch.unique(
    all_predictions,
    return_counts=True
)

print("\nFinal training prediction distribution:")

for cls, count in zip(
    unique_pred.tolist(),
    pred_counts.tolist()
):
    print(f"  Class {cls}: {count}")


print("\n" + "=" * 70)
print("STEP 6B COMPLETE")
print("=" * 70)

STEP 6B — GCE TRAINING SMOKE TEST
GCE q       : 0.7
Learning rate: 0.0001
Epochs       : 3
Epoch 01/3 | GCE Loss: 0.2001 | Accuracy: 0.8553


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79c6b142ce00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79c6b142ce00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 02/3 | GCE Loss: 0.1201 | Accuracy: 0.9158
Epoch 03/3 | GCE Loss: 0.1201 | Accuracy: 0.9158

Final training prediction distribution:
  Class 1: 843

STEP 6B COMPLETE


In [13]:
# ============================================================
# STEP 6C — DIAGNOSE CLASS COLLAPSE
# ============================================================

from collections import defaultdict

smoke_model.eval()

correct_by_class = defaultdict(int)
total_by_class = defaultdict(int)

pred_by_class = defaultdict(int)

all_true = []
all_pred = []

with torch.no_grad():

    for batch in train_loader:

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = smoke_model(images)
        predictions = logits.argmax(dim=1)

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

        for y, p in zip(
            labels.cpu().tolist(),
            predictions.cpu().tolist()
        ):

            total_by_class[y] += 1
            pred_by_class[p] += 1

            if y == p:
                correct_by_class[y] += 1


print("=" * 70)
print("STEP 6C — CLASS COLLAPSE DIAGNOSIS")
print("=" * 70)

print("\nTrue class distribution:")

for cls in sorted(total_by_class):
    print(
        f"  Class {cls}: "
        f"{total_by_class[cls]}"
    )


print("\nPredicted class distribution:")

for cls in sorted(pred_by_class):
    print(
        f"  Class {cls}: "
        f"{pred_by_class[cls]}"
    )


print("\nPer-class training accuracy:")

for cls in sorted(total_by_class):

    acc = (
        correct_by_class[cls]
        / total_by_class[cls]
    )

    print(
        f"  Class {cls}: "
        f"{acc:.4f} "
        f"({correct_by_class[cls]}/"
        f"{total_by_class[cls]})"
    )


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

confusion = torch.zeros(
    2,
    2,
    dtype=torch.int64
)

for y, p in zip(all_true, all_pred):
    confusion[y, p] += 1


print("\nConfusion matrix:")
print("Rows = true class")
print("Cols = predicted class")
print()
print(confusion)


print("\n" + "=" * 70)
print("STEP 6C COMPLETE")
print("=" * 70)

STEP 6C — CLASS COLLAPSE DIAGNOSIS

True class distribution:
  Class 0: 71
  Class 1: 772

Predicted class distribution:
  Class 1: 843

Per-class training accuracy:
  Class 0: 0.0000 (0/71)
  Class 1: 1.0000 (772/772)

Confusion matrix:
Rows = true class
Cols = predicted class

tensor([[  0,  71],
        [  0, 772]])

STEP 6C COMPLETE


In [15]:
import os
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================
HAM_ROOT = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
META_PATH = os.path.join(HAM_ROOT, "HAM10000_metadata.csv")

CHECKPOINT_ROOT = "/kaggle/working/biaspruner_checkpoints"
SPLIT_PATH = os.path.join(
    CHECKPOINT_ROOT,
    "splits",
    "ham10000_lesion_split_seed42.csv"
)

# ============================================================
# LOAD ORIGINAL HAM10000 METADATA
# ============================================================
metadata = pd.read_csv(META_PATH)

print("Metadata loaded:", metadata.shape)

# ============================================================
# LOAD LOCKED LESION-LEVEL SPLIT
# ============================================================
split_df = pd.read_csv(SPLIT_PATH)

print("Split loaded:", split_df.shape)

# ============================================================
# MERGE SPLIT INFORMATION
# ============================================================
merged_df = metadata.merge(
    split_df[["image_id", "split"]],
    on="image_id",
    how="left",
    validate="one_to_one"
)

# ============================================================
# CREATE AGE GROUP
# ============================================================
merged_df["age_group"] = np.where(
    merged_df["age"].isna(),
    "missing",
    np.where(
        merged_df["age"] >= 60,
        "age>=60",
        "age<60"
    )
)

# ============================================================
# SANITY CHECKS
# ============================================================
print("\nMerged dataframe:")
print(merged_df.shape)

print("\nColumns:")
print(merged_df.columns.tolist())

print("\nMissing split:", merged_df["split"].isna().sum())
print("Missing age_group:", merged_df["age_group"].isna().sum())

print("\nSplit distribution:")
print(merged_df["split"].value_counts())

print("\nAge-group distribution:")
print(merged_df["age_group"].value_counts())

print("\nClass distribution:")
print(merged_df["dx"].value_counts())

assert len(merged_df) == len(metadata)
assert merged_df["split"].notna().all()
assert merged_df["image_id"].is_unique

print("\n✓ merged_df successfully rebuilt")

Metadata loaded: (10015, 7)
Split loaded: (10015, 9)

Merged dataframe:
(10015, 9)

Columns:
['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'split', 'age_group']

Missing split: 0
Missing age_group: 0

Split distribution:
split
train    6981
val      2033
test     1001
Name: count, dtype: int64

Age-group distribution:
age_group
age<60     6356
age>=60    3602
missing      57
Name: count, dtype: int64

Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

✓ merged_df successfully rebuilt


In [16]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
IMAGE_SIZE = 224

TRAIN_DF = merged_df[merged_df["split"] == "train"].copy()

print("Training images:", len(TRAIN_DF))

# ============================================================
# BUILD IMAGE LOOKUP
# ============================================================
image_lookup = {}

for root, dirs, files in os.walk(HAM_ROOT):
    for fname in files:
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            image_id = os.path.splitext(fname)[0]
            
            # Keep only one file per image_id
            if image_id not in image_lookup:
                image_lookup[image_id] = os.path.join(root, fname)

print("Unique image files indexed:", len(image_lookup))

# Check that every training image exists
missing_images = [
    image_id
    for image_id in TRAIN_DF["image_id"]
    if image_id not in image_lookup
]

print("Missing training images:", len(missing_images))

assert len(missing_images) == 0, "Some training images are missing!"

# ============================================================
# COMPUTE MEAN AND STD
# ============================================================
channel_sum = np.zeros(3, dtype=np.float64)
channel_sum_sq = np.zeros(3, dtype=np.float64)

total_pixels = 0

for image_id in tqdm(
    TRAIN_DF["image_id"],
    desc="Computing HAM normalization statistics"
):
    
    path = image_lookup[image_id]
    
    image = Image.open(path).convert("RGB")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    
    arr = np.asarray(image, dtype=np.float32) / 255.0
    
    # H x W x C -> C x H x W
    arr = arr.transpose(2, 0, 1)
    
    pixels = arr.reshape(3, -1)
    
    channel_sum += pixels.sum(axis=1)
    channel_sum_sq += (pixels ** 2).sum(axis=1)
    
    total_pixels += pixels.shape[1]

# ============================================================
# FINAL STATISTICS
# ============================================================
mean = channel_sum / total_pixels

variance = (channel_sum_sq / total_pixels) - (mean ** 2)
variance = np.maximum(variance, 0)

std = np.sqrt(variance)

print("\n" + "=" * 60)
print("HAM10000 TRAINING-SET NORMALIZATION")
print("=" * 60)

print("Images:", len(TRAIN_DF))
print("Image size:", f"{IMAGE_SIZE}x{IMAGE_SIZE}")
print("Total pixels per channel:", total_pixels)

print("\nMean:")
print(mean)

print("\nStd:")
print(std)

# ============================================================
# SAVE PERMANENTLY
# ============================================================
norm_path = os.path.join(
    CHECKPOINT_ROOT,
    "splits",
    "ham_normalization_stats.txt"
)

with open(norm_path, "w") as f:
    f.write("HAM10000 normalization statistics\n")
    f.write("Computed from locked lesion-level TRAIN split only\n")
    f.write(f"Number of training images: {len(TRAIN_DF)}\n")
    f.write(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}\n")
    f.write(f"Total pixels per channel: {total_pixels}\n")
    f.write(f"Mean: {mean.tolist()}\n")
    f.write(f"Std: {std.tolist()}\n")

print("\nSaved to:")
print(norm_path)

print("\n✓ STEP 7 COMPLETE")

Training images: 6981
Unique image files indexed: 10015
Missing training images: 0


Computing HAM normalization statistics:   0%|          | 0/6981 [00:00<?, ?it/s]


HAM10000 TRAINING-SET NORMALIZATION
Images: 6981
Image size: 224x224
Total pixels per channel: 350278656

Mean:
[0.76322653 0.54579144 0.57025727]

Std:
[0.13989648 0.15306559 0.17014335]

Saved to:
/kaggle/working/biaspruner_checkpoints/splits/ham_normalization_stats.txt

✓ STEP 7 COMPLETE


In [17]:
# ============================================================
# STEP 8 — CUSTOM TASK DEFINITIONS + DATASETS + DATALOADERS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ============================================================
# 1. CUSTOM TASK ORDER
# ============================================================

TASK_DEFINITIONS = {
    1: ["bkl", "mel"],
    2: ["bcc", "akiec"],
    3: ["nv", "vasc", "df"],
}

print("=" * 70)
print("CUSTOM TASK DEFINITIONS")
print("=" * 70)

for task_id, classes in TASK_DEFINITIONS.items():
    n = merged_df[merged_df["dx"].isin(classes)].shape[0]
    print(f"Task {task_id}: {classes} -> {n} images")


# ============================================================
# 2. SAVE TASK DEFINITIONS
# ============================================================

task_def_path = os.path.join(
    CHECKPOINT_ROOT,
    "splits",
    "task_definitions.txt"
)

with open(task_def_path, "w") as f:
    f.write("BiasPruner HAM10000 Custom Experimental Task Definitions\n")
    f.write("=" * 60 + "\n")
    f.write("NOTE: This is a custom experimental task order.\n")
    f.write("The paper specifies 3 tasks with class counts [2,2,3],\n")
    f.write("but does not specify the exact HAM10000 class assignment\n")
    f.write("used here.\n\n")

    for task_id, classes in TASK_DEFINITIONS.items():
        n = merged_df[merged_df["dx"].isin(classes)].shape[0]
        f.write(
            f"Task {task_id}: {classes} | "
            f"Total images: {n}\n"
        )

print("\nTask definitions saved:")
print(task_def_path)


# ============================================================
# 3. FROZEN HAM NORMALIZATION
# ============================================================

HAM_MEAN = [0.76322653, 0.54579144, 0.57025727]
HAM_STD  = [0.13989648, 0.15306559, 0.17014335]

IMAGE_SIZE = 224

# Training augmentation:
# - Resize to 224x224
# - Random horizontal flip
# - Dataset-specific normalization
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(HAM_MEAN, HAM_STD),
])

# Validation/test:
# No random augmentation
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(HAM_MEAN, HAM_STD),
])


# ============================================================
# 4. DATASET CLASS
# ============================================================

class HAM10000TaskDataset(Dataset):

    def __init__(self, dataframe, image_lookup, classes, transform=None):

        self.df = dataframe[
            dataframe["dx"].isin(classes)
        ].reset_index(drop=True)

        self.image_lookup = image_lookup
        self.classes = list(classes)
        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(self.classes)
        }
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_id = row["image_id"]
        image_path = self.image_lookup[image_id]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = self.class_to_idx[row["dx"]]

        # Age group is retained for fairness evaluation.
        age_group = row["age_group"]

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "age_group": age_group,
            "image_id": image_id,
            "class_name": row["dx"],
        }


# ============================================================
# 5. HELPER TO BUILD LOADERS
# ============================================================

def build_task_loaders(task_id, batch_size=32):

    classes = TASK_DEFINITIONS[task_id]

    task_df = merged_df[
        merged_df["dx"].isin(classes)
    ].copy()

    train_df = task_df[
        task_df["split"] == "train"
    ].copy()

    val_df = task_df[
        task_df["split"] == "val"
    ].copy()

    test_df = task_df[
        task_df["split"] == "test"
    ].copy()

    train_dataset = HAM10000TaskDataset(
        train_df,
        image_lookup,
        classes,
        transform=train_transform
    )

    val_dataset = HAM10000TaskDataset(
        val_df,
        image_lookup,
        classes,
        transform=eval_transform
    )

    test_dataset = HAM10000TaskDataset(
        test_df,
        image_lookup,
        classes,
        transform=eval_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    return {
        "classes": classes,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "test_dataset": test_dataset,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }


# ============================================================
# 6. BUILD TASK 1
# ============================================================

task1 = build_task_loaders(1, batch_size=32)


# ============================================================
# 7. BUILD TASK 2
# ============================================================

task2 = build_task_loaders(2, batch_size=32)


# ============================================================
# 8. PRINT TASK DISTRIBUTIONS
# ============================================================

def print_task_summary(task_id, task):

    print("\n" + "=" * 70)
    print(f"TASK {task_id} SUMMARY")
    print("=" * 70)

    print("Classes:", task["classes"])

    print("\nTotal:")
    print(
        task["train_df"].shape[0]
        + task["val_df"].shape[0]
        + task["test_df"].shape[0]
    )

    print("\nSplit sizes:")
    print(
        "Train:", len(task["train_df"]),
        "| Val:", len(task["val_df"]),
        "| Test:", len(task["test_df"])
    )

    print("\nClass distribution:")
    print(
        task["train_df"]["dx"].value_counts().sort_index()
    )

    print("\nSplit × class:")
    print(
        pd.crosstab(
            task["train_df"]["dx"],
            task["train_df"]["split"]
        )
    )

    print("\nAge groups:")
    print(
        task["train_df"]["age_group"].value_counts()
    )


print_task_summary(1, task1)
print_task_summary(2, task2)


# ============================================================
# 9. VERIFY TASKS ARE DISJOINT
# ============================================================

task1_ids = set(
    merged_df[
        merged_df["dx"].isin(TASK_DEFINITIONS[1])
    ]["image_id"]
)

task2_ids = set(
    merged_df[
        merged_df["dx"].isin(TASK_DEFINITIONS[2])
    ]["image_id"]
)

overlap = task1_ids.intersection(task2_ids)

print("\n" + "=" * 70)
print("TASK DISJOINTNESS CHECK")
print("=" * 70)

print("Task 1 images:", len(task1_ids))
print("Task 2 images:", len(task2_ids))
print("Overlap:", len(overlap))

assert len(overlap) == 0

print("\n✓ Task 1 and Task 2 are completely disjoint.")


# ============================================================
# 10. TEST ONE BATCH
# ============================================================

batch = next(iter(task1["train_loader"]))

print("\n" + "=" * 70)
print("TASK 1 BATCH CHECK")
print("=" * 70)

print("Image shape:", batch["image"].shape)
print("Image dtype:", batch["image"].dtype)
print("Label shape:", batch["label"].shape)
print("Labels:", batch["label"][:10].tolist())
print("Age groups:", batch["age_group"][:10])
print("Image IDs:", batch["image_id"][:3])

print("\n✓ STEP 8 COMPLETE")

CUSTOM TASK DEFINITIONS
Task 1: ['bkl', 'mel'] -> 2212 images
Task 2: ['bcc', 'akiec'] -> 841 images
Task 3: ['nv', 'vasc', 'df'] -> 6962 images

Task definitions saved:
/kaggle/working/biaspruner_checkpoints/splits/task_definitions.txt

TASK 1 SUMMARY
Classes: ['bkl', 'mel']

Total:
2212

Split sizes:
Train: 1545 | Val: 441 | Test: 226

Class distribution:
dx
bkl    772
mel    773
Name: count, dtype: int64

Split × class:
split  train
dx          
bkl      772
mel      773

Age groups:
age_group
age>=60    997
age<60     541
missing      7
Name: count, dtype: int64

TASK 2 SUMMARY
Classes: ['bcc', 'akiec']

Total:
841

Split sizes:
Train: 583 | Val: 178 | Test: 80

Class distribution:
dx
akiec    222
bcc      361
Name: count, dtype: int64

Split × class:
split  train
dx          
akiec    222
bcc      361

Age groups:
age_group
age>=60    453
age<60     130
Name: count, dtype: int64

TASK DISJOINTNESS CHECK
Task 1 images: 2212
Task 2 images: 841
Overlap: 0

✓ Task 1 and Task 2 are com

In [18]:
# ============================================================
# STEP 9 — GCE SMOKE TEST ON NEW TASK 1
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
GCE_Q = 0.7
SMOKE_EPOCHS = 3
LR = 1e-4

print("=" * 70)
print("GCE SMOKE TEST — TASK 1 (bkl + mel)")
print("=" * 70)

# ------------------------------------------------------------
# FRESH RESNET-50
# ------------------------------------------------------------
smoke_model = resnet50(weights=None)

num_features = smoke_model.fc.in_features
smoke_model.fc = nn.Linear(
    num_features,
    2
)

smoke_model = smoke_model.to(device)

optimizer = optim.Adam(
    smoke_model.parameters(),
    lr=LR
)

# ------------------------------------------------------------
# GCE
# ------------------------------------------------------------
def gce_loss(logits, targets, q=0.7):

    probabilities = torch.softmax(logits, dim=1)

    p_y = probabilities[
        torch.arange(targets.size(0), device=targets.device),
        targets
    ]

    loss = (1.0 - p_y.pow(q)) / q

    return loss.mean()


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------
for epoch in range(SMOKE_EPOCHS):

    smoke_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in task1["train_loader"]:

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = smoke_model(images)

        loss = gce_loss(
            logits,
            labels,
            q=GCE_Q
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(
        f"Epoch {epoch+1:02d}/{SMOKE_EPOCHS} | "
        f"GCE Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )


# ------------------------------------------------------------
# TRAINING PREDICTION DISTRIBUTION
# ------------------------------------------------------------
smoke_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in task1["train_loader"]:

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        logits = smoke_model(images)

        predictions = logits.argmax(dim=1)

        all_predictions.append(
            predictions.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

print("\n" + "=" * 70)
print("FINAL TRAINING PREDICTION DISTRIBUTION")
print("=" * 70)

unique_pred, pred_counts = torch.unique(
    all_predictions,
    return_counts=True
)

for cls, count in zip(unique_pred, pred_counts):
    print(
        f"Class {cls.item()}: {count.item()}"
    )

print("\nTrue class distribution:")

unique_true, true_counts = torch.unique(
    all_labels,
    return_counts=True
)

for cls, count in zip(unique_true, true_counts):
    print(
        f"Class {cls.item()}: {count.item()}"
    )

# ------------------------------------------------------------
# PER-CLASS ACCURACY
# ------------------------------------------------------------
print("\nPer-class training accuracy:")

for cls in [0, 1]:

    mask = all_labels == cls

    cls_acc = (
        (all_predictions[mask] == cls).float().mean().item()
    )

    print(
        f"Class {cls}: {cls_acc:.4f}"
    )

print("\n✓ STEP 9 COMPLETE")

GCE SMOKE TEST — TASK 1 (bkl + mel)
Epoch 01/3 | GCE Loss: 0.4678 | Accuracy: 0.6350
Epoch 02/3 | GCE Loss: 0.3842 | Accuracy: 0.7107
Epoch 03/3 | GCE Loss: 0.3281 | Accuracy: 0.7476

FINAL TRAINING PREDICTION DISTRIBUTION
Class 0: 1286
Class 1: 259

True class distribution:
Class 0: 772
Class 1: 773

Per-class training accuracy:
Class 0: 0.9728
Class 1: 0.3079

✓ STEP 9 COMPLETE


In [19]:
# ============================================================
# STEP 10 — TASK 1 REAL GCE TRAINING
# ============================================================

import os
import json
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import resnet50


# ============================================================
# CONFIG
# ============================================================

TASK_ID = 1

GCE_Q = 0.7
MAX_EPOCHS = 15
LEARNING_RATE = 1e-4
BATCH_SIZE = 32

# Early stopping
PATIENCE = 3
MIN_DELTA = 1e-4

TASK1_DIR = os.path.join(
    CHECKPOINT_ROOT,
    "task1"
)

os.makedirs(TASK1_DIR, exist_ok=True)

print("=" * 70)
print("TASK 1 — REAL GCE TRAINING")
print("=" * 70)

print("Classes:", task1["classes"])
print("GCE q:", GCE_Q)
print("Max epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Early stopping patience:", PATIENCE)


# ============================================================
# MODEL
# ============================================================

model_task1 = resnet50(weights=None)

num_features = model_task1.fc.in_features

model_task1.fc = nn.Linear(
    num_features,
    len(task1["classes"])
)

model_task1 = model_task1.to(device)

print("\nModel:")
print("ResNet-50")
print("Output classes:", len(task1["classes"]))


# ============================================================
# OPTIMIZER
# ============================================================

optimizer_task1 = optim.Adam(
    model_task1.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# GCE LOSS
# ============================================================

def generalized_cross_entropy(
    logits,
    targets,
    q=0.7
):

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    p_y = probabilities[
        torch.arange(
            targets.size(0),
            device=targets.device
        ),
        targets
    ]

    loss = (
        1.0 - p_y.pow(q)
    ) / q

    return loss.mean()


# ============================================================
# VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_gce_model(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_predictions = []
    all_labels = []

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        logits = model(images)

        loss = generalized_cross_entropy(
            logits,
            labels,
            q=GCE_Q
        )

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

        all_predictions.append(
            predictions.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

    avg_loss = total_loss / total_samples

    accuracy = total_correct / total_samples

    all_predictions = torch.cat(
        all_predictions
    )

    all_labels = torch.cat(
        all_labels
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "predictions": all_predictions,
        "labels": all_labels
    }


# ============================================================
# TRAINING LOOP
# ============================================================

history = []

best_val_acc = -np.inf
best_epoch = 0
best_state = None

epochs_without_improvement = 0


for epoch in range(1, MAX_EPOCHS + 1):

    model_task1.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in task1["train_loader"]:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer_task1.zero_grad(
            set_to_none=True
        )

        logits = model_task1(images)

        loss = generalized_cross_entropy(
            logits,
            labels,
            q=GCE_Q
        )

        loss.backward()

        optimizer_task1.step()

        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += batch_size

    train_loss = running_loss / total
    train_accuracy = correct / total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_result = evaluate_gce_model(
        model_task1,
        task1["val_loader"]
    )

    val_loss = val_result["loss"]
    val_accuracy = val_result["accuracy"]


    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    }

    history.append(row)


    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    improved = (
        val_accuracy >
        best_val_acc + MIN_DELTA
    )

    if improved:

        best_val_acc = val_accuracy
        best_epoch = epoch

        best_state = copy.deepcopy(
            model_task1.state_dict()
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_state,
                "optimizer_state_dict":
                    optimizer_task1.state_dict(),
                "classes": task1["classes"],
                "gce_q": GCE_Q,
                "learning_rate": LEARNING_RATE,
                "best_val_accuracy":
                    best_val_acc
            },
            os.path.join(
                TASK1_DIR,
                "task1_gce_best.pt"
            )
        )

    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

assert best_state is not None

model_task1.load_state_dict(
    best_state
)

print("\n" + "=" * 70)
print("BEST TASK 1 MODEL")
print("=" * 70)

print("Best epoch:", best_epoch)
print("Best validation accuracy:", best_val_acc)


# ============================================================
# SAVE FINAL/BEST MODEL
# ============================================================

final_checkpoint_path = os.path.join(
    TASK1_DIR,
    "task1_gce_final.pt"
)

torch.save(
    {
        "task_id": TASK_ID,
        "classes": task1["classes"],
        "model_state_dict":
            model_task1.state_dict(),
        "optimizer_state_dict":
            optimizer_task1.state_dict(),
        "best_epoch": best_epoch,
        "best_val_accuracy":
            best_val_acc,
        "gce_q": GCE_Q,
        "max_epochs": MAX_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "normalization_mean": HAM_MEAN,
        "normalization_std": HAM_STD
    },
    final_checkpoint_path
)


# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(history)

history_path = os.path.join(
    TASK1_DIR,
    "task1_gce_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)


# ============================================================
# SAVE CONFIG
# ============================================================

config = {
    "task_id": TASK_ID,
    "classes": task1["classes"],
    "gce_q": GCE_Q,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "early_stopping_patience": PATIENCE,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_acc,
    "normalization_mean": HAM_MEAN,
    "normalization_std": HAM_STD
}

config_path = os.path.join(
    TASK1_DIR,
    "task1_gce_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nSaved files:")

print(
    "Best checkpoint:",
    os.path.join(
        TASK1_DIR,
        "task1_gce_best.pt"
    )
)

print(
    "Final checkpoint:",
    final_checkpoint_path
)

print(
    "Training history:",
    history_path
)

print(
    "Configuration:",
    config_path
)

print("\n✓ STEP 10 COMPLETE")

TASK 1 — REAL GCE TRAINING
Classes: ['bkl', 'mel']
GCE q: 0.7
Max epochs: 15
Learning rate: 0.0001
Batch size: 32
Early stopping patience: 3

Model:
ResNet-50
Output classes: 2
Epoch 001/15 | Train Loss: 0.4326 | Train Acc: 0.6621 | Val Loss: 0.3743 | Val Acc: 0.7029
Epoch 002/15 | Train Loss: 0.3755 | Train Acc: 0.7204 | Val Loss: 0.2856 | Val Acc: 0.7823
Epoch 003/15 | Train Loss: 0.3279 | Train Acc: 0.7495 | Val Loss: 0.4634 | Val Acc: 0.6304
Epoch 004/15 | Train Loss: 0.3304 | Train Acc: 0.7502 | Val Loss: 0.2780 | Val Acc: 0.7937
Epoch 005/15 | Train Loss: 0.3161 | Train Acc: 0.7612 | Val Loss: 0.2668 | Val Acc: 0.7914
Epoch 006/15 | Train Loss: 0.2983 | Train Acc: 0.7786 | Val Loss: 0.2734 | Val Acc: 0.7914
Epoch 007/15 | Train Loss: 0.2843 | Train Acc: 0.7896 | Val Loss: 0.2774 | Val Acc: 0.7914

Early stopping at epoch 7.

BEST TASK 1 MODEL
Best epoch: 4
Best validation accuracy: 0.7936507936507936

Saved files:
Best checkpoint: /kaggle/working/biaspruner_checkpoints/task1/task

In [20]:
# ============================================================
# STEP 11A — EASY / HARD SAMPLE PARTITION DIAGNOSTIC
# ============================================================

import torch
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

TAU = 0.70

print("=" * 70)
print("TASK 1 — EASY / HARD SAMPLE PARTITION")
print("=" * 70)

# ------------------------------------------------------------
# LOAD BEST GCE CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_gce_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

# Load best model weights
model_task1.load_state_dict(
    checkpoint["model_state_dict"]
)

model_task1.eval()

print("Loaded checkpoint:")
print(checkpoint_path)
print("Best epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["best_val_accuracy"])

# ------------------------------------------------------------
# COLLECT TRAINING PREDICTIONS
# ------------------------------------------------------------

easy_samples = defaultdict(list)
hard_samples = defaultdict(list)

total_samples = 0
correct_samples = 0

with torch.no_grad():

    for batch in task1["train_loader"]:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        logits = model_task1(images)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        confidence = probabilities[
            torch.arange(
                labels.size(0),
                device=device
            ),
            labels
        ]

        predictions = logits.argmax(
            dim=1
        )

        for i in range(labels.size(0)):

            label = labels[i].item()
            prediction = predictions[i].item()
            conf = confidence[i].item()
            image_id = batch["image_id"][i]

            total_samples += 1

            if prediction == label:

                correct_samples += 1

                # Easy = correct + high confidence
                if conf >= TAU:
                    easy_samples[label].append({
                        "image_id": image_id,
                        "confidence": conf
                    })

            else:

                # Hard = incorrect + high confidence
                if conf >= TAU:
                    hard_samples[label].append({
                        "image_id": image_id,
                        "confidence": conf
                    })


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("\nTotal training samples:", total_samples)
print("Correct predictions:", correct_samples)
print(
    "Training accuracy:",
    correct_samples / total_samples
)

print("\nThreshold tau:", TAU)

print("\n" + "-" * 70)
print("EASY SAMPLES")
print("-" * 70)

for class_idx, class_name in enumerate(task1["classes"]):

    count = len(
        easy_samples[class_idx]
    )

    print(
        f"Class {class_name} "
        f"(label {class_idx}): {count}"
    )


print("\n" + "-" * 70)
print("HARD SAMPLES")
print("-" * 70)

for class_idx, class_name in enumerate(task1["classes"]):

    count = len(
        hard_samples[class_idx]
    )

    print(
        f"Class {class_name} "
        f"(label {class_idx}): {count}"
    )


# ------------------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EASY / HARD SUMMARY")
print("=" * 70)

for class_idx, class_name in enumerate(task1["classes"]):

    easy_count = len(
        easy_samples[class_idx]
    )

    hard_count = len(
        hard_samples[class_idx]
    )

    print(
        f"{class_name:>5} | "
        f"Easy: {easy_count:4d} | "
        f"Hard: {hard_count:4d}"
    )

# ------------------------------------------------------------
# SANITY CHECK
# ------------------------------------------------------------

for class_idx in range(len(task1["classes"])):

    assert (
        len(easy_samples[class_idx]) > 0
    ), f"No easy samples for class {class_idx}"

    assert (
        len(hard_samples[class_idx]) > 0
    ), f"No hard samples for class {class_idx}"

print("\n✓ STEP 11A COMPLETE")

TASK 1 — EASY / HARD SAMPLE PARTITION
Loaded checkpoint:
/kaggle/working/biaspruner_checkpoints/task1/task1_gce_best.pt
Best epoch: 4
Best validation accuracy: 0.7936507936507936

Total training samples: 1545
Correct predictions: 1193
Training accuracy: 0.772168284789644

Threshold tau: 0.7

----------------------------------------------------------------------
EASY SAMPLES
----------------------------------------------------------------------
Class bkl (label 0): 482
Class mel (label 1): 661

----------------------------------------------------------------------
HARD SAMPLES
----------------------------------------------------------------------
Class bkl (label 0): 0
Class mel (label 1): 0

EASY / HARD SUMMARY
  bkl | Easy:  482 | Hard:    0
  mel | Easy:  661 | Hard:    0


AssertionError: No hard samples for class 0

In [21]:
# ============================================================
# STEP 11B — INSPECT MISCLASSIFIED SAMPLE CONFIDENCES
# ============================================================

model_task1.eval()

wrong_confidences = []

with torch.no_grad():

    for batch in task1["train_loader"]:

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        logits = model_task1(images)

        probabilities = torch.softmax(logits, dim=1)

        predictions = logits.argmax(dim=1)

        true_confidence = probabilities[
            torch.arange(labels.size(0), device=device),
            labels
        ]

        wrong_mask = predictions != labels

        if wrong_mask.any():

            wrong_confidences.extend(
                true_confidence[wrong_mask]
                .cpu()
                .numpy()
                .tolist()
            )

wrong_confidences = np.array(wrong_confidences)

print("=" * 70)
print("MISCLASSIFIED SAMPLE CONFIDENCE")
print("=" * 70)

print("Number of misclassified samples:", len(wrong_confidences))

if len(wrong_confidences) > 0:

    print("Minimum:", wrong_confidences.min())
    print("Maximum:", wrong_confidences.max())
    print("Mean:", wrong_confidences.mean())
    print("Median:", np.median(wrong_confidences))

    print("\nPercentiles:")
    for p in [25, 50, 75, 90, 95]:
        print(
            f"{p}th percentile:",
            np.percentile(wrong_confidences, p)
        )

    for tau in [0.50, 0.60, 0.70]:

        count = np.sum(
            wrong_confidences >= tau
        )

        print(
            f"\nWrong samples with true-class confidence >= {tau}: "
            f"{count}"
        )

else:

    print("No misclassified samples.")

print("\n✓ STEP 11B COMPLETE")

MISCLASSIFIED SAMPLE CONFIDENCE
Number of misclassified samples: 364
Minimum: 1.0014106731348615e-11
Maximum: 0.499142587184906
Mean: 0.08662561958365805
Median: 0.009854720439761877

Percentiles:
25th percentile: 0.000561160093639046
50th percentile: 0.009854720439761877
75th percentile: 0.11232244223356247
90th percentile: 0.34412690699100473
95th percentile: 0.41599307805299757

Wrong samples with true-class confidence >= 0.5: 0

Wrong samples with true-class confidence >= 0.6: 0

Wrong samples with true-class confidence >= 0.7: 0

✓ STEP 11B COMPLETE


In [22]:
# ============================================================
# STEP 8 — CHECK TASK 1 CHECKPOINT + CURRENT NOTEBOOK STATE
# ============================================================

import os
import glob
import torch

print("=" * 70)
print("BIASPRUNER — TASK 1 CHECKPOINT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check checkpoint directory
# ------------------------------------------------------------
checkpoint_root = "/kaggle/working/biaspruner_checkpoints"

print("\n[1] Checkpoint root:")
print(checkpoint_root)
print("Exists:", os.path.exists(checkpoint_root))

if os.path.exists(checkpoint_root):
    print("\nCheckpoint files:")
    files = glob.glob(checkpoint_root + "/**/*", recursive=True)

    if len(files) == 0:
        print("  No files found.")
    else:
        for f in files:
            if os.path.isfile(f):
                size_mb = os.path.getsize(f) / (1024 * 1024)
                print(f"  {f}   ({size_mb:.2f} MB)")

# ------------------------------------------------------------
# 2. Look specifically for Task 1 checkpoints
# ------------------------------------------------------------
print("\n[2] Task 1 checkpoint candidates:")

task1_candidates = []

for pattern in [
    "/kaggle/working/biaspruner_checkpoints/task1/**/*",
    "/kaggle/working/biaspruner_checkpoints/**/*task1*",
    "/kaggle/working/**/*task1*.pt",
    "/kaggle/working/**/*task1*.pth",
    "/kaggle/working/**/*.pt",
    "/kaggle/working/**/*.pth",
]:
    task1_candidates.extend(
        glob.glob(pattern, recursive=True)
    )

# remove duplicates
task1_candidates = sorted(set(task1_candidates))

if len(task1_candidates) == 0:
    print("  No .pt/.pth Task 1 checkpoint found.")
else:
    for f in task1_candidates:
        if os.path.isfile(f):
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  {f}   ({size_mb:.2f} MB)")

# ------------------------------------------------------------
# 3. Check common model variables
# ------------------------------------------------------------
print("\n[3] Common model variables currently in notebook:")

common_model_names = [
    "model",
    "task1_model",
    "best_model",
    "gce_model",
    "model_task1",
    "net",
]

found_models = []

for name in common_model_names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, torch.nn.Module):
            found_models.append(name)
            print(f"  ✓ {name} -> {type(obj).__name__}")

if len(found_models) == 0:
    print("  No common PyTorch model variable detected.")

# ------------------------------------------------------------
# 4. Check common DataLoader variables
# ------------------------------------------------------------
print("\n[4] Common DataLoader variables:")

common_loader_names = [
    "train_loader",
    "val_loader",
    "test_loader",
    "task1_train_loader",
    "task1_val_loader",
    "task1_test_loader",
]

found_loaders = []

for name in common_loader_names:
    if name in globals():
        obj = globals()[name]

        # DataLoader has batch_size / dataset attributes
        if hasattr(obj, "dataset"):
            found_loaders.append(name)

            try:
                dataset_len = len(obj.dataset)
            except Exception:
                dataset_len = "unknown"

            print(
                f"  ✓ {name} "
                f"(dataset size={dataset_len}, "
                f"batch_size={getattr(obj, 'batch_size', 'unknown')})"
            )

if len(found_loaders) == 0:
    print("  No common DataLoader variable detected.")

# ------------------------------------------------------------
# 5. Check important Task 1 variables
# ------------------------------------------------------------
print("\n[5] Important variables:")

important_names = [
    "TASK1_CLASSES",
    "task1_classes",
    "TASK1",
    "task1",
    "q",
    "tau",
    "gamma",
    "device",
    "best_epoch",
    "best_val_acc",
]

for name in important_names:
    if name in globals():
        try:
            print(f"  {name} = {globals()[name]}")
        except Exception:
            print(f"  {name} = <unable to print>")

# ------------------------------------------------------------
# 6. Check normalization file
# ------------------------------------------------------------
norm_file = (
    "/kaggle/working/biaspruner_checkpoints/"
    "splits/ham_normalization_stats.txt"
)

print("\n[6] Normalization statistics:")
print("Path:", norm_file)
print("Exists:", os.path.exists(norm_file))

if os.path.exists(norm_file):
    print("\nContents:")
    with open(norm_file, "r") as f:
        print(f.read())

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)
print("\nDo NOT start another training run yet.")

BIASPRUNER — TASK 1 CHECKPOINT AUDIT

[1] Checkpoint root:
/kaggle/working/biaspruner_checkpoints
Exists: True

Checkpoint files:
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_best.pt   (269.51 MB)
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_config.json   (0.00 MB)
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_final.pt   (269.51 MB)
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_history.csv   (0.00 MB)
  /kaggle/working/biaspruner_checkpoints/splits/split_config.txt   (0.00 MB)
  /kaggle/working/biaspruner_checkpoints/splits/ham_normalization_stats.txt   (0.00 MB)
  /kaggle/working/biaspruner_checkpoints/splits/ham10000_lesion_split_seed42.csv   (0.66 MB)
  /kaggle/working/biaspruner_checkpoints/splits/task_definitions.txt   (0.00 MB)

[2] Task 1 checkpoint candidates:
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_best.pt   (269.51 MB)
  /kaggle/working/biaspruner_checkpoints/task1/task1_gce_config.json   (0.00 MB)
  /kaggle/worki

In [24]:
# ============================================================
# STEP 8B — CORRECT TASK 1 DATASETS + LOAD BEST GCE CHECKPOINT
# ============================================================

import os
import torch
from torch.utils.data import DataLoader

print("=" * 70)
print("BIASPRUNER — TASK 1 CHECKPOINT + DATASET VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Task 1 definition
# ------------------------------------------------------------

TASK1_CLASSES = ["bkl", "mel"]

print("\nTask 1 classes:", TASK1_CLASSES)

# ------------------------------------------------------------
# 2. Task 1 dataframes
# ------------------------------------------------------------

task1_train_df = task1["train_df"].copy()
task1_val_df   = task1["val_df"].copy()
task1_test_df  = task1["test_df"].copy()

print("\nTask 1 dataframe sizes:")
print("Train:", len(task1_train_df))
print("Val  :", len(task1_val_df))
print("Test :", len(task1_test_df))

print("\nTrain class distribution:")
print(task1_train_df["dx"].value_counts())

print("\nValidation class distribution:")
print(task1_val_df["dx"].value_counts())

print("\nTest class distribution:")
print(task1_test_df["dx"].value_counts())

# ------------------------------------------------------------
# 3. Check image_lookup
# ------------------------------------------------------------

print("\nChecking image_lookup...")

assert "image_lookup" in globals(), (
    "image_lookup is not available in the notebook."
)

print("image_lookup type:", type(image_lookup))
print("image_lookup entries:", len(image_lookup))

# ------------------------------------------------------------
# 4. Recreate Task 1 datasets
# ------------------------------------------------------------

task1_train_dataset = HAM10000TaskDataset(
    task1_train_df,
    image_lookup,
    TASK1_CLASSES,
    transform=train_transform
)

task1_val_dataset = HAM10000TaskDataset(
    task1_val_df,
    image_lookup,
    TASK1_CLASSES,
    transform=val_transform
)

task1_test_dataset = HAM10000TaskDataset(
    task1_test_df,
    image_lookup,
    TASK1_CLASSES,
    transform=val_transform
)

# ------------------------------------------------------------
# 5. Recreate DataLoaders
# ------------------------------------------------------------

task1_train_loader = DataLoader(
    task1_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

task1_val_loader = DataLoader(
    task1_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

task1_test_loader = DataLoader(
    task1_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nCorrect Task 1 loaders:")
print("Train:", len(task1_train_loader.dataset))
print("Val  :", len(task1_val_loader.dataset))
print("Test :", len(task1_test_loader.dataset))

# ------------------------------------------------------------
# 6. Verify one batch
# ------------------------------------------------------------

batch = next(iter(task1_train_loader))

print("\nOne training batch:")

assert isinstance(batch, dict), (
    f"Expected dictionary batch, got {type(batch)}"
)

print("Keys:", list(batch.keys()))
print("Images:", batch["image"].shape)
print("Labels:", batch["label"].shape)

print(
    "Unique labels:",
    torch.unique(batch["label"]).cpu().tolist()
)

print(
    "First image IDs:",
    batch["image_id"][:5]
)

print(
    "First age groups:",
    batch["age_group"][:5]
)

# ------------------------------------------------------------
# 7. Load BEST Task 1 GCE checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_gce_best.pt"
)

print("\nCheckpoint:")
print(checkpoint_path)
print("Exists:", os.path.exists(checkpoint_path))

assert os.path.exists(checkpoint_path)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    print(list(checkpoint.keys()))

# ------------------------------------------------------------
# 8. Load checkpoint into model_task1
# ------------------------------------------------------------

assert "model_task1" in globals()

model_task1 = model_task1.to(device)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    elif (
        "model" in checkpoint
        and isinstance(checkpoint["model"], dict)
    ):
        state_dict = checkpoint["model"]

    else:
        state_dict = checkpoint

else:
    raise RuntimeError(
        "Unexpected checkpoint format."
    )

# Remove DataParallel prefix if present
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value

missing_keys, unexpected_keys = model_task1.load_state_dict(
    clean_state_dict,
    strict=False
)

print("\nCheckpoint loading result:")
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# ------------------------------------------------------------
# 9. Evaluation mode
# ------------------------------------------------------------

model_task1.eval()

# ------------------------------------------------------------
# 10. Forward-pass verification
# ------------------------------------------------------------

with torch.no_grad():

    images = batch["image"].to(device)
    labels = batch["label"].to(device)

    logits = model_task1(images)
    probs = torch.softmax(logits, dim=1)

    predictions = probs.argmax(dim=1)

batch_accuracy = (
    predictions == labels
).float().mean().item()

print("\nForward-pass verification:")
print("Images :", tuple(images.shape))
print("Logits :", tuple(logits.shape))
print("Probs  :", tuple(probs.shape))
print("Probability min:", float(probs.min()))
print("Probability max:", float(probs.max()))
print("Batch accuracy:", batch_accuracy)

print("\n" + "=" * 70)
print("STEP 8B COMPLETE")
print("=" * 70)

BIASPRUNER — TASK 1 CHECKPOINT + DATASET VERIFICATION

Task 1 classes: ['bkl', 'mel']

Task 1 dataframe sizes:
Train: 1545
Val  : 441
Test : 226

Train class distribution:
dx
mel    773
bkl    772
Name: count, dtype: int64

Validation class distribution:
dx
mel    226
bkl    215
Name: count, dtype: int64

Test class distribution:
dx
mel    114
bkl    112
Name: count, dtype: int64

Checking image_lookup...
image_lookup type: <class 'dict'>
image_lookup entries: 10015


NameError: name 'val_transform' is not defined

In [25]:
# ============================================================
# STEP 8C — RECREATE THE LOCKED PAPER PREPROCESSING
#             + BUILD CORRECT TASK 1 LOADERS
# ============================================================

import os
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

print("=" * 70)
print("BIASPRUNER — TASK 1 PREPROCESSING + DATA LOADERS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOCKED HAM10000 normalization statistics
# ------------------------------------------------------------

HAM_MEAN = [
    0.7632265267001987,
    0.5457914364232362,
    0.5702572717523297
]

HAM_STD = [
    0.13989647800631033,
    0.15306559321188856,
    0.17014335417943227
]

IMAGE_SIZE = 224

# ------------------------------------------------------------
# 2. Paper preprocessing
#
# Training:
#   Resize -> Random Horizontal Flip -> Tensor -> Normalize
#
# Validation/Test:
#   Resize -> Tensor -> Normalize
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=HAM_MEAN,
        std=HAM_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=HAM_MEAN,
        std=HAM_STD
    )
])

print("\nTransforms created:")
print("  Train:", train_transform)
print("  Val/Test:", val_transform)

# ------------------------------------------------------------
# 3. Correct Task 1
# ------------------------------------------------------------

TASK1_CLASSES = ["bkl", "mel"]

task1_train_df = task1["train_df"].copy()
task1_val_df   = task1["val_df"].copy()
task1_test_df  = task1["test_df"].copy()

# ------------------------------------------------------------
# 4. Build datasets
# ------------------------------------------------------------

task1_train_dataset = HAM10000TaskDataset(
    task1_train_df,
    image_lookup,
    TASK1_CLASSES,
    transform=train_transform
)

task1_val_dataset = HAM10000TaskDataset(
    task1_val_df,
    image_lookup,
    TASK1_CLASSES,
    transform=val_transform
)

task1_test_dataset = HAM10000TaskDataset(
    task1_test_df,
    image_lookup,
    TASK1_CLASSES,
    transform=val_transform
)

# ------------------------------------------------------------
# 5. Build loaders
# ------------------------------------------------------------

task1_train_loader = DataLoader(
    task1_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

task1_val_loader = DataLoader(
    task1_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

task1_test_loader = DataLoader(
    task1_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nDataset sizes:")
print("  Train:", len(task1_train_dataset))
print("  Val  :", len(task1_val_dataset))
print("  Test :", len(task1_test_dataset))

# ------------------------------------------------------------
# 6. Verify a batch
# ------------------------------------------------------------

batch = next(iter(task1_train_loader))

print("\nBatch verification:")
print("  Keys:", list(batch.keys()))
print("  Image shape:", tuple(batch["image"].shape))
print("  Label shape:", tuple(batch["label"].shape))

print(
    "  Unique labels:",
    torch.unique(batch["label"]).cpu().tolist()
)

print(
    "  First image IDs:",
    batch["image_id"][:5]
)

print(
    "  First age groups:",
    batch["age_group"][:5]
)

# ------------------------------------------------------------
# 7. Load BEST Task 1 GCE checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_gce_best.pt"
)

assert os.path.exists(checkpoint_path), (
    f"Checkpoint not found: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

print("\nCheckpoint loaded:")
print("  Path:", checkpoint_path)
print("  Type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("  Keys:", list(checkpoint.keys()))

# ------------------------------------------------------------
# 8. Load state dict
# ------------------------------------------------------------

assert "model_task1" in globals(), (
    "model_task1 does not exist."
)

model_task1 = model_task1.to(device)

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    elif (
        "model" in checkpoint
        and isinstance(checkpoint["model"], dict)
    ):
        state_dict = checkpoint["model"]

    else:
        state_dict = checkpoint

else:
    raise RuntimeError(
        "Unexpected checkpoint format."
    )

# Remove DataParallel prefix if present
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value

missing_keys, unexpected_keys = model_task1.load_state_dict(
    clean_state_dict,
    strict=False
)

print("\nState-dict loading:")
print("  Missing keys:", missing_keys)
print("  Unexpected keys:", unexpected_keys)

# ------------------------------------------------------------
# 9. Evaluation mode
# ------------------------------------------------------------

model_task1.eval()

# ------------------------------------------------------------
# 10. Forward pass
# ------------------------------------------------------------

with torch.no_grad():

    images = batch["image"].to(device)
    labels = batch["label"].to(device)

    logits = model_task1(images)
    probs = torch.softmax(logits, dim=1)

    predictions = probs.argmax(dim=1)

accuracy = (
    predictions == labels
).float().mean().item()

print("\nForward-pass verification:")
print("  Images :", tuple(images.shape))
print("  Logits :", tuple(logits.shape))
print("  Probs  :", tuple(probs.shape))
print("  Probability min:", float(probs.min()))
print("  Probability max:", float(probs.max()))
print("  Batch accuracy:", accuracy)

print("\n" + "=" * 70)
print("STEP 8C COMPLETE")
print("=" * 70)

print("\nDO NOT RUN PRUNING YET.")

BIASPRUNER — TASK 1 PREPROCESSING + DATA LOADERS

Transforms created:
  Train: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.7632265267001987, 0.5457914364232362, 0.5702572717523297], std=[0.13989647800631033, 0.15306559321188856, 0.17014335417943227])
)
  Val/Test: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.7632265267001987, 0.5457914364232362, 0.5702572717523297], std=[0.13989647800631033, 0.15306559321188856, 0.17014335417943227])
)

Dataset sizes:
  Train: 1545
  Val  : 441
  Test : 226

Batch verification:
  Keys: ['image', 'label', 'age_group', 'image_id', 'class_name']
  Image shape: (32, 3, 224, 224)
  Label shape: (32,)
  Unique labels: [0, 1]
  First image IDs: ['ISIC_0033496', 'ISIC_0032514', 'ISIC_0034205', 'ISIC_0033528', 'ISIC_0033754']
  First age groups: ['age<60', 'age>=6

In [26]:
# ============================================================
# STEP 9 — BIASPRUNER SAMPLE PARTITION
#
# Repository-aligned:
#   S_ba = correctly classified
#   S_bc = incorrectly classified
#
# We then inspect them per ground-truth class.
# ============================================================

import torch
from collections import Counter

print("=" * 70)
print("BIASPRUNER — TASK 1 SAMPLE PARTITION")
print("=" * 70)

model_task1.eval()

# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

correct_indices = []
wrong_indices = []

correct_by_class = {
    0: [],
    1: []
}

wrong_by_class = {
    0: [],
    1: []
}

all_predictions = []
all_labels = []
all_probs = []
all_image_ids = []

# ------------------------------------------------------------
# Evaluate every Task 1 TRAINING sample
# ------------------------------------------------------------

with torch.no_grad():

    global_index = 0

    for batch_idx, batch in enumerate(task1_train_loader):

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model_task1(images)
        probs = torch.softmax(logits, dim=1)

        predictions = probs.argmax(dim=1)

        batch_size = labels.size(0)

        for j in range(batch_size):

            label = int(labels[j].item())
            pred = int(predictions[j].item())

            image_id = batch["image_id"][j]

            # Store probability assigned to TRUE class
            true_prob = float(
                probs[j, label].item()
            )

            all_labels.append(label)
            all_predictions.append(pred)
            all_probs.append(true_prob)
            all_image_ids.append(image_id)

            if pred == label:

                correct_indices.append(global_index)
                correct_by_class[label].append(global_index)

            else:

                wrong_indices.append(global_index)
                wrong_by_class[label].append(global_index)

            global_index += 1

# ------------------------------------------------------------
# Convert to tensors
# ------------------------------------------------------------

all_labels_tensor = torch.tensor(
    all_labels,
    dtype=torch.long
)

all_predictions_tensor = torch.tensor(
    all_predictions,
    dtype=torch.long
)

all_probs_tensor = torch.tensor(
    all_probs,
    dtype=torch.float32
)

# ------------------------------------------------------------
# Repository-aligned sets
# ------------------------------------------------------------

S_ba = set(correct_indices)
S_bc = set(wrong_indices)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

total = len(all_labels)

print("\nTotal Task 1 training samples:", total)

print("\nRepository-aligned partition:")
print("  S_ba (correct):", len(S_ba))
print("  S_bc (wrong)  :", len(S_bc))
print("  Total         :", len(S_ba) + len(S_bc))

print(
    "  Coverage      :",
    len(S_ba) + len(S_bc) == total
)

# ------------------------------------------------------------
# Overall accuracy
# ------------------------------------------------------------

overall_acc = (
    all_predictions_tensor == all_labels_tensor
).float().mean().item()

print(
    "\nTraining accuracy:",
    f"{overall_acc * 100:.2f}%"
)

# ------------------------------------------------------------
# Per-class statistics
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("PER-CLASS PARTITION")
print("-" * 70)

for class_id, class_name in enumerate(TASK1_CLASSES):

    class_mask = (
        all_labels_tensor == class_id
    )

    class_total = int(class_mask.sum().item())

    class_correct = len(
        correct_by_class[class_id]
    )

    class_wrong = len(
        wrong_by_class[class_id]
    )

    class_acc = (
        class_correct / class_total
        if class_total > 0
        else float("nan")
    )

    print(f"\nClass {class_id}: {class_name}")
    print("  Total   :", class_total)
    print("  Correct :", class_correct)
    print("  Wrong   :", class_wrong)
    print(
        "  Accuracy:",
        f"{class_acc * 100:.2f}%"
    )

# ------------------------------------------------------------
# Confidence statistics
# ------------------------------------------------------------

tau = 0.70

easy_high_conf_mask = (
    (all_predictions_tensor == all_labels_tensor)
    &
    (all_probs_tensor >= tau)
)

wrong_high_conf_mask = (
    (all_predictions_tensor != all_labels_tensor)
    &
    (all_probs_tensor >= tau)
)

print("\n" + "-" * 70)
print("CONFIDENCE DIAGNOSTIC")
print("-" * 70)

print(
    "\nCorrect AND true-class probability >= 0.70:",
    int(easy_high_conf_mask.sum().item())
)

print(
    "Wrong AND true-class probability >= 0.70:",
    int(wrong_high_conf_mask.sum().item())
)

if len(wrong_indices) > 0:

    wrong_probs = all_probs_tensor[
        all_predictions_tensor != all_labels_tensor
    ]

    print(
        "\nMaximum true-class probability among WRONG samples:",
        float(wrong_probs.max().item())
    )

    print(
        "Mean true-class probability among WRONG samples:",
        float(wrong_probs.mean().item())
    )

# ------------------------------------------------------------
# Per-class wrong-sample counts
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("WRONG SAMPLES BY GROUND-TRUTH CLASS")
print("-" * 70)

for class_id, class_name in enumerate(TASK1_CLASSES):

    wrong_mask = (
        (all_labels_tensor == class_id)
        &
        (all_predictions_tensor != class_id)
    )

    wrong_count = int(
        wrong_mask.sum().item()
    )

    print(
        f"  {class_name}: {wrong_count}"
    )

# ------------------------------------------------------------
# Store useful objects for next step
# ------------------------------------------------------------

partition_info = {
    "S_ba": S_ba,
    "S_bc": S_bc,
    "correct_by_class": correct_by_class,
    "wrong_by_class": wrong_by_class,
    "labels": all_labels_tensor,
    "predictions": all_predictions_tensor,
    "true_class_prob": all_probs_tensor,
    "image_ids": all_image_ids,
    "tau": tau,
}

print("\n" + "=" * 70)
print("STEP 9 COMPLETE")
print("=" * 70)

print("\nS_ba and S_bc are now stored in memory.")
print("NO PRUNING HAS BEEN PERFORMED.")
print("NO MODEL WEIGHTS HAVE BEEN CHANGED.")

BIASPRUNER — TASK 1 SAMPLE PARTITION

Total Task 1 training samples: 1545

Repository-aligned partition:
  S_ba (correct): 1187
  S_bc (wrong)  : 358
  Total         : 1545
  Coverage      : True

Training accuracy: 76.83%

----------------------------------------------------------------------
PER-CLASS PARTITION
----------------------------------------------------------------------

Class 0: bkl
  Total   : 772
  Correct : 511
  Wrong   : 261
  Accuracy: 66.19%

Class 1: mel
  Total   : 773
  Correct : 676
  Wrong   : 97
  Accuracy: 87.45%

----------------------------------------------------------------------
CONFIDENCE DIAGNOSTIC
----------------------------------------------------------------------

Correct AND true-class probability >= 0.70: 1146
Wrong AND true-class probability >= 0.70: 0

Maximum true-class probability among WRONG samples: 0.4994451403617859
Mean true-class probability among WRONG samples: 0.08244346082210541

----------------------------------------------------

In [27]:
# ============================================================
# STEP 10 — INSPECT RESNET-50 ACTIVATION STRUCTURE
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("BIASPRUNER — RESNET-50 ACTIVATION STRUCTURE")
print("=" * 70)

model_task1.eval()

# ------------------------------------------------------------
# 1. Basic model information
# ------------------------------------------------------------

print("\nModel type:")
print(type(model_task1))

print("\nTotal parameters:")
print(
    f"{sum(p.numel() for p in model_task1.parameters()):,}"
)

# ------------------------------------------------------------
# 2. Count convolution and ReLU modules
# ------------------------------------------------------------

conv_modules = []
relu_modules = []

for name, module in model_task1.named_modules():

    if isinstance(module, nn.Conv2d):
        conv_modules.append((name, module))

    elif isinstance(module, nn.ReLU):
        relu_modules.append((name, module))

print("\nModule counts:")
print("  Conv2d:", len(conv_modules))
print("  ReLU  :", len(relu_modules))

# ------------------------------------------------------------
# 3. Print convolutional units
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CONVOLUTIONAL LAYERS")
print("-" * 70)

total_conv_channels = 0

for name, module in conv_modules:

    channels = module.out_channels
    total_conv_channels += channels

    print(
        f"{name:45s} "
        f"out_channels={channels:4d} "
        f"kernel={module.kernel_size} "
        f"stride={module.stride}"
    )

print("\nTotal convolutional output channels:")
print(f"{total_conv_channels:,}")

# ------------------------------------------------------------
# 4. Print ReLU modules
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("RELU LAYERS")
print("-" * 70)

for name, module in relu_modules:
    print(name)

# ------------------------------------------------------------
# 5. Check model's classifier
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL CLASSIFIER")
print("-" * 70)

for name, module in model_task1.named_modules():

    if isinstance(module, nn.Linear):

        print(
            f"{name}: "
            f"in_features={module.in_features}, "
            f"out_features={module.out_features}"
        )

# ------------------------------------------------------------
# 6. Determine architecture family
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("ARCHITECTURE CHECK")
print("-" * 70)

if hasattr(model_task1, "layer1") and hasattr(model_task1, "layer4"):
    print("✓ ResNet-style architecture detected.")

if hasattr(model_task1, "fc"):
    print("✓ Final classifier attribute 'fc' detected.")

# ------------------------------------------------------------
# 7. Sample activation shapes
#
# We attach temporary hooks to ReLU modules and run ONE batch.
# This does NOT change the model.
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SAMPLE RELU ACTIVATION SHAPES")
print("-" * 70)

sample_shapes = {}
hooks = []

def make_shape_hook(layer_name):

    def hook(module, inputs, output):

        if torch.is_tensor(output):

            sample_shapes[layer_name] = tuple(
                output.shape
            )

    return hook


for name, module in relu_modules:

    hooks.append(
        module.register_forward_hook(
            make_shape_hook(name)
        )
    )

# One batch only
sample_batch = next(iter(task1_train_loader))

with torch.no_grad():

    sample_images = sample_batch["image"].to(device)

    _ = model_task1(sample_images)

# Remove hooks immediately
for h in hooks:
    h.remove()

# ------------------------------------------------------------
# 8. Print activation shapes
# ------------------------------------------------------------

for name, shape in sample_shapes.items():

    print(
        f"{name:45s} -> {shape}"
    )

print("\n" + "=" * 70)
print("STEP 10 COMPLETE")
print("=" * 70)

print(
    "\nNo weights were changed. "
    "No pruning was performed."
)

BIASPRUNER — RESNET-50 ACTIVATION STRUCTURE

Model type:
<class 'torchvision.models.resnet.ResNet'>

Total parameters:
23,512,130

Module counts:
  Conv2d: 53
  ReLU  : 17

----------------------------------------------------------------------
CONVOLUTIONAL LAYERS
----------------------------------------------------------------------
conv1                                         out_channels=  64 kernel=(7, 7) stride=(2, 2)
layer1.0.conv1                                out_channels=  64 kernel=(1, 1) stride=(1, 1)
layer1.0.conv2                                out_channels=  64 kernel=(3, 3) stride=(1, 1)
layer1.0.conv3                                out_channels= 256 kernel=(1, 1) stride=(1, 1)
layer1.0.downsample.0                         out_channels= 256 kernel=(1, 1) stride=(1, 1)
layer1.1.conv1                                out_channels=  64 kernel=(1, 1) stride=(1, 1)
layer1.1.conv2                                out_channels=  64 kernel=(3, 3) stride=(1, 1)
layer1.1.conv3      

In [28]:
# ============================================================
# STEP 11 — BIASPRUNER SPATIAL-VARIANCE BIAS SCORE
#
# Repository-aligned partition:
#   Correct samples -> E-like group
#   Wrong samples   -> H-like group
#
# Paper bias score:
#
# S_c,n =
#     mean_{i in E_c} Var(a_i^n)
#     -
#     mean_{i in H_c} Var(a_i^n)
#
# where Var(.) is over spatial dimensions.
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn

print("=" * 70)
print("BIASPRUNER — SPATIAL-VARIANCE BIAS SCORE")
print("=" * 70)

model_task1.eval()

# ------------------------------------------------------------
# 1. Identify the ReLU layers
# ------------------------------------------------------------

relu_layers = []

for name, module in model_task1.named_modules():

    if isinstance(module, nn.ReLU):
        relu_layers.append((name, module))

print("\nReLU layers:", len(relu_layers))

# ------------------------------------------------------------
# 2. Determine channel count for every ReLU layer
# ------------------------------------------------------------

layer_info = {}

total_units = 0

for name, module in relu_layers:

    layer_info[name] = {
        "channels": None,
        "correct_sum": None,
        "wrong_sum": None,
        "correct_count": None,
        "wrong_count": None,
    }

# ------------------------------------------------------------
# 3. Accumulators
#
# We accumulate:
#
#   sum of spatial variances for correct samples
#   sum of spatial variances for wrong samples
#
# separately for each ground-truth class.
# ------------------------------------------------------------

# We first discover channel dimensions using one batch.

first_batch = next(iter(task1_train_loader))

activation_cache = {}
hooks = []

def make_activation_hook(layer_name):

    def hook(module, inputs, output):

        activation_cache[layer_name] = output

    return hook


for name, module in relu_layers:

    hooks.append(
        module.register_forward_hook(
            make_activation_hook(name)
        )
    )

with torch.no_grad():

    first_images = first_batch["image"].to(device)

    _ = model_task1(first_images)

# Remove temporary hooks
for h in hooks:
    h.remove()

# ------------------------------------------------------------
# Initialize accumulators
# ------------------------------------------------------------

for name, activation in activation_cache.items():

    channels = activation.shape[1]

    layer_info[name]["channels"] = channels

    # class 0 and class 1
    layer_info[name]["correct_sum"] = torch.zeros(
        (2, channels),
        dtype=torch.float64
    )

    layer_info[name]["wrong_sum"] = torch.zeros(
        (2, channels),
        dtype=torch.float64
    )

    layer_info[name]["correct_count"] = torch.zeros(
        2,
        dtype=torch.long
    )

    layer_info[name]["wrong_count"] = torch.zeros(
        2,
        dtype=torch.long
    )

    total_units += channels

# Free first-batch activation cache
del activation_cache

print("\nUnits being scored:")
for name, info in layer_info.items():
    print(
        f"  {name:35s} "
        f"{info['channels']:5d} channels"
    )

print("\nTotal ReLU feature-map units:", total_units)

# ------------------------------------------------------------
# 4. Register permanent scoring hooks
# ------------------------------------------------------------

activation_cache = {}

hooks = []

for name, module in relu_layers:

    hooks.append(
        module.register_forward_hook(
            make_activation_hook(name)
        )
    )

# ------------------------------------------------------------
# 5. Stream through Task 1 training data
# ------------------------------------------------------------

print("\nCalculating spatial variances...")

processed = 0

with torch.no_grad():

    for batch_idx, batch in enumerate(task1_train_loader):

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        # Forward pass
        logits = model_task1(images)

        predictions = logits.argmax(dim=1)

        batch_size = labels.shape[0]

        # ----------------------------------------------------
        # Process every ReLU layer
        # ----------------------------------------------------

        for name, info in layer_info.items():

            activation = activation_cache[name]

            # activation:
            # [batch, channels, height, width]
            #
            # Paper:
            # Var(a_i^n) over spatial dimensions.
            #
            # We use population variance (unbiased=False)
            # because the spatial locations constitute the
            # complete feature map whose variance is being
            # measured.
            spatial_variance = activation.var(
                dim=(2, 3),
                unbiased=False
            )

            # Move only the small [batch, channels] matrix
            # to CPU.
            spatial_variance = (
                spatial_variance
                .detach()
                .cpu()
                .double()
            )

            labels_cpu = labels.detach().cpu()
            predictions_cpu = predictions.detach().cpu()

            # ----------------------------------------------
            # Correct / E-like samples
            # ----------------------------------------------

            correct_mask = (
                predictions_cpu == labels_cpu
            )

            # ----------------------------------------------
            # Wrong / H-like samples
            # ----------------------------------------------

            wrong_mask = (
                predictions_cpu != labels_cpu
            )

            # ----------------------------------------------
            # Accumulate class-specific statistics
            # ----------------------------------------------

            for class_id in [0, 1]:

                class_correct = (
                    correct_mask
                    & (labels_cpu == class_id)
                )

                class_wrong = (
                    wrong_mask
                    & (labels_cpu == class_id)
                )

                if class_correct.any():

                    values = spatial_variance[
                        class_correct
                    ]

                    info["correct_sum"][class_id] += (
                        values.sum(dim=0)
                    )

                    info["correct_count"][class_id] += (
                        int(class_correct.sum().item())
                    )

                if class_wrong.any():

                    values = spatial_variance[
                        class_wrong
                    ]

                    info["wrong_sum"][class_id] += (
                        values.sum(dim=0)
                    )

                    info["wrong_count"][class_id] += (
                        int(class_wrong.sum().item())
                    )

        processed += batch_size

        # Clear activation references
        activation_cache.clear()

        if (
            batch_idx == 0
            or (batch_idx + 1) % 10 == 0
            or processed == len(task1_train_dataset)
        ):

            print(
                f"  Processed {processed}/"
                f"{len(task1_train_dataset)}"
            )

# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

# ------------------------------------------------------------
# 6. Calculate class-specific bias scores
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CALCULATING CLASS-SPECIFIC BIAS SCORES")
print("-" * 70)

class_bias_scores = {}

for name, info in layer_info.items():

    channels = info["channels"]

    scores = torch.full(
        (2, channels),
        float("nan"),
        dtype=torch.float64
    )

    for class_id in [0, 1]:

        n_correct = int(
            info["correct_count"][class_id]
        )

        n_wrong = int(
            info["wrong_count"][class_id]
        )

        print(
            f"\n{name} | "
            f"class={TASK1_CLASSES[class_id]} | "
            f"correct={n_correct} | "
            f"wrong={n_wrong}"
        )

        # Paper formula requires both groups.
        if n_correct > 0 and n_wrong > 0:

            mean_correct = (
                info["correct_sum"][class_id]
                / n_correct
            )

            mean_wrong = (
                info["wrong_sum"][class_id]
                / n_wrong
            )

            scores[class_id] = (
                mean_correct - mean_wrong
            )

    class_bias_scores[name] = scores

# ------------------------------------------------------------
# 7. Final unit bias score
#
# Paper:
# "The final unit-based bias score is computed by
# averaging across all class-specific scores."
# ------------------------------------------------------------

final_bias_scores = {}

for name, scores in class_bias_scores.items():

    valid_scores = []

    for class_id in [0, 1]:

        class_scores = scores[class_id]

        if torch.isfinite(class_scores).any():

            valid_scores.append(
                class_scores
            )

    if len(valid_scores) == 0:

        final_scores = torch.full_like(
            scores[0],
            float("nan")
        )

    else:

        stacked = torch.stack(
            valid_scores,
            dim=0
        )

        final_scores = torch.nanmean(
            stacked,
            dim=0
        )

    final_bias_scores[name] = final_scores

# ------------------------------------------------------------
# 8. Global ranking of all ReLU units
# ------------------------------------------------------------

ranking_entries = []

for name, scores in final_bias_scores.items():

    for channel_id in range(len(scores)):

        score = float(scores[channel_id].item())

        if np.isfinite(score):

            ranking_entries.append(
                (
                    score,
                    name,
                    channel_id
                )
            )

# Highest bias first
ranking_entries.sort(
    key=lambda x: x[0],
    reverse=True
)

print("\n" + "-" * 70)
print("GLOBAL BIAS-SCORE SUMMARY")
print("-" * 70)

print(
    "\nScored units:",
    len(ranking_entries)
)

if len(ranking_entries) > 0:

    all_scores = np.array(
        [x[0] for x in ranking_entries]
    )

    print(
        "Score min:",
        float(all_scores.min())
    )

    print(
        "Score max:",
        float(all_scores.max())
    )

    print(
        "Score mean:",
        float(all_scores.mean())
    )

    print(
        "Score median:",
        float(np.median(all_scores))
    )

# ------------------------------------------------------------
# 9. Top 20 most biased units
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TOP 20 HIGHEST-BIAS UNITS")
print("-" * 70)

for rank, (score, layer_name, channel_id) in enumerate(
    ranking_entries[:20],
    start=1
):

    print(
        f"{rank:2d}. "
        f"{layer_name:35s} "
        f"channel={channel_id:4d} "
        f"score={score:.8f}"
    )

# ------------------------------------------------------------
# 10. Save scores
# ------------------------------------------------------------

save_dir = (
    "/kaggle/working/"
    "biaspruner_checkpoints/task1"
)

os.makedirs(
    save_dir,
    exist_ok=True
)

# Save readable NumPy object
npz_data = {}

for name, scores in final_bias_scores.items():

    safe_name = (
        name
        .replace(".", "_")
        .replace("/", "_")
    )

    npz_data[safe_name] = (
        scores.numpy()
    )

bias_score_path = os.path.join(
    save_dir,
    "task1_spatial_variance_bias_scores.npz"
)

np.savez(
    bias_score_path,
    **npz_data
)

# Save ranking
ranking_path = os.path.join(
    save_dir,
    "task1_bias_ranking.csv"
)

with open(ranking_path, "w") as f:

    f.write(
        "rank,layer,channel,bias_score\n"
    )

    for rank, (
        score,
        layer_name,
        channel_id
    ) in enumerate(
        ranking_entries,
        start=1
    ):

        f.write(
            f"{rank},"
            f"{layer_name},"
            f"{channel_id},"
            f"{score:.12f}\n"
        )

print("\nSaved:")
print(" ", bias_score_path)
print(" ", ranking_path)

print("\n" + "=" * 70)
print("STEP 11 COMPLETE")
print("=" * 70)

print("\nIMPORTANT:")
print("No model weights were changed.")
print("No channels were pruned yet.")

BIASPRUNER — SPATIAL-VARIANCE BIAS SCORE

ReLU layers: 17

Units being scored:
  relu                                   64 channels
  layer1.0.relu                         256 channels
  layer1.1.relu                         256 channels
  layer1.2.relu                         256 channels
  layer2.0.relu                         512 channels
  layer2.1.relu                         512 channels
  layer2.2.relu                         512 channels
  layer2.3.relu                         512 channels
  layer3.0.relu                        1024 channels
  layer3.1.relu                        1024 channels
  layer3.2.relu                        1024 channels
  layer3.3.relu                        1024 channels
  layer3.4.relu                        1024 channels
  layer3.5.relu                        1024 channels
  layer4.0.relu                        2048 channels
  layer4.1.relu                        2048 channels
  layer4.2.relu                        2048 channels

Total ReLU feature-

In [29]:
# ============================================================
# STEP 11 CORRECTED
#
# Deterministic analysis of Task 1 training samples.
#
# IMPORTANT:
#   No RandomHorizontalFlip during attribution.
#
# This ensures:
#   prediction <-> activation <-> sample
# are deterministic and consistent.
# ============================================================

import os
import csv
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

print("=" * 70)
print("BIASPRUNER — CORRECTED DETERMINISTIC BIAS ANALYSIS")
print("=" * 70)

model_task1.eval()

# ------------------------------------------------------------
# 1. Create deterministic analysis dataset
# ------------------------------------------------------------

task1_analysis_dataset = HAM10000TaskDataset(
    task1_train_df,
    image_lookup,
    TASK1_CLASSES,
    transform=val_transform
)

task1_analysis_loader = DataLoader(
    task1_analysis_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nAnalysis dataset:")
print("  Samples:", len(task1_analysis_dataset))
print("  Shuffle: False")
print("  Random flip: False")

# ------------------------------------------------------------
# 2. Recompute fixed correct/wrong partition
# ------------------------------------------------------------

all_labels = []
all_predictions = []
all_true_probs = []
all_image_ids = []

correct_by_class = {
    0: [],
    1: []
}

wrong_by_class = {
    0: [],
    1: []
}

global_index = 0

with torch.no_grad():

    for batch in task1_analysis_loader:

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model_task1(images)
        probs = torch.softmax(logits, dim=1)
        predictions = probs.argmax(dim=1)

        for j in range(labels.size(0)):

            label = int(labels[j].item())
            pred = int(predictions[j].item())

            all_labels.append(label)
            all_predictions.append(pred)
            all_true_probs.append(
                float(probs[j, label].item())
            )
            all_image_ids.append(
                batch["image_id"][j]
            )

            if pred == label:
                correct_by_class[label].append(
                    global_index
                )
            else:
                wrong_by_class[label].append(
                    global_index
                )

            global_index += 1

# ------------------------------------------------------------
# 3. Verify partition
# ------------------------------------------------------------

all_labels_tensor = torch.tensor(
    all_labels,
    dtype=torch.long
)

all_predictions_tensor = torch.tensor(
    all_predictions,
    dtype=torch.long
)

all_true_probs_tensor = torch.tensor(
    all_true_probs,
    dtype=torch.float32
)

S_ba = set(
    sum(correct_by_class.values(), [])
)

S_bc = set(
    sum(wrong_by_class.values(), [])
)

print("\n" + "-" * 70)
print("FIXED PARTITION")
print("-" * 70)

print("Total:", len(all_labels))
print("S_ba correct:", len(S_ba))
print("S_bc wrong:", len(S_bc))
print("Coverage:", len(S_ba) + len(S_bc))

accuracy = (
    all_predictions_tensor == all_labels_tensor
).float().mean().item()

print(
    "Training accuracy:",
    f"{accuracy * 100:.2f}%"
)

for class_id, class_name in enumerate(TASK1_CLASSES):

    total = int(
        (all_labels_tensor == class_id).sum()
    )

    correct = len(
        correct_by_class[class_id]
    )

    wrong = len(
        wrong_by_class[class_id]
    )

    print(
        f"{class_name}: "
        f"total={total}, "
        f"correct={correct}, "
        f"wrong={wrong}, "
        f"accuracy={100*correct/total:.2f}%"
    )

# ------------------------------------------------------------
# 4. Confidence diagnostic
# ------------------------------------------------------------

tau = 0.70

wrong_mask = (
    all_predictions_tensor
    != all_labels_tensor
)

wrong_true_probs = (
    all_true_probs_tensor[wrong_mask]
)

print("\nConfidence diagnostic:")
print(
    "Correct with p_true >= 0.70:",
    int(
        (
            (all_predictions_tensor == all_labels_tensor)
            &
            (all_true_probs_tensor >= tau)
        ).sum()
    )
)

print(
    "Wrong with p_true >= 0.70:",
    int(
        (
            wrong_mask
            &
            (all_true_probs_tensor >= tau)
        ).sum()
    )
)

if len(wrong_true_probs) > 0:

    print(
        "Max p_true among wrong:",
        float(wrong_true_probs.max())
    )

# ------------------------------------------------------------
# 5. Identify ReLU layers
# ------------------------------------------------------------

relu_layers = []

for name, module in model_task1.named_modules():

    if isinstance(module, nn.ReLU):
        relu_layers.append(
            (name, module)
        )

print("\nReLU layers:", len(relu_layers))

# ------------------------------------------------------------
# 6. Determine channel counts
# ------------------------------------------------------------

activation_cache = {}
shape_hooks = []

def make_shape_hook(name):

    def hook(module, inputs, output):
        activation_cache[name] = output

    return hook

for name, module in relu_layers:

    shape_hooks.append(
        module.register_forward_hook(
            make_shape_hook(name)
        )
    )

with torch.no_grad():

    sample = next(
        iter(task1_analysis_loader)
    )

    _ = model_task1(
        sample["image"].to(device)
    )

for h in shape_hooks:
    h.remove()

layer_info = {}

total_units = 0

for name, activation in activation_cache.items():

    channels = activation.shape[1]

    layer_info[name] = {
        "channels": channels,

        "correct_sum": torch.zeros(
            (2, channels),
            dtype=torch.float64
        ),

        "wrong_sum": torch.zeros(
            (2, channels),
            dtype=torch.float64
        ),

        "correct_count": torch.zeros(
            2,
            dtype=torch.long
        ),

        "wrong_count": torch.zeros(
            2,
            dtype=torch.long
        )
    }

    total_units += channels

activation_cache.clear()

print(
    "Total ReLU feature-map units:",
    total_units
)

# ------------------------------------------------------------
# 7. Register activation hooks
# ------------------------------------------------------------

activation_cache = {}
hooks = []

for name, module in relu_layers:

    hooks.append(
        module.register_forward_hook(
            make_shape_hook(name)
        )
    )

# ------------------------------------------------------------
# 8. Calculate spatial variance
# ------------------------------------------------------------

processed = 0

print("\nCalculating spatial variances...")

with torch.no_grad():

    for batch_idx, batch in enumerate(
        task1_analysis_loader
    ):

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model_task1(images)
        predictions = logits.argmax(dim=1)

        labels_cpu = labels.cpu()
        predictions_cpu = predictions.cpu()

        for name, info in layer_info.items():

            activation = activation_cache[name]

            # [B,C,H,W]
            #
            # Variance over spatial dimensions only:
            # [H,W]
            #
            spatial_variance = activation.var(
                dim=(2, 3),
                unbiased=False
            ).cpu().double()

            correct_mask = (
                predictions_cpu == labels_cpu
            )

            wrong_mask_batch = (
                predictions_cpu != labels_cpu
            )

            for class_id in [0, 1]:

                correct_mask_class = (
                    correct_mask
                    &
                    (labels_cpu == class_id)
                )

                wrong_mask_class = (
                    wrong_mask_batch
                    &
                    (labels_cpu == class_id)
                )

                if correct_mask_class.any():

                    values = spatial_variance[
                        correct_mask_class
                    ]

                    info["correct_sum"][class_id] += (
                        values.sum(dim=0)
                    )

                    info["correct_count"][class_id] += (
                        int(
                            correct_mask_class.sum()
                        )
                    )

                if wrong_mask_class.any():

                    values = spatial_variance[
                        wrong_mask_class
                    ]

                    info["wrong_sum"][class_id] += (
                        values.sum(dim=0)
                    )

                    info["wrong_count"][class_id] += (
                        int(
                            wrong_mask_class.sum()
                        )
                    )

        processed += images.size(0)
        activation_cache.clear()

        if (
            processed == len(task1_analysis_dataset)
            or processed % 320 == 0
        ):

            print(
                f"  Processed "
                f"{processed}/"
                f"{len(task1_analysis_dataset)}"
            )

# Remove hooks
for h in hooks:
    h.remove()

# ------------------------------------------------------------
# 9. Compute class-specific bias scores
# ------------------------------------------------------------

class_bias_scores = {}
final_bias_scores = {}

for name, info in layer_info.items():

    channels = info["channels"]

    scores = torch.full(
        (2, channels),
        float("nan"),
        dtype=torch.float64
    )

    for class_id in [0, 1]:

        n_correct = int(
            info["correct_count"][class_id]
        )

        n_wrong = int(
            info["wrong_count"][class_id]
        )

        if n_correct > 0 and n_wrong > 0:

            mean_correct = (
                info["correct_sum"][class_id]
                / n_correct
            )

            mean_wrong = (
                info["wrong_sum"][class_id]
                / n_wrong
            )

            # Paper Eq. (5)
            scores[class_id] = (
                mean_correct
                -
                mean_wrong
            )

    class_bias_scores[name] = scores

    # Average across classes
    final_bias_scores[name] = torch.nanmean(
        scores,
        dim=0
    )

# ------------------------------------------------------------
# 10. Global ranking
# ------------------------------------------------------------

ranking_entries = []

for name, scores in final_bias_scores.items():

    for channel_id, score_tensor in enumerate(
        scores
    ):

        score = float(
            score_tensor.item()
        )

        if np.isfinite(score):

            ranking_entries.append(
                (
                    score,
                    name,
                    channel_id
                )
            )

ranking_entries.sort(
    key=lambda x: x[0],
    reverse=True
)

# ------------------------------------------------------------
# 11. Print results
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL BIAS-SCORE SUMMARY")
print("-" * 70)

scores_np = np.array(
    [x[0] for x in ranking_entries]
)

print(
    "Scored units:",
    len(ranking_entries)
)

print(
    "Min:",
    float(scores_np.min())
)

print(
    "Max:",
    float(scores_np.max())
)

print(
    "Mean:",
    float(scores_np.mean())
)

print(
    "Median:",
    float(np.median(scores_np))
)

print("\nTop 20 highest-bias units:")

for rank, (
    score,
    layer_name,
    channel_id
) in enumerate(
    ranking_entries[:20],
    start=1
):

    print(
        f"{rank:2d}. "
        f"{layer_name:35s} "
        f"channel={channel_id:4d} "
        f"score={score:.8f}"
    )

# ------------------------------------------------------------
# 12. Save corrected results
# ------------------------------------------------------------

save_dir = (
    "/kaggle/working/"
    "biaspruner_checkpoints/task1"
)

os.makedirs(
    save_dir,
    exist_ok=True
)

npz_data = {}

for name, scores in final_bias_scores.items():

    safe_name = (
        name
        .replace(".", "_")
        .replace("/", "_")
    )

    npz_data[safe_name] = (
        scores.numpy()
    )

np.savez(
    os.path.join(
        save_dir,
        "task1_spatial_variance_bias_scores.npz"
    ),
    **npz_data
)

with open(
    os.path.join(
        save_dir,
        "task1_bias_ranking.csv"
    ),
    "w"
) as f:

    f.write(
        "rank,layer,channel,bias_score\n"
    )

    for rank, (
        score,
        layer_name,
        channel_id
    ) in enumerate(
        ranking_entries,
        start=1
    ):

        f.write(
            f"{rank},"
            f"{layer_name},"
            f"{channel_id},"
            f"{score:.12f}\n"
        )

# ------------------------------------------------------------
# 13. Save deterministic partition
# ------------------------------------------------------------

partition_info = {
    "S_ba": S_ba,
    "S_bc": S_bc,
    "correct_by_class": correct_by_class,
    "wrong_by_class": wrong_by_class,
    "labels": all_labels_tensor,
    "predictions": all_predictions_tensor,
    "true_class_prob": all_true_probs_tensor,
    "image_ids": all_image_ids,
    "tau": tau
}

print("\nSaved corrected bias scores and ranking.")

print("\n" + "=" * 70)
print("STEP 11 CORRECTED COMPLETE")
print("=" * 70)

print("No model weights changed.")
print("No pruning performed.")

BIASPRUNER — CORRECTED DETERMINISTIC BIAS ANALYSIS

Analysis dataset:
  Samples: 1545
  Shuffle: False
  Random flip: False

----------------------------------------------------------------------
FIXED PARTITION
----------------------------------------------------------------------
Total: 1545
S_ba correct: 1185
S_bc wrong: 360
Coverage: 1545
Training accuracy: 76.70%
bkl: total=772, correct=510, wrong=262, accuracy=66.06%
mel: total=773, correct=675, wrong=98, accuracy=87.32%

Confidence diagnostic:
Correct with p_true >= 0.70: 1137
Wrong with p_true >= 0.70: 0
Max p_true among wrong: 0.4999188184738159

ReLU layers: 17
Total ReLU feature-map units: 15168

Calculating spatial variances...
  Processed 320/1545
  Processed 640/1545
  Processed 960/1545
  Processed 1280/1545
  Processed 1545/1545

----------------------------------------------------------------------
FINAL BIAS-SCORE SUMMARY
----------------------------------------------------------------------
Scored units: 15168
Min: -

In [30]:
# ================================================================
# STEP 12 — VERIFY BIAS RANKING + PRUNING COUNT
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("BIASPRUNER — STEP 12: RANKING VERIFICATION")
print("=" * 70)

score_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_spatial_variance_bias_scores.npz"
)

ranking_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_bias_ranking.csv"
)

# ------------------------------------------------
# Load saved files
# ------------------------------------------------
data = np.load(score_path, allow_pickle=True)
scores = data["scores"]

ranking_df = pd.read_csv(ranking_path)

print(f"Loaded scores: {len(scores)}")
print(f"Ranking rows:  {len(ranking_df)}")

# ------------------------------------------------
# Basic consistency
# ------------------------------------------------
assert len(scores) == 15168, "Unexpected number of scored units."
assert len(ranking_df) == 15168, "Ranking length mismatch."

assert ranking_df["score"].notna().all(), "NaN scores found."

# Check descending order
is_descending = np.all(
    ranking_df["score"].values[:-1] >= ranking_df["score"].values[1:]
)

print(f"Ranking descending: {is_descending}")

assert is_descending, "Ranking is not sorted descending."

# ------------------------------------------------
# Paper pruning ratio
# ------------------------------------------------
GAMMA = 0.60

n_units = len(ranking_df)
n_prune = int(np.ceil(GAMMA * n_units))
n_keep = n_units - n_prune

print()
print("-" * 70)
print("PRUNING COUNT")
print("-" * 70)

print(f"Total units : {n_units}")
print(f"Gamma       : {GAMMA}")
print(f"Units prune : {n_prune}")
print(f"Units keep  : {n_keep}")

# ------------------------------------------------
# Threshold
# ------------------------------------------------
threshold = ranking_df.iloc[n_prune - 1]["score"]

print()
print(f"60% pruning threshold score: {threshold:.8f}")

print()
print("Top 5:")
print(
    ranking_df[
        ["layer", "channel", "score"]
    ].head(5).to_string(index=False)
)

print()
print("Boundary:")
print(
    ranking_df.iloc[
        max(0, n_prune - 3):n_prune + 3
    ][["layer", "channel", "score"]].to_string(index=True)
)

# ------------------------------------------------
# Save verification metadata
# ------------------------------------------------
verification_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_pruning_plan.txt"
)

with open(verification_path, "w") as f:
    f.write("BiasPruner Task 1 pruning plan\n")
    f.write("==============================\n")
    f.write(f"Total scored ReLU units: {n_units}\n")
    f.write(f"Gamma: {GAMMA}\n")
    f.write(f"Units to prune: {n_prune}\n")
    f.write(f"Units to keep: {n_keep}\n")
    f.write(f"Pruning threshold score: {threshold:.8f}\n")
    f.write("Ranking source: task1_bias_ranking.csv\n")
    f.write("Bias score source: task1_spatial_variance_bias_scores.npz\n")

print()
print(f"Saved pruning plan:")
print(f"  {verification_path}")

print()
print("=" * 70)
print("STEP 12 COMPLETE")
print("=" * 70)

BIASPRUNER — STEP 12: RANKING VERIFICATION


KeyError: 'scores is not a file in the archive'

In [31]:
# ================================================================
# STEP 12A — INSPECT SAVED BIAS-SCORE FILE
# ================================================================

import os
import numpy as np

score_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_spatial_variance_bias_scores.npz"
)

data = np.load(score_path, allow_pickle=True)

print("=" * 70)
print("SAVED BIAS-SCORE FILE CONTENTS")
print("=" * 70)

print("File:", score_path)
print("Keys:", list(data.keys()))

for key in data.keys():
    arr = data[key]
    print(f"\nKey: {key}")
    print(f"  Type:  {type(arr)}")
    print(f"  Shape: {getattr(arr, 'shape', None)}")
    print(f"  Dtype: {getattr(arr, 'dtype', None)}")

print()
print("=" * 70)
print("STEP 12A COMPLETE")
print("=" * 70)

SAVED BIAS-SCORE FILE CONTENTS
File: /kaggle/working/biaspruner_checkpoints/task1/task1_spatial_variance_bias_scores.npz
Keys: ['relu', 'layer1_0_relu', 'layer1_1_relu', 'layer1_2_relu', 'layer2_0_relu', 'layer2_1_relu', 'layer2_2_relu', 'layer2_3_relu', 'layer3_0_relu', 'layer3_1_relu', 'layer3_2_relu', 'layer3_3_relu', 'layer3_4_relu', 'layer3_5_relu', 'layer4_0_relu', 'layer4_1_relu', 'layer4_2_relu']

Key: relu
  Type:  <class 'numpy.ndarray'>
  Shape: (64,)
  Dtype: float64

Key: layer1_0_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (256,)
  Dtype: float64

Key: layer1_1_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (256,)
  Dtype: float64

Key: layer1_2_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (256,)
  Dtype: float64

Key: layer2_0_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (512,)
  Dtype: float64

Key: layer2_1_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (512,)
  Dtype: float64

Key: layer2_2_relu
  Type:  <class 'numpy.ndarray'>
  Shape: (512,)
  Dtype: float64


In [32]:
# ================================================================
# STEP 12B — FLATTEN + VERIFY BIAS RANKING
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("BIASPRUNER — STEP 12B: RANKING VERIFICATION")
print("=" * 70)

score_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_spatial_variance_bias_scores.npz"
)

ranking_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_bias_ranking.csv"
)

data = np.load(score_path, allow_pickle=True)

# ------------------------------------------------
# Flatten all layer-wise scores
# ------------------------------------------------
records = []

for key in data.keys():
    scores_layer = data[key]

    # Convert saved key back to readable layer name
    layer_name = key.replace("_", ".")

    for channel, score in enumerate(scores_layer):
        records.append({
            "layer": layer_name,
            "channel": channel,
            "score": float(score)
        })

scores_df = pd.DataFrame(records)

print(f"Total flattened units: {len(scores_df)}")

# ------------------------------------------------
# Sort exactly by bias score
# ------------------------------------------------
scores_df = scores_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------
# Verify expected number
# ------------------------------------------------
assert len(scores_df) == 15168, \
    f"Expected 15168 units, got {len(scores_df)}"

assert scores_df["score"].notna().all(), \
    "NaN bias scores found."

is_descending = np.all(
    scores_df["score"].values[:-1]
    >= scores_df["score"].values[1:]
)

print(f"Ranking descending: {is_descending}")

assert is_descending

# ------------------------------------------------
# Compare with saved CSV
# ------------------------------------------------
saved_ranking = pd.read_csv(ranking_path)

print(f"Saved ranking rows: {len(saved_ranking)}")

assert len(saved_ranking) == 15168

# Compare scores
max_score_difference = np.max(
    np.abs(
        scores_df["score"].values
        - saved_ranking["score"].values
    )
)

print(
    f"Maximum difference vs saved ranking: "
    f"{max_score_difference:.12e}"
)

# ------------------------------------------------
# Gamma = 0.60
# ------------------------------------------------
GAMMA = 0.60

n_units = len(scores_df)

n_prune = int(np.ceil(GAMMA * n_units))
n_keep = n_units - n_prune

threshold = scores_df.iloc[n_prune - 1]["score"]

print()
print("-" * 70)
print("PRUNING PLAN")
print("-" * 70)

print(f"Total units       : {n_units}")
print(f"Gamma             : {GAMMA}")
print(f"Units to prune    : {n_prune}")
print(f"Units to keep     : {n_keep}")
print(f"Threshold score   : {threshold:.8f}")

# ------------------------------------------------
# Top units
# ------------------------------------------------
print()
print("Top 10 highest-bias units:")
print(
    scores_df[
        ["layer", "channel", "score"]
    ].head(10).to_string(index=False)
)

# ------------------------------------------------
# Boundary
# ------------------------------------------------
print()
print("Pruning boundary:")
print(
    scores_df.iloc[
        n_prune - 3:n_prune + 3
    ][
        ["layer", "channel", "score"]
    ].to_string(index=True)
)

# ------------------------------------------------
# Save corrected pruning plan
# ------------------------------------------------
verification_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_pruning_plan.txt"
)

with open(verification_path, "w") as f:
    f.write("BiasPruner Task 1 pruning plan\n")
    f.write("==============================\n")
    f.write(f"Total scored ReLU units: {n_units}\n")
    f.write(f"Gamma: {GAMMA}\n")
    f.write(f"Units to prune: {n_prune}\n")
    f.write(f"Units to keep: {n_keep}\n")
    f.write(f"Pruning threshold score: {threshold:.8f}\n")

print()
print(f"Saved: {verification_path}")

print()
print("=" * 70)
print("STEP 12B COMPLETE")
print("=" * 70)

BIASPRUNER — STEP 12B: RANKING VERIFICATION
Total flattened units: 15168
Ranking descending: True
Saved ranking rows: 15168


KeyError: 'score'

In [33]:
# ================================================================
# STEP 12C — INSPECT SAVED RANKING COLUMNS
# ================================================================

import os
import pandas as pd

ranking_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_bias_ranking.csv"
)

saved_ranking = pd.read_csv(ranking_path)

print("=" * 70)
print("SAVED RANKING CSV STRUCTURE")
print("=" * 70)

print("Columns:")
print(list(saved_ranking.columns))

print()
print("Shape:", saved_ranking.shape)

print()
print("First 5 rows:")
print(saved_ranking.head().to_string(index=False))

print()
print("=" * 70)
print("STEP 12C COMPLETE")
print("=" * 70)

SAVED RANKING CSV STRUCTURE
Columns:
['rank', 'layer', 'channel', 'bias_score']

Shape: (15168, 4)

First 5 rows:
 rank         layer  channel  bias_score
    1 layer4.2.relu       29    9.822034
    2 layer3.4.relu      234    8.074468
    3 layer4.2.relu     1333    7.822301
    4 layer4.2.relu      415    7.713757
    5 layer3.5.relu      794    7.297316

STEP 12C COMPLETE


In [34]:
# ================================================================
# STEP 12D — FINAL RANKING VERIFICATION
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("BIASPRUNER — STEP 12D: FINAL RANKING VERIFICATION")
print("=" * 70)

score_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_spatial_variance_bias_scores.npz"
)

ranking_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_bias_ranking.csv"
)

# ------------------------------------------------
# Load layer-wise scores
# ------------------------------------------------
data = np.load(score_path, allow_pickle=True)

records = []

for key in data.keys():
    layer_scores = data[key]

    # Convert saved key to readable layer name
    layer_name = key.replace("_", ".")

    for channel, score in enumerate(layer_scores):
        records.append({
            "layer": layer_name,
            "channel": channel,
            "bias_score": float(score)
        })

reconstructed = pd.DataFrame(records)

# Sort exactly as ranking should be
reconstructed = reconstructed.sort_values(
    "bias_score",
    ascending=False
).reset_index(drop=True)

reconstructed["rank"] = np.arange(1, len(reconstructed) + 1)

print(f"Reconstructed units: {len(reconstructed)}")

# ------------------------------------------------
# Load saved ranking
# ------------------------------------------------
saved = pd.read_csv(ranking_path)

print(f"Saved ranking units: {len(saved)}")

# ------------------------------------------------
# Structural checks
# ------------------------------------------------
assert len(reconstructed) == 15168
assert len(saved) == 15168

assert list(saved.columns) == [
    "rank",
    "layer",
    "channel",
    "bias_score"
]

# ------------------------------------------------
# Compare ranking
# ------------------------------------------------
same_rank = np.array_equal(
    reconstructed["rank"].values,
    saved["rank"].values
)

same_layer = np.array_equal(
    reconstructed["layer"].values,
    saved["layer"].values
)

same_channel = np.array_equal(
    reconstructed["channel"].values,
    saved["channel"].values
)

max_score_difference = np.max(
    np.abs(
        reconstructed["bias_score"].values
        - saved["bias_score"].values
    )
)

print()
print("-" * 70)
print("CONSISTENCY CHECK")
print("-" * 70)

print(f"Same ranks       : {same_rank}")
print(f"Same layers      : {same_layer}")
print(f"Same channels    : {same_channel}")
print(
    f"Max score diff   : "
    f"{max_score_difference:.12e}"
)

# ------------------------------------------------
# Pruning ratio
# ------------------------------------------------
GAMMA = 0.60

n_units = len(saved)

n_prune = int(np.ceil(GAMMA * n_units))
n_keep = n_units - n_prune

threshold = saved.iloc[n_prune - 1]["bias_score"]

print()
print("-" * 70)
print("TASK 1 PRUNING PLAN")
print("-" * 70)

print(f"Total scored units : {n_units}")
print(f"Gamma              : {GAMMA}")
print(f"Units to prune     : {n_prune}")
print(f"Units to keep      : {n_keep}")
print(f"Threshold score    : {threshold:.8f}")

# ------------------------------------------------
# Show pruning boundary
# ------------------------------------------------
print()
print("Around pruning boundary:")
print(
    saved.iloc[
        n_prune - 3:n_prune + 3
    ].to_string(index=False)
)

# ------------------------------------------------
# Save final pruning plan
# ------------------------------------------------
plan_path = os.path.join(
    CHECKPOINT_ROOT,
    "task1",
    "task1_pruning_plan.txt"
)

with open(plan_path, "w") as f:
    f.write("BiasPruner Task 1 Pruning Plan\n")
    f.write("==============================\n")
    f.write(f"Total scored units: {n_units}\n")
    f.write(f"Gamma: {GAMMA}\n")
    f.write(f"Units to prune: {n_prune}\n")
    f.write(f"Units to keep: {n_keep}\n")
    f.write(f"Threshold score: {threshold:.8f}\n")
    f.write("\n")
    f.write("Ranking verified against saved layer-wise scores.\n")

print()
print(f"Saved pruning plan:")
print(f"  {plan_path}")

print()
print("=" * 70)
print("STEP 12D COMPLETE")
print("=" * 70)

BIASPRUNER — STEP 12D: FINAL RANKING VERIFICATION
Reconstructed units: 15168
Saved ranking units: 15168

----------------------------------------------------------------------
CONSISTENCY CHECK
----------------------------------------------------------------------
Same ranks       : True
Same layers      : True
Same channels    : True
Max score diff   : 4.999004682427e-13

----------------------------------------------------------------------
TASK 1 PRUNING PLAN
----------------------------------------------------------------------
Total scored units : 15168
Gamma              : 0.6
Units to prune     : 9101
Units to keep      : 6067
Threshold score    : 0.11385791

Around pruning boundary:
 rank         layer  channel  bias_score
 9099 layer3.1.relu      150    0.113950
 9100 layer2.1.relu      499    0.113907
 9101 layer3.2.relu      922    0.113858
 9102 layer1.0.relu       84    0.113845
 9103 layer4.1.relu     1549    0.113841
 9104 layer4.0.relu     1446    0.113802

Saved prunin

In [35]:
# ================================================================
# STEP 13 — RESNET-50 PRUNING STRUCTURE INSPECTION
# READ ONLY — NO WEIGHTS CHANGED
# ================================================================

import torch
import torch.nn as nn

print("=" * 70)
print("BIASPRUNER — STEP 13: RESNET-50 PRUNING STRUCTURE")
print("=" * 70)

model = model_task1

# ------------------------------------------------
# 1. List convolutional layers
# ------------------------------------------------
print()
print("-" * 70)
print("CONVOLUTIONAL LAYERS")
print("-" * 70)

conv_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        conv_layers.append({
            "name": name,
            "in_channels": module.in_channels,
            "out_channels": module.out_channels,
            "kernel": module.kernel_size,
            "stride": module.stride,
            "groups": module.groups
        })

for i, x in enumerate(conv_layers):
    print(
        f"{i:02d} | {x['name']:25s} | "
        f"in={x['in_channels']:4d} "
        f"out={x['out_channels']:4d} | "
        f"k={x['kernel']} stride={x['stride']} "
        f"groups={x['groups']}"
    )

print(f"\nTotal Conv2d layers: {len(conv_layers)}")

# ------------------------------------------------
# 2. BatchNorm layers
# ------------------------------------------------
print()
print("-" * 70)
print("BATCH NORMALIZATION LAYERS")
print("-" * 70)

bn_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.BatchNorm2d):
        bn_layers.append({
            "name": name,
            "channels": module.num_features
        })

for i, x in enumerate(bn_layers):
    print(
        f"{i:02d} | {x['name']:25s} | "
        f"channels={x['channels']:4d}"
    )

print(f"\nTotal BatchNorm2d layers: {len(bn_layers)}")

# ------------------------------------------------
# 3. ReLU layers
# ------------------------------------------------
print()
print("-" * 70)
print("RELU LAYERS USED FOR BIAS SCORES")
print("-" * 70)

relu_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.ReLU):
        relu_layers.append({
            "name": name
        })

for i, x in enumerate(relu_layers):
    print(f"{i:02d} | {x['name']}")

print(f"\nTotal ReLU layers: {len(relu_layers)}")

# ------------------------------------------------
# 4. Residual blocks
# ------------------------------------------------
print()
print("-" * 70)
print("RESNET-50 RESIDUAL BLOCKS")
print("-" * 70)

for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
    layer = getattr(model, layer_name)

    print(
        f"{layer_name}: "
        f"{len(layer)} Bottleneck blocks"
    )

    for block_idx, block in enumerate(layer):
        print(
            f"  {layer_name}.{block_idx}: "
            f"conv1={block.conv1.out_channels}, "
            f"conv2={block.conv2.out_channels}, "
            f"conv3={block.conv3.out_channels}, "
            f"downsample="
            f"{block.downsample is not None}"
        )

# ------------------------------------------------
# 5. Verify bias-score layers correspond to ReLUs
# ------------------------------------------------
print()
print("-" * 70)
print("BIAS-SCORE LAYER MAPPING")
print("-" * 70)

for key in data.keys():
    layer_name = key.replace("_", ".")
    score_count = len(data[key])

    print(
        f"{layer_name:25s} | "
        f"bias units={score_count:4d}"
    )

# ------------------------------------------------
# 6. Important architecture observation
# ------------------------------------------------
print()
print("-" * 70)
print("ARCHITECTURE SUMMARY")
print("-" * 70)

print(
    "This is a torchvision ResNet-50 with Bottleneck blocks."
)
print(
    "Bias scores are attached to ReLU feature maps."
)
print(
    "Physical Conv/BN pruning must account for residual "
    "connections and channel dependencies."
)
print(
    "NO pruning or weight modification was performed."
)

print()
print("=" * 70)
print("STEP 13 COMPLETE")
print("=" * 70)

BIASPRUNER — STEP 13: RESNET-50 PRUNING STRUCTURE

----------------------------------------------------------------------
CONVOLUTIONAL LAYERS
----------------------------------------------------------------------
00 | conv1                     | in=   3 out=  64 | k=(7, 7) stride=(2, 2) groups=1
01 | layer1.0.conv1            | in=  64 out=  64 | k=(1, 1) stride=(1, 1) groups=1
02 | layer1.0.conv2            | in=  64 out=  64 | k=(3, 3) stride=(1, 1) groups=1
03 | layer1.0.conv3            | in=  64 out= 256 | k=(1, 1) stride=(1, 1) groups=1
04 | layer1.0.downsample.0     | in=  64 out= 256 | k=(1, 1) stride=(1, 1) groups=1
05 | layer1.1.conv1            | in= 256 out=  64 | k=(1, 1) stride=(1, 1) groups=1
06 | layer1.1.conv2            | in=  64 out=  64 | k=(3, 3) stride=(1, 1) groups=1
07 | layer1.1.conv3            | in=  64 out= 256 | k=(1, 1) stride=(1, 1) groups=1
08 | layer1.2.conv1            | in= 256 out=  64 | k=(1, 1) stride=(1, 1) groups=1
09 | layer1.2.conv2           

In [36]:
# ================================================================
# STEP 14 — CHECK AVAILABLE BIASPRUNER REPOSITORY CODE
# READ ONLY
# ================================================================

import os
import subprocess
import textwrap

print("=" * 70)
print("BIASPRUNER — STEP 14: REPOSITORY CODE CHECK")
print("=" * 70)

repo_dir = "/kaggle/working/BiasPruner"

# Clone only if repository is not already present
if not os.path.exists(repo_dir):
    print("Cloning official BiasPruner repository...")
    result = subprocess.run(
        [
            "git", "clone",
            "https://github.com/nourhanb/BiasPruner.git",
            repo_dir
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Repository clone failed.")
else:
    print("Repository already exists.")

print()
print("Repository files:")

for root, dirs, files in os.walk(repo_dir):
    # Avoid printing .git internals
    dirs[:] = [d for d in dirs if d != ".git"]

    level = root.replace(repo_dir, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(files):
        print(f"{indent}  {filename}")

print()
print("=" * 70)
print("STEP 14 COMPLETE")
print("No model weights changed.")
print("No pruning performed.")
print("=" * 70)

BIASPRUNER — STEP 14: REPOSITORY CODE CHECK
Cloning official BiasPruner repository...


Repository files:
BiasPruner/
  DermoDatasets.py
  README.md
  attributes.py
  datasets.py
  functions.py
  get_dataloaders.py
  libs.py
  main.py
  models.py
  my_fairlearn.py
  overview.png
  param.py
  readme.txt
  test.py
  train.py
  trainer.py
  GbP/
    .keep
    Data_Entry_2017.csv
    NIH_15classes_Filtered_Data.csv
    NIH_filtered_14classes.csv
    class_statistics.csv
    flipped_biased_train_set.csv
    flipped_total_biased_train_set_split.csv
    flipped_total_biased_val_set_split.csv
    flipped_total_unbiased_train_set_split.csv
    flipped_total_unbiased_val_set_split.csv
    flipped_train_set.csv
    flipped_unbiased_test_set.csv
    flipped_unbiased_train_set.csv
    test.csv
    test_dataset.csv
    total_biased_train_set.csv
    total_biased_train_set_split.csv
    total_biased_val_set_split.csv
    total_unbiased_test_set.csv
    total_unbiased_train_set.csv
    total_unbiased_

In [37]:
# ================================================================
# STEP 15 — INSPECT OFFICIAL BIASPRUNER PRUNING CODE
# READ ONLY
# ================================================================

import os
import re

repo_dir = "/kaggle/working/BiasPruner"

files_to_search = [
    os.path.join(repo_dir, "models.py"),
    os.path.join(repo_dir, "functions.py"),
    os.path.join(repo_dir, "train.py"),
    os.path.join(repo_dir, "trainer.py"),
]

keywords = [
    "prun",
    "mask",
    "cfg",
    "freeze",
    "subnetwork",
    "protected",
    "apply_unit",
]

print("=" * 70)
print("BIASPRUNER — STEP 15: PRUNING CODE INSPECTION")
print("=" * 70)

for filepath in files_to_search:

    if not os.path.exists(filepath):
        continue

    print()
    print("-" * 70)
    print(f"FILE: {os.path.basename(filepath)}")
    print("-" * 70)

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    matches = []

    for i, line in enumerate(lines):
        lower = line.lower()

        if any(keyword in lower for keyword in keywords):
            matches.append(i)

    # Print matching lines with surrounding context
    printed_ranges = []

    for idx in matches:
        start = max(0, idx - 4)
        end = min(len(lines), idx + 7)

        # Avoid printing the same region repeatedly
        if any(
            start >= a and end <= b
            for a, b in printed_ranges
        ):
            continue

        printed_ranges.append((start, end))

        print()
        print(f"--- lines {start + 1}–{end} ---")

        for j in range(start, end):
            print(f"{j + 1:4d}: {lines[j].rstrip()}")

print()
print("=" * 70)
print("STEP 15 COMPLETE")
print("READ ONLY — NO MODEL CHANGES")
print("=" * 70)

BIASPRUNER — STEP 15: PRUNING CODE INSPECTION

----------------------------------------------------------------------
FILE: models.py
----------------------------------------------------------------------

--- lines 22–32 ---
  22: relu2 = nn.ReLU(inplace=True)
  23: class BasicBlock(nn.Module):
  24:     expansion=1
  25: 
  26:     def __init__(self, inplanes, planes, stride=1, downsample=None,cfg=None):
  27:         #print('cfg=',cfg)
  28:         super(BasicBlock, self).__init__()
  29:         self.conv1 = conv3x3(inplanes, cfg, stride)
  30:         self.bn1 = nn.BatchNorm2d(cfg)
  31:         self.relu1 = nn.ReLU(inplace=True)
  32:         self.conv2 = conv3x3(cfg, planes)

--- lines 23–33 ---
  23: class BasicBlock(nn.Module):
  24:     expansion=1
  25: 
  26:     def __init__(self, inplanes, planes, stride=1, downsample=None,cfg=None):
  27:         #print('cfg=',cfg)
  28:         super(BasicBlock, self).__init__()
  29:         self.conv1 = conv3x3(inplanes, cfg, stride)

In [38]:
# ================================================================
# STEP 16 — LOCATE EXACT PRUNING / MASK FUNCTIONS
# READ ONLY
# ================================================================

import os
import re

repo_dir = "/kaggle/working/BiasPruner"

files_to_check = [
    os.path.join(repo_dir, "functions.py"),
    os.path.join(repo_dir, "models.py"),
    os.path.join(repo_dir, "train.py"),
    os.path.join(repo_dir, "trainer.py"),
]

patterns = [
    r"def .*prun",
    r"def .*mask",
    r"def .*unit",
    r"def .*cfg",
    r"def .*freeze",
    r"def .*subnetwork",
    r"def .*protected",
    r"prun",
    r"mask",
    r"apply_unit",
    r"freeze_previous",
    r"protected_snapshot",
    r"max_output",
]

print("=" * 70)
print("BIASPRUNER — STEP 16: EXACT PRUNING FUNCTIONS")
print("=" * 70)

for filepath in files_to_check:

    print()
    print("-" * 70)
    print(os.path.basename(filepath))
    print("-" * 70)

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    found = set()

    for i, line in enumerate(lines):

        if line.lstrip().startswith("def "):
            for pattern in patterns[:7]:
                if re.search(pattern, line, re.IGNORECASE):
                    found.add(i)

        else:
            for pattern in patterns[7:]:
                if re.search(pattern, line, re.IGNORECASE):
                    found.add(i)

    for i in sorted(found):
        start = max(0, i - 5)
        end = min(len(lines), i + 12)

        print()
        print(f"--- around line {i + 1} ---")

        for j in range(start, end):
            print(f"{j + 1:4d}: {lines[j].rstrip()}")

print()
print("=" * 70)
print("STEP 16 COMPLETE")
print("READ ONLY — NO MODEL CHANGES")
print("=" * 70)

BIASPRUNER — STEP 16: EXACT PRUNING FUNCTIONS

----------------------------------------------------------------------
functions.py
----------------------------------------------------------------------

----------------------------------------------------------------------
models.py
----------------------------------------------------------------------

--- around line 26 ---
  21: 
  22: relu2 = nn.ReLU(inplace=True)
  23: class BasicBlock(nn.Module):
  24:     expansion=1
  25: 
  26:     def __init__(self, inplanes, planes, stride=1, downsample=None,cfg=None):
  27:         #print('cfg=',cfg)
  28:         super(BasicBlock, self).__init__()
  29:         self.conv1 = conv3x3(inplanes, cfg, stride)
  30:         self.bn1 = nn.BatchNorm2d(cfg)
  31:         self.relu1 = nn.ReLU(inplace=True)
  32:         self.conv2 = conv3x3(cfg, planes)
  33:         self.bn2 = nn.BatchNorm2d(planes)
  34:         self.downsample = downsample
  35:         self.stride = stride
  36: 
  37:     def f

In [39]:
# ============================================================
# STEP 17 — TASK 1 PROTECTED MASK (DRY RUN)
# ============================================================
# IMPORTANT:
# - No model weights are changed.
# - No training is performed.
# - This creates a functional mask over the 15,168 ReLU units.
# - This is our controlled deviation from structural pruning.
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
TASK1_DIR = "/kaggle/working/biaspruner_checkpoints/task1"

RANKING_PATH = os.path.join(
    TASK1_DIR,
    "task1_bias_ranking.csv"
)

PLAN_PATH = os.path.join(
    TASK1_DIR,
    "task1_pruning_plan.txt"
)

MASK_PATH = os.path.join(
    TASK1_DIR,
    "task1_protected_relu_mask.npz"
)

# ------------------------------------------------------------
# 2. ReLU layers used for BiasPruner scoring
# ------------------------------------------------------------
RELU_NAMES = [
    "relu",
    "layer1.0.relu",
    "layer1.1.relu",
    "layer1.2.relu",
    "layer2.0.relu",
    "layer2.1.relu",
    "layer2.2.relu",
    "layer2.3.relu",
    "layer3.0.relu",
    "layer3.1.relu",
    "layer3.2.relu",
    "layer3.3.relu",
    "layer3.4.relu",
    "layer3.5.relu",
    "layer4.0.relu",
    "layer4.1.relu",
    "layer4.2.relu",
]

RELU_CHANNELS = {
    "relu": 64,

    "layer1.0.relu": 256,
    "layer1.1.relu": 256,
    "layer1.2.relu": 256,

    "layer2.0.relu": 512,
    "layer2.1.relu": 512,
    "layer2.2.relu": 512,
    "layer2.3.relu": 512,

    "layer3.0.relu": 1024,
    "layer3.1.relu": 1024,
    "layer3.2.relu": 1024,
    "layer3.3.relu": 1024,
    "layer3.4.relu": 1024,
    "layer3.5.relu": 1024,

    "layer4.0.relu": 2048,
    "layer4.1.relu": 2048,
    "layer4.2.relu": 2048,
}

TOTAL_UNITS = sum(RELU_CHANNELS.values())

print("Expected total ReLU units:", TOTAL_UNITS)

assert TOTAL_UNITS == 15168, (
    f"Unexpected number of ReLU units: {TOTAL_UNITS}"
)

# ------------------------------------------------------------
# 3. Load the verified bias ranking
# ------------------------------------------------------------
import pandas as pd

ranking = pd.read_csv(RANKING_PATH)

print("\nRanking shape:", ranking.shape)
print("Columns:", ranking.columns.tolist())

assert len(ranking) == TOTAL_UNITS
assert list(ranking.columns) == [
    "rank",
    "layer",
    "channel",
    "bias_score"
]

# Verify every layer/channel appears exactly once
assert ranking[["layer", "channel"]].duplicated().sum() == 0

print("\nRanking verification: PASSED")

# ------------------------------------------------------------
# 4. Bias-pruning ratio
# ------------------------------------------------------------
gamma = 0.60

num_pruned = int(np.ceil(gamma * TOTAL_UNITS))
num_kept = TOTAL_UNITS - num_pruned

print("\nPruning ratio gamma:", gamma)
print("Units to prune:", num_pruned)
print("Units to keep:", num_kept)

assert num_pruned == 9101
assert num_kept == 6067

# ------------------------------------------------------------
# 5. Create masks
#
# protected_mask[layer] = 1
#     -> unit is retained/protected after Task 1
#
# protected_mask[layer] = 0
#     -> unit is pruned/free for future tasks
#
# IMPORTANT:
# BiasPruner ranks HIGH bias scores first.
# Therefore the highest 60% are masked.
# ------------------------------------------------------------
protected_masks = {}

for layer in RELU_NAMES:
    protected_masks[layer] = np.ones(
        RELU_CHANNELS[layer],
        dtype=np.float32
    )

# Highest-bias 60% are removed from Task-1 subnetwork
pruned_rows = ranking.sort_values(
    "bias_score",
    ascending=False
).head(num_pruned)

for _, row in pruned_rows.iterrows():

    layer = row["layer"]
    channel = int(row["channel"])

    protected_masks[layer][channel] = 0.0

# ------------------------------------------------------------
# 6. Validate mask counts
# ------------------------------------------------------------
total_protected = 0
total_free = 0

print("\nLayer-wise mask statistics:")

for layer in RELU_NAMES:

    mask = protected_masks[layer]

    protected = int(mask.sum())
    free = int((mask == 0).sum())

    total_protected += protected
    total_free += free

    print(
        f"{layer:20s} | "
        f"channels={len(mask):4d} | "
        f"protected={protected:4d} | "
        f"free={free:4d}"
    )

print("\nTOTAL")
print("Protected units:", total_protected)
print("Free units:", total_free)
print("Total units:", total_protected + total_free)

assert total_protected == num_kept
assert total_free == num_pruned
assert total_protected + total_free == TOTAL_UNITS

print("\nMask count verification: PASSED")

# ------------------------------------------------------------
# 7. Verify that the mask corresponds EXACTLY to the ranking
# ------------------------------------------------------------
# The first 9101 ranking rows must be free (=0)
# The remaining 6067 must be protected (=1).

ranking_sorted = ranking.sort_values(
    "bias_score",
    ascending=False
).reset_index(drop=True)

mismatch = 0

for i, row in ranking_sorted.iterrows():

    layer = row["layer"]
    channel = int(row["channel"])

    expected = 0.0 if i < num_pruned else 1.0
    actual = protected_masks[layer][channel]

    if actual != expected:
        mismatch += 1

print("\nRanking ↔ mask mismatches:", mismatch)

assert mismatch == 0

print("Ranking-to-mask verification: PASSED")

# ------------------------------------------------------------
# 8. Save the mask
# ------------------------------------------------------------
save_dict = {}

for layer in RELU_NAMES:
    # npz keys cannot contain dots conveniently for some workflows,
    # so use a deterministic safe key.
    key = layer.replace(".", "_")
    save_dict[key] = protected_masks[layer]

np.savez(
    MASK_PATH,
    **save_dict
)

print("\nSaved mask:")
print(MASK_PATH)

# ------------------------------------------------------------
# 9. Save human-readable summary
# ------------------------------------------------------------
with open(
    os.path.join(TASK1_DIR, "task1_mask_summary.txt"),
    "w"
) as f:

    f.write("BiasPruner Task 1 Functional Mask\n")
    f.write("=================================\n\n")

    f.write("IMPORTANT:\n")
    f.write(
        "This is a mask-based functional implementation of "
        "bias-aware pruning.\n"
    )
    f.write(
        "It is NOT claimed to be structural channel pruning.\n\n"
    )

    f.write(f"Total ReLU units: {TOTAL_UNITS}\n")
    f.write(f"Pruning ratio gamma: {gamma}\n")
    f.write(f"Free/masked units: {num_pruned}\n")
    f.write(f"Protected Task-1 units: {num_kept}\n\n")

    for layer in RELU_NAMES:
        protected = int(protected_masks[layer].sum())
        free = int((protected_masks[layer] == 0).sum())

        f.write(
            f"{layer}: "
            f"protected={protected}, "
            f"free={free}\n"
        )

print("\nSummary saved.")

# ------------------------------------------------------------
# 10. CRITICAL SAFETY CHECK
# ------------------------------------------------------------
# Make sure model_task1 has NOT been modified.
# We only inspect its state; we do not apply the mask yet.
# ------------------------------------------------------------

print("\nModel safety check:")
print("model_task1 exists:", "model_task1" in globals())

if "model_task1" in globals():
    print(
        "model_task1 parameter count:",
        sum(p.numel() for p in model_task1.parameters())
    )

print("\n================================================")
print("STEP 17 COMPLETE — DRY RUN ONLY")
print("No weights changed.")
print("No training performed.")
print("No mask applied to the model yet.")
print("================================================")

Expected total ReLU units: 15168

Ranking shape: (15168, 4)
Columns: ['rank', 'layer', 'channel', 'bias_score']

Ranking verification: PASSED

Pruning ratio gamma: 0.6
Units to prune: 9101
Units to keep: 6067

Layer-wise mask statistics:
relu                 | channels=  64 | protected=  64 | free=   0
layer1.0.relu        | channels= 256 | protected= 247 | free=   9
layer1.1.relu        | channels= 256 | protected= 238 | free=  18
layer1.2.relu        | channels= 256 | protected= 237 | free=  19
layer2.0.relu        | channels= 512 | protected= 471 | free=  41
layer2.1.relu        | channels= 512 | protected= 414 | free=  98
layer2.2.relu        | channels= 512 | protected= 352 | free= 160
layer2.3.relu        | channels= 512 | protected= 317 | free= 195
layer3.0.relu        | channels=1024 | protected= 729 | free= 295
layer3.1.relu        | channels=1024 | protected= 445 | free= 579
layer3.2.relu        | channels=1024 | protected= 330 | free= 694
layer3.3.relu        | channels=1024

In [40]:
# ============================================================
# STEP 18 — FORWARD-PASS MASK VALIDATION
# ============================================================
# No training.
# No optimizer.
# No weight modification.
# The mask is applied temporarily through forward hooks.
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------

assert "model_task1" in globals(), "model_task1 not found"
assert "protected_masks" in globals(), "protected_masks not found"
assert "task1_analysis_loader" in globals(), "task1_analysis_loader not found"

model_task1.eval()

device = next(model_task1.parameters()).device

print("Device:", device)
print("Model mode:", "eval" if not model_task1.training else "train")

# ------------------------------------------------------------
# 2. Get one deterministic Task-1 batch
# ------------------------------------------------------------

batch = next(iter(task1_analysis_loader))

images = batch["image"].to(device)
labels = batch["label"].to(device)

print("\nInput shape:", tuple(images.shape))
print("Labels shape:", tuple(labels.shape))

# ------------------------------------------------------------
# 3. Run ORIGINAL model first
# ------------------------------------------------------------

with torch.no_grad():
    original_logits = model_task1(images)

print("\nOriginal forward pass:")
print("Logits shape:", tuple(original_logits.shape))
print("Logits finite:", torch.isfinite(original_logits).all().item())

# ------------------------------------------------------------
# 4. Register temporary masking hooks
# ------------------------------------------------------------

hooks = []

activation_stats = {}

def make_mask_hook(layer_name):

    mask_tensor = torch.tensor(
        protected_masks[layer_name],
        dtype=torch.float32,
        device=device
    ).view(1, -1, 1, 1)

    def hook(module, inputs, output):

        # Save statistics BEFORE masking
        with torch.no_grad():

            pre_nonzero = int((output != 0).sum().item())

            # Apply protected/free mask
            masked_output = output * mask_tensor

            post_nonzero = int(
                (masked_output != 0).sum().item()
            )

            # Verify free channels are numerically zero
            free_mask = (mask_tensor == 0)

            if free_mask.any():
                free_values = masked_output[
                    :, free_mask.view(-1), :, :
                ]

                max_free_abs = (
                    free_values.abs().max().item()
                    if free_values.numel() > 0
                    else 0.0
                )
            else:
                max_free_abs = 0.0

            activation_stats[layer_name] = {
                "channels": output.shape[1],
                "protected": int(mask_tensor.sum().item()),
                "free": int(
                    (mask_tensor == 0).sum().item()
                ),
                "pre_nonzero": pre_nonzero,
                "post_nonzero": post_nonzero,
                "max_abs_free_after_mask": max_free_abs,
            }

        return masked_output

    return hook


# Attach hooks to exactly the 17 ReLU layers
for name, module in model_task1.named_modules():

    if name in protected_masks:

        h = module.register_forward_hook(
            make_mask_hook(name)
        )

        hooks.append(h)

print("\nRegistered masking hooks:", len(hooks))

assert len(hooks) == 17

# ------------------------------------------------------------
# 5. Masked forward pass
# ------------------------------------------------------------

with torch.no_grad():
    masked_logits = model_task1(images)

print("\nMasked forward pass:")
print("Logits shape:", tuple(masked_logits.shape))
print("Logits finite:", torch.isfinite(masked_logits).all().item())

# ------------------------------------------------------------
# 6. Inspect every masked layer
# ------------------------------------------------------------

print("\nLayer-wise mask validation:")
print("-" * 95)

all_free_zero = True

for layer_name in protected_masks:

    s = activation_stats[layer_name]

    print(
        f"{layer_name:20s} | "
        f"protected={s['protected']:4d} | "
        f"free={s['free']:4d} | "
        f"max free abs={s['max_abs_free_after_mask']:.3e}"
    )

    if s["max_abs_free_after_mask"] != 0.0:
        all_free_zero = False

print("-" * 95)

print(
    "\nAll free channels exactly zero:",
    all_free_zero
)

assert all_free_zero, (
    "ERROR: Some free channels are not zero after masking."
)

# ------------------------------------------------------------
# 7. Verify protected channels were NOT zeroed by the mask
# ------------------------------------------------------------

protected_nonzero_layers = 0

for layer_name, s in activation_stats.items():

    if s["protected"] > 0:

        # post_nonzero should be > 0 for normal activations
        if s["post_nonzero"] > 0:
            protected_nonzero_layers += 1

print(
    "Layers with non-zero protected activation:",
    protected_nonzero_layers,
    "/ 17"
)

assert protected_nonzero_layers > 0

# ------------------------------------------------------------
# 8. Compare original vs masked model output
# ------------------------------------------------------------

logit_difference = (
    masked_logits - original_logits
).abs()

print("\nOutput comparison:")
print(
    "Maximum absolute logit difference:",
    logit_difference.max().item()
)

print(
    "Mean absolute logit difference:",
    logit_difference.mean().item()
)

print(
    "Original predictions:",
    original_logits.argmax(dim=1).bincount(
        minlength=2
    ).tolist()
)

print(
    "Masked predictions:",
    masked_logits.argmax(dim=1).bincount(
        minlength=2
    ).tolist()
)

# ------------------------------------------------------------
# 9. Remove ALL temporary hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

hooks.clear()

print("\nTemporary hooks removed:", len(hooks))

# ------------------------------------------------------------
# 10. Verify normal forward works again
# ------------------------------------------------------------

with torch.no_grad():
    restored_logits = model_task1(images)

restore_difference = (
    restored_logits - original_logits
).abs()

print(
    "Restored-vs-original max difference:",
    restore_difference.max().item()
)

assert restore_difference.max().item() == 0.0

# ------------------------------------------------------------
# 11. Final safety message
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 18 COMPLETE")
print("=" * 60)

print("✓ Mask applied only through temporary hooks")
print("✓ No model weights changed")
print("✓ Free channels verified as zero")
print("✓ Protected channels remain active")
print("✓ Hooks removed")
print("✓ Original model output restored")
print("=" * 60)

Device: cuda:0
Model mode: eval

Input shape: (32, 3, 224, 224)
Labels shape: (32,)

Original forward pass:
Logits shape: (32, 2)
Logits finite: True

Registered masking hooks: 17


RuntimeError: The size of tensor a (64) must match the size of tensor b (256) at non-singleton dimension 1

In [41]:
# ============================================================
# STEP 18A — VERIFY RELEVANT MODULES AND MASK DIMENSIONS
# ============================================================

print("Checking actual ResNet-50 ReLU modules against masks...\n")

relu_module_info = []

for name, module in model_task1.named_modules():

    if name in protected_masks:

        # ReLU itself does not have a channel attribute,
        # so infer it from the mask and verify it dynamically
        # using a temporary forward hook.

        mask_len = len(protected_masks[name])

        relu_module_info.append(
            (name, type(module).__name__, mask_len)
        )

for name, module_type, mask_len in relu_module_info:

    print(
        f"{name:20s} | "
        f"{module_type:12s} | "
        f"mask channels = {mask_len}"
    )

print("\nNumber of matched ReLU modules:", len(relu_module_info))

assert len(relu_module_info) == 17

print("\nNow checking actual activation channel counts...")

actual_channels = {}

temporary_hooks = []

def make_shape_hook(name):

    def hook(module, inputs, output):

        actual_channels[name] = output.shape[1]

    return hook


for name, module in model_task1.named_modules():

    if name in protected_masks:

        temporary_hooks.append(
            module.register_forward_hook(
                make_shape_hook(name)
            )
        )

# One deterministic batch
with torch.no_grad():
    _ = model_task1(images)

for h in temporary_hooks:
    h.remove()

print()

for name in RELU_NAMES:

    actual = actual_channels[name]
    expected = len(protected_masks[name])

    print(
        f"{name:20s} | "
        f"actual={actual:4d} | "
        f"mask={expected:4d} | "
        f"{'OK' if actual == expected else 'MISMATCH'}"
    )

    assert actual == expected

print("\n" + "=" * 60)
print("STEP 18A PASSED")
print("=" * 60)
print("All 17 ReLU modules match their mask dimensions.")
print("No weights changed.")
print("=" * 60)

Checking actual ResNet-50 ReLU modules against masks...

relu                 | ReLU         | mask channels = 64
layer1.0.relu        | ReLU         | mask channels = 256
layer1.1.relu        | ReLU         | mask channels = 256
layer1.2.relu        | ReLU         | mask channels = 256
layer2.0.relu        | ReLU         | mask channels = 512
layer2.1.relu        | ReLU         | mask channels = 512
layer2.2.relu        | ReLU         | mask channels = 512
layer2.3.relu        | ReLU         | mask channels = 512
layer3.0.relu        | ReLU         | mask channels = 1024
layer3.1.relu        | ReLU         | mask channels = 1024
layer3.2.relu        | ReLU         | mask channels = 1024
layer3.3.relu        | ReLU         | mask channels = 1024
layer3.4.relu        | ReLU         | mask channels = 1024
layer3.5.relu        | ReLU         | mask channels = 1024
layer4.0.relu        | ReLU         | mask channels = 2048
layer4.1.relu        | ReLU         | mask channels = 2048
layer4.2

RuntimeError: The size of tensor a (64) must match the size of tensor b (256) at non-singleton dimension 1

In [42]:
# ============================================================
# STEP 18B — FORCE CLEANUP OF TEMPORARY FORWARD HOOKS
# ============================================================

removed = 0

for name, module in model_task1.named_modules():

    # Remove forward hooks registered on this module
    if len(module._forward_hooks) > 0:

        count = len(module._forward_hooks)

        module._forward_hooks.clear()

        removed += count

        print(
            f"Removed {count} forward hook(s) from: {name}"
        )

print("\nTotal forward hooks removed:", removed)

# Also clear the Python-side hook list if it exists
if "hooks" in globals():
    hooks.clear()

print("Python hook list cleared.")

# ------------------------------------------------------------
# Verify no forward hooks remain
# ------------------------------------------------------------

remaining = []

for name, module in model_task1.named_modules():

    if len(module._forward_hooks) > 0:
        remaining.append(
            (name, len(module._forward_hooks))
        )

print("\nRemaining forward hooks:", len(remaining))

assert len(remaining) == 0

# ------------------------------------------------------------
# Verify normal forward pass
# ------------------------------------------------------------

model_task1.eval()

with torch.no_grad():
    test_logits = model_task1(images)

print("\nNormal forward pass after cleanup:")
print("Logits shape:", tuple(test_logits.shape))
print("Logits finite:", torch.isfinite(test_logits).all().item())

assert test_logits.shape == (images.shape[0], 2)
assert torch.isfinite(test_logits).all()

print("\n" + "=" * 60)
print("STEP 18B PASSED")
print("=" * 60)
print("All temporary hooks removed.")
print("Normal ResNet-50 forward pass works.")
print("No weights were modified.")
print("=" * 60)

Removed 2 forward hook(s) from: relu
Removed 2 forward hook(s) from: layer1.0.relu
Removed 2 forward hook(s) from: layer1.1.relu
Removed 2 forward hook(s) from: layer1.2.relu
Removed 2 forward hook(s) from: layer2.0.relu
Removed 2 forward hook(s) from: layer2.1.relu
Removed 2 forward hook(s) from: layer2.2.relu
Removed 2 forward hook(s) from: layer2.3.relu
Removed 2 forward hook(s) from: layer3.0.relu
Removed 2 forward hook(s) from: layer3.1.relu
Removed 2 forward hook(s) from: layer3.2.relu
Removed 2 forward hook(s) from: layer3.3.relu
Removed 2 forward hook(s) from: layer3.4.relu
Removed 2 forward hook(s) from: layer3.5.relu
Removed 2 forward hook(s) from: layer4.0.relu
Removed 2 forward hook(s) from: layer4.1.relu
Removed 2 forward hook(s) from: layer4.2.relu

Total forward hooks removed: 34
Python hook list cleared.

Remaining forward hooks: 0

Normal forward pass after cleanup:
Logits shape: (32, 2)
Logits finite: True

STEP 18B PASSED
All temporary hooks removed.
Normal ResNet-50

In [43]:
# ============================================================
# STEP 18C — SAFE FUNCTIONAL MASK FORWARD VALIDATION
# ============================================================

model_task1.eval()

device = next(model_task1.parameters()).device

# ------------------------------------------------------------
# 1. Fresh deterministic batch
# ------------------------------------------------------------

batch = next(iter(task1_analysis_loader))

images = batch["image"].to(device)
labels = batch["label"].to(device)

# ------------------------------------------------------------
# 2. Original output
# ------------------------------------------------------------

with torch.no_grad():
    original_logits = model_task1(images)

print("Original logits:", tuple(original_logits.shape))

# ------------------------------------------------------------
# 3. Create masks as GPU tensors
# ------------------------------------------------------------

mask_tensors = {}

for name in RELU_NAMES:

    mask = torch.tensor(
        protected_masks[name],
        dtype=torch.float32,
        device=device
    ).view(1, -1, 1, 1)

    mask_tensors[name] = mask

# ------------------------------------------------------------
# 4. Temporary hooks
# ------------------------------------------------------------

hooks = []
mask_stats = {}

def create_safe_mask_hook(layer_name):

    def hook(module, inputs, output):

        # ----------------------------------------------------
        # Critical safety check
        # ----------------------------------------------------

        actual_channels = output.shape[1]
        mask_channels = mask_tensors[layer_name].shape[1]

        assert actual_channels == mask_channels, (
            f"CHANNEL MISMATCH at {layer_name}: "
            f"activation={actual_channels}, "
            f"mask={mask_channels}"
        )

        mask = mask_tensors[layer_name]

        # ----------------------------------------------------
        # Apply mask
        # ----------------------------------------------------

        masked_output = output * mask

        # ----------------------------------------------------
        # Verify free channels
        # ----------------------------------------------------

        free_indices = (
            protected_masks[layer_name] == 0
        )

        free_indices = torch.tensor(
            free_indices,
            dtype=torch.bool,
            device=device
        )

        if free_indices.any():

            free_values = masked_output[
                :, free_indices, :, :
            ]

            max_free_abs = (
                free_values.abs().max().item()
            )

        else:
            max_free_abs = 0.0

        # ----------------------------------------------------
        # Verify protected channels
        # ----------------------------------------------------

        protected_indices = (
            protected_masks[layer_name] == 1
        )

        protected_indices = torch.tensor(
            protected_indices,
            dtype=torch.bool,
            device=device
        )

        if protected_indices.any():

            protected_values = masked_output[
                :, protected_indices, :, :
            ]

            protected_abs_sum = (
                protected_values.abs().sum().item()
            )

        else:
            protected_abs_sum = 0.0

        mask_stats[layer_name] = {
            "activation_channels": actual_channels,
            "mask_channels": mask_channels,
            "free_channels": int(free_indices.sum()),
            "protected_channels": int(
                protected_indices.sum()
            ),
            "max_free_abs": max_free_abs,
            "protected_abs_sum": protected_abs_sum,
        }

        return masked_output

    return hook


# Register exactly one hook per ReLU
for name, module in model_task1.named_modules():

    if name in mask_tensors:

        hooks.append(
            module.register_forward_hook(
                create_safe_mask_hook(name)
            )
        )

print("Registered hooks:", len(hooks))

assert len(hooks) == 17

# ------------------------------------------------------------
# 5. Masked forward
# ------------------------------------------------------------

with torch.no_grad():
    masked_logits = model_task1(images)

print("Masked logits:", tuple(masked_logits.shape))

# ------------------------------------------------------------
# 6. Validate every layer
# ------------------------------------------------------------

print("\nLayer validation")
print("-" * 100)

all_zero = True

for name in RELU_NAMES:

    s = mask_stats[name]

    print(
        f"{name:20s} | "
        f"channels={s['activation_channels']:4d} | "
        f"protected={s['protected_channels']:4d} | "
        f"free={s['free_channels']:4d} | "
        f"max_free_abs={s['max_free_abs']:.3e}"
    )

    if s["max_free_abs"] != 0.0:
        all_zero = False

print("-" * 100)

print("\nAll free channels exactly zero:", all_zero)

assert all_zero

# ------------------------------------------------------------
# 7. Masked output sanity
# ------------------------------------------------------------

assert masked_logits.shape == original_logits.shape
assert torch.isfinite(masked_logits).all()

output_diff = (
    masked_logits - original_logits
).abs()

print(
    "\nMaximum output difference:",
    output_diff.max().item()
)

print(
    "Mean output difference:",
    output_diff.mean().item()
)

# ------------------------------------------------------------
# 8. Remove hooks immediately
# ------------------------------------------------------------

for h in hooks:
    h.remove()

hooks.clear()

print("\nTemporary hooks removed.")

# ------------------------------------------------------------
# 9. Verify complete cleanup
# ------------------------------------------------------------

remaining_hooks = 0

for name, module in model_task1.named_modules():

    remaining_hooks += len(module._forward_hooks)

print(
    "Remaining forward hooks:",
    remaining_hooks
)

assert remaining_hooks == 0

# ------------------------------------------------------------
# 10. Verify original model restored
# ------------------------------------------------------------

with torch.no_grad():
    restored_logits = model_task1(images)

restore_diff = (
    restored_logits - original_logits
).abs()

print(
    "Restored-vs-original max difference:",
    restore_diff.max().item()
)

assert restore_diff.max().item() == 0.0

# ------------------------------------------------------------
# 11. Final
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 18C PASSED")
print("=" * 65)
print("✓ 17 masks applied")
print("✓ Every mask matched its activation channel count")
print("✓ Free channels became exactly zero")
print("✓ Protected channels remained available")
print("✓ Masked logits are finite")
print("✓ All temporary hooks removed")
print("✓ Original model output restored exactly")
print("✓ No model weights changed")
print("=" * 65)

Original logits: (32, 2)
Registered hooks: 17


AssertionError: CHANNEL MISMATCH at layer1.0.relu: activation=64, mask=256

In [44]:
# ============================================================
# STEP 18D — CORRECT RESNET-50 ACTIVATION POINTS
# ============================================================

model_task1.eval()

# Remove any accidental hooks
for name, module in model_task1.named_modules():
    module._forward_hooks.clear()

print("All forward hooks cleared.")

# ------------------------------------------------------------
# Activation points:
#
# stem ReLU
# +
# output of each Bottleneck block
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task1.relu,

    "layer1.0": model_task1.layer1[0],
    "layer1.1": model_task1.layer1[1],
    "layer1.2": model_task1.layer1[2],

    "layer2.0": model_task1.layer2[0],
    "layer2.1": model_task1.layer2[1],
    "layer2.2": model_task1.layer2[2],
    "layer2.3": model_task1.layer2[3],

    "layer3.0": model_task1.layer3[0],
    "layer3.1": model_task1.layer3[1],
    "layer3.2": model_task1.layer3[2],
    "layer3.3": model_task1.layer3[3],
    "layer3.4": model_task1.layer3[4],
    "layer3.5": model_task1.layer3[5],

    "layer4.0": model_task1.layer4[0],
    "layer4.1": model_task1.layer4[1],
    "layer4.2": model_task1.layer4[2],
}

expected_channels = {
    "relu": 64,

    "layer1.0": 256,
    "layer1.1": 256,
    "layer1.2": 256,

    "layer2.0": 512,
    "layer2.1": 512,
    "layer2.2": 512,
    "layer2.3": 512,

    "layer3.0": 1024,
    "layer3.1": 1024,
    "layer3.2": 1024,
    "layer3.3": 1024,
    "layer3.4": 1024,
    "layer3.5": 1024,

    "layer4.0": 2048,
    "layer4.1": 2048,
    "layer4.2": 2048,
}

captured = {}

hooks = []

def make_capture_hook(name):

    def hook(module, inputs, output):

        captured[name] = {
            "shape": tuple(output.shape),
            "channels": output.shape[1]
        }

    return hook


for name, module in activation_modules.items():

    hooks.append(
        module.register_forward_hook(
            make_capture_hook(name)
        )
    )

# ------------------------------------------------------------
# One forward pass
# ------------------------------------------------------------

batch = next(iter(task1_analysis_loader))

images = batch["image"].to(
    next(model_task1.parameters()).device
)

with torch.no_grad():
    logits = model_task1(images)

# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

hooks.clear()

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nActivation point verification")
print("-" * 80)

total_channels = 0

for name in expected_channels:

    actual = captured[name]["channels"]
    expected = expected_channels[name]

    print(
        f"{name:12s} | "
        f"shape={captured[name]['shape']} | "
        f"expected={expected:4d} | "
        f"actual={actual:4d} | "
        f"{'OK' if actual == expected else 'ERROR'}"
    )

    assert actual == expected

    total_channels += actual

print("-" * 80)

print("Total activation channels:", total_channels)

assert total_channels == 15168

print("\nRemaining hooks:")

remaining = sum(
    len(module._forward_hooks)
    for module in model_task1.modules()
)

print(remaining)

assert remaining == 0

print("\n" + "=" * 65)
print("STEP 18D PASSED")
print("=" * 65)
print("✓ Correct ResNet-50 activation points identified")
print("✓ Every activation point has fixed channel dimension")
print("✓ Total = 15,168 units")
print("✓ No hooks remain")
print("✓ No weights changed")
print("=" * 65)

All forward hooks cleared.

Activation point verification
--------------------------------------------------------------------------------
relu         | shape=(32, 64, 112, 112) | expected=  64 | actual=  64 | OK
layer1.0     | shape=(32, 256, 56, 56) | expected= 256 | actual= 256 | OK
layer1.1     | shape=(32, 256, 56, 56) | expected= 256 | actual= 256 | OK
layer1.2     | shape=(32, 256, 56, 56) | expected= 256 | actual= 256 | OK
layer2.0     | shape=(32, 512, 28, 28) | expected= 512 | actual= 512 | OK
layer2.1     | shape=(32, 512, 28, 28) | expected= 512 | actual= 512 | OK
layer2.2     | shape=(32, 512, 28, 28) | expected= 512 | actual= 512 | OK
layer2.3     | shape=(32, 512, 28, 28) | expected= 512 | actual= 512 | OK
layer3.0     | shape=(32, 1024, 14, 14) | expected=1024 | actual=1024 | OK
layer3.1     | shape=(32, 1024, 14, 14) | expected=1024 | actual=1024 | OK
layer3.2     | shape=(32, 1024, 14, 14) | expected=1024 | actual=1024 | OK
layer3.3     | shape=(32, 1024, 14, 14) | e

In [45]:
# ============================================================
# STEP 19 — CORRECT BIAS SCORE CALCULATION
# ============================================================
# Correct activation points:
#   stem ReLU
#   + output of every ResNet-50 Bottleneck block
#
# Partition:
#   S_ba = correctly classified
#   S_bc = incorrectly classified
#
# Bias score:
#
# S_c,n =
#   mean variance over correctly classified samples
#   -
#   mean variance over incorrectly classified samples
#
# Final channel score:
#   average over classes
#
# NOTE:
# This uses the repository-aligned correct/incorrect partition,
# while using the paper's spatial-variance score.
# This is explicitly documented rather than claiming the literal
# tau-based formal definition from the paper.
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Safety / setup
# ------------------------------------------------------------

model_task1.eval()

device = next(model_task1.parameters()).device

print("Device:", device)
print("Model:", type(model_task1).__name__)

# ------------------------------------------------------------
# 2. Correct activation points
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task1.relu,

    "layer1.0": model_task1.layer1[0],
    "layer1.1": model_task1.layer1[1],
    "layer1.2": model_task1.layer1[2],

    "layer2.0": model_task1.layer2[0],
    "layer2.1": model_task1.layer2[1],
    "layer2.2": model_task1.layer2[2],
    "layer2.3": model_task1.layer2[3],

    "layer3.0": model_task1.layer3[0],
    "layer3.1": model_task1.layer3[1],
    "layer3.2": model_task1.layer3[2],
    "layer3.3": model_task1.layer3[3],
    "layer3.4": model_task1.layer3[4],
    "layer3.5": model_task1.layer3[5],

    "layer4.0": model_task1.layer4[0],
    "layer4.1": model_task1.layer4[1],
    "layer4.2": model_task1.layer4[2],
}

EXPECTED_CHANNELS = {
    "relu": 64,

    "layer1.0": 256,
    "layer1.1": 256,
    "layer1.2": 256,

    "layer2.0": 512,
    "layer2.1": 512,
    "layer2.2": 512,
    "layer2.3": 512,

    "layer3.0": 1024,
    "layer3.1": 1024,
    "layer3.2": 1024,
    "layer3.3": 1024,
    "layer3.4": 1024,
    "layer3.5": 1024,

    "layer4.0": 2048,
    "layer4.1": 2048,
    "layer4.2": 2048,
}

assert sum(EXPECTED_CHANNELS.values()) == 15168

# ------------------------------------------------------------
# 3. Deterministic analysis loader
# ------------------------------------------------------------

assert "task1_analysis_loader" in globals()

analysis_loader = task1_analysis_loader

print("Analysis samples:", len(task1_analysis_loader.dataset))

# ------------------------------------------------------------
# 4. Class names
# ------------------------------------------------------------

assert "TASK1_CLASSES" in globals()

classes = list(TASK1_CLASSES)

print("Task 1 classes:", classes)

assert len(classes) == 2

# ------------------------------------------------------------
# 5. Accumulators
#
# For every layer and class:
#
# correct_sum[layer][class][channel]
# incorrect_sum[layer][class][channel]
#
# We accumulate the spatial variance of each sample's
# feature map.
# ------------------------------------------------------------

correct_sum = {
    layer: {
        cls: np.zeros(
            EXPECTED_CHANNELS[layer],
            dtype=np.float64
        )
        for cls in classes
    }
    for layer in activation_modules
}

incorrect_sum = {
    layer: {
        cls: np.zeros(
            EXPECTED_CHANNELS[layer],
            dtype=np.float64
        )
        for cls in classes
    }
    for layer in activation_modules
}

correct_count = {
    cls: 0
    for cls in classes
}

incorrect_count = {
    cls: 0
    for cls in classes
}

total_samples = 0
total_correct = 0
total_incorrect = 0

# ------------------------------------------------------------
# 6. Temporary activation storage
# ------------------------------------------------------------

activations = {}

hooks = []

def make_activation_hook(name):

    def hook(module, inputs, output):

        # Detach immediately.
        activations[name] = output.detach()

    return hook


for name, module in activation_modules.items():

    hooks.append(
        module.register_forward_hook(
            make_activation_hook(name)
        )
    )

assert len(hooks) == 17

# ------------------------------------------------------------
# 7. Process deterministic training data
# ------------------------------------------------------------

print("\nProcessing training samples...")

for batch_idx, batch in enumerate(analysis_loader):

    images = batch["image"].to(device)
    labels = batch["label"].to(device)

    batch_size = labels.shape[0]

    with torch.no_grad():

        logits = model_task1(images)

        predictions = logits.argmax(dim=1)

    # --------------------------------------------------------
    # Correct / incorrect partition
    # --------------------------------------------------------

    correct_mask = predictions == labels
    incorrect_mask = ~correct_mask

    batch_correct = int(correct_mask.sum().item())
    batch_incorrect = int(incorrect_mask.sum().item())

    total_samples += batch_size
    total_correct += batch_correct
    total_incorrect += batch_incorrect

    # --------------------------------------------------------
    # Process every sample
    # --------------------------------------------------------

    for local_idx in range(batch_size):

        class_idx = int(labels[local_idx].item())
        class_name = classes[class_idx]

        is_correct = bool(
            correct_mask[local_idx].item()
        )

        if is_correct:
            correct_count[class_name] += 1
        else:
            incorrect_count[class_name] += 1

        # ----------------------------------------------------
        # For every activation point:
        #
        # activation:
        # [C, H, W]
        #
        # variance:
        # one value per channel
        # ----------------------------------------------------

        for layer_name in activation_modules:

            activation = activations[layer_name][
                local_idx
            ]

            # Spatial variance over H,W.
            #
            # unbiased=False gives population variance over
            # the spatial feature map.
            variance = torch.var(
                activation,
                dim=(1, 2),
                unbiased=False
            )

            variance_np = (
                variance.cpu()
                .numpy()
                .astype(np.float64)
            )

            if is_correct:

                correct_sum[
                    layer_name
                ][class_name] += variance_np

            else:

                incorrect_sum[
                    layer_name
                ][class_name] += variance_np

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = total_samples

    if (
        batch_idx == 0
        or processed % 320 == 0
        or processed == len(analysis_loader.dataset)
    ):

        print(
            f"Processed {processed}/"
            f"{len(analysis_loader.dataset)}"
        )

# ------------------------------------------------------------
# 8. Remove hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

hooks.clear()

activations.clear()

# Verify no hooks remain
remaining_hooks = sum(
    len(module._forward_hooks)
    for module in model_task1.modules()
)

assert remaining_hooks == 0

# ------------------------------------------------------------
# 9. Partition summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PARTITION SUMMARY")
print("=" * 70)

print("Total samples:", total_samples)
print("Correct:", total_correct)
print("Incorrect:", total_incorrect)

print(
    "Accuracy:",
    total_correct / total_samples
)

assert total_samples == len(
    analysis_loader.dataset
)

assert total_correct + total_incorrect == total_samples

for cls in classes:

    total_cls = (
        correct_count[cls]
        + incorrect_count[cls]
    )

    print(
        f"{cls:8s} | "
        f"total={total_cls:4d} | "
        f"correct={correct_count[cls]:4d} | "
        f"incorrect={incorrect_count[cls]:4d} | "
        f"accuracy="
        f"{correct_count[cls] / total_cls:.4f}"
    )

# ------------------------------------------------------------
# 10. Compute paper-style spatial variance bias scores
# ------------------------------------------------------------
#
# For each class:
#
# score[c,n] =
#       mean_correct_variance
#       -
#       mean_incorrect_variance
#
# Then final score[n] =
#       mean over classes
#
# ------------------------------------------------------------

layer_scores = {}

all_scores = []

for layer_name in activation_modules:

    channels = EXPECTED_CHANNELS[layer_name]

    class_scores = {}

    for cls in classes:

        nc = correct_count[cls]
        ni = incorrect_count[cls]

        if nc == 0 or ni == 0:

            raise RuntimeError(
                f"Cannot compute bias score for "
                f"class={cls}, layer={layer_name}: "
                f"correct={nc}, incorrect={ni}"
            )

        mean_correct = (
            correct_sum[layer_name][cls] / nc
        )

        mean_incorrect = (
            incorrect_sum[layer_name][cls] / ni
        )

        score = (
            mean_correct
            - mean_incorrect
        )

        class_scores[cls] = score

    # Average class-specific scores
    final_score = np.mean(
        np.stack(
            list(class_scores.values()),
            axis=0
        ),
        axis=0
    )

    layer_scores[layer_name] = final_score

    all_scores.append(final_score)

# ------------------------------------------------------------
# 11. Global statistics
# ------------------------------------------------------------

flat_scores = np.concatenate(all_scores)

print("\n" + "=" * 70)
print("BIAS SCORE SUMMARY")
print("=" * 70)

print("Total units:", len(flat_scores))
print("Min:", float(flat_scores.min()))
print("Max:", float(flat_scores.max()))
print("Mean:", float(flat_scores.mean()))
print("Median:", float(np.median(flat_scores)))

assert len(flat_scores) == 15168
assert np.isfinite(flat_scores).all()

# ------------------------------------------------------------
# 12. Build ranking
# ------------------------------------------------------------

ranking_rows = []

rank = 1

for layer_name in activation_modules:

    scores = layer_scores[layer_name]

    for channel, score in enumerate(scores):

        ranking_rows.append({
            "rank": None,
            "layer": layer_name,
            "channel": int(channel),
            "bias_score": float(score)
        })

ranking_df = pd.DataFrame(ranking_rows)

ranking_df = ranking_df.sort_values(
    "bias_score",
    ascending=False
).reset_index(drop=True)

ranking_df["rank"] = np.arange(
    1,
    len(ranking_df) + 1
)

# ------------------------------------------------------------
# 13. Verify ranking
# ------------------------------------------------------------

assert len(ranking_df) == 15168

assert ranking_df[
    ["layer", "channel"]
].duplicated().sum() == 0

assert ranking_df["rank"].tolist() == list(
    range(1, 15169)
)

assert np.isfinite(
    ranking_df["bias_score"]
).all()

# ------------------------------------------------------------
# 14. Display top 20
# ------------------------------------------------------------

print("\nTop 20 highest-bias activation units:")

print(
    ranking_df.head(20).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 15. Save CORRECTED results
# ------------------------------------------------------------

TASK1_DIR = "/kaggle/working/biaspruner_checkpoints/task1"

CORRECTED_NPZ = os.path.join(
    TASK1_DIR,
    "task1_correct_spatial_variance_bias_scores.npz"
)

CORRECTED_CSV = os.path.join(
    TASK1_DIR,
    "task1_correct_bias_ranking.csv"
)

CORRECTED_SUMMARY = os.path.join(
    TASK1_DIR,
    "task1_correct_bias_score_summary.txt"
)

np.savez(
    CORRECTED_NPZ,
    **{
        layer: layer_scores[layer]
        for layer in activation_modules
    }
)

ranking_df.to_csv(
    CORRECTED_CSV,
    index=False
)

with open(CORRECTED_SUMMARY, "w") as f:

    f.write(
        "Corrected Task 1 Bias Score Calculation\n"
    )
    f.write(
        "========================================\n\n"
    )

    f.write(
        "Activation points: stem ReLU + "
        "Bottleneck block outputs\n"
    )

    f.write(
        "Partition: repository-aligned "
        "correct / incorrect\n"
    )

    f.write(
        "Bias score: spatial variance difference\n"
    )

    f.write(
        "Classes: " + ", ".join(classes) + "\n\n"
    )

    f.write(
        f"Total samples: {total_samples}\n"
    )

    f.write(
        f"Correct: {total_correct}\n"
    )

    f.write(
        f"Incorrect: {total_incorrect}\n"
    )

    f.write(
        f"Accuracy: "
        f"{total_correct / total_samples:.6f}\n\n"
    )

    f.write(
        f"Total units: {len(flat_scores)}\n"
    )

    f.write(
        f"Min score: {flat_scores.min():.12f}\n"
    )

    f.write(
        f"Max score: {flat_scores.max():.12f}\n"
    )

    f.write(
        f"Mean score: {flat_scores.mean():.12f}\n"
    )

    f.write(
        f"Median score: "
        f"{np.median(flat_scores):.12f}\n"
    )

print("\nSaved corrected files:")

print(CORRECTED_NPZ)
print(CORRECTED_CSV)
print(CORRECTED_SUMMARY)

print("\n" + "=" * 70)
print("STEP 19 COMPLETE")
print("=" * 70)

print("✓ Correct activation points")
print("✓ Deterministic analysis data")
print("✓ Correct/incorrect partition")
print("✓ Spatial variance calculated")
print("✓ 15,168 bias scores calculated")
print("✓ Corrected ranking saved")
print("✓ No model weights changed")
print("=" * 70)

Device: cuda:0
Model: ResNet
Analysis samples: 1545
Task 1 classes: ['bkl', 'mel']

Processing training samples...
Processed 32/1545
Processed 320/1545
Processed 640/1545
Processed 960/1545
Processed 1280/1545
Processed 1545/1545

PARTITION SUMMARY
Total samples: 1545
Correct: 1185
Incorrect: 360
Accuracy: 0.7669902912621359
bkl      | total= 772 | correct= 510 | incorrect= 262 | accuracy=0.6606
mel      | total= 773 | correct= 675 | incorrect=  98 | accuracy=0.8732

BIAS SCORE SUMMARY
Total units: 15168
Min: -0.5099906370126117
Max: 9.822033590665779
Mean: 0.5443250408096336
Median: 0.21880589762223862

Top 20 highest-bias activation units:
 rank    layer  channel  bias_score
    1 layer4.2       29    9.822034
    2 layer3.4      234    8.074468
    3 layer4.2     1333    7.822301
    4 layer4.2      415    7.713757
    5 layer3.5      794    7.297316
    6 layer4.2      688    7.202202
    7 layer4.2     1291    7.155865
    8 layer3.5     1001    6.919786
    9 layer4.2     1028   

In [46]:
# ============================================================
# STEP 20 — CORRECT TASK-1 PRUNING MASK
# ============================================================
# Uses the CORRECTED bias ranking from Step 19.
#
# Highest-bias 60%  -> free/pruned
# Remaining 40%     -> Task-1 protected
#
# No model weights are changed.
# ============================================================

import os
import numpy as np
import pandas as pd

TASK1_DIR = "/kaggle/working/biaspruner_checkpoints/task1"

CORRECTED_RANKING = os.path.join(
    TASK1_DIR,
    "task1_correct_bias_ranking.csv"
)

CORRECTED_MASK = os.path.join(
    TASK1_DIR,
    "task1_correct_protected_mask.npz"
)

CORRECTED_PLAN = os.path.join(
    TASK1_DIR,
    "task1_correct_pruning_plan.txt"
)

# ------------------------------------------------------------
# 1. Load corrected ranking
# ------------------------------------------------------------

ranking = pd.read_csv(CORRECTED_RANKING)

print("Ranking shape:", ranking.shape)

assert len(ranking) == 15168
assert ranking["rank"].tolist() == list(range(1, 15169))

# ------------------------------------------------------------
# 2. Activation architecture
# ------------------------------------------------------------

EXPECTED_CHANNELS = {
    "relu": 64,

    "layer1.0": 256,
    "layer1.1": 256,
    "layer1.2": 256,

    "layer2.0": 512,
    "layer2.1": 512,
    "layer2.2": 512,
    "layer2.3": 512,

    "layer3.0": 1024,
    "layer3.1": 1024,
    "layer3.2": 1024,
    "layer3.3": 1024,
    "layer3.4": 1024,
    "layer3.5": 1024,

    "layer4.0": 2048,
    "layer4.1": 2048,
    "layer4.2": 2048,
}

assert sum(EXPECTED_CHANNELS.values()) == 15168

# ------------------------------------------------------------
# 3. Pruning ratio
# ------------------------------------------------------------

gamma = 0.60

total_units = 15168

num_free = int(
    np.ceil(gamma * total_units)
)

num_protected = total_units - num_free

print("Pruning ratio:", gamma)
print("Free/pruned units:", num_free)
print("Protected units:", num_protected)

assert num_free == 9101
assert num_protected == 6067

# ------------------------------------------------------------
# 4. Initialize protected masks
# ------------------------------------------------------------

protected_masks_correct = {
    layer: np.ones(
        channels,
        dtype=np.float32
    )
    for layer, channels in EXPECTED_CHANNELS.items()
}

# ------------------------------------------------------------
# 5. Mark highest-bias units as free/pruned
# ------------------------------------------------------------

highest_bias = ranking.head(num_free)

for _, row in highest_bias.iterrows():

    layer = row["layer"]
    channel = int(row["channel"])

    protected_masks_correct[layer][channel] = 0.0

# ------------------------------------------------------------
# 6. Verify counts
# ------------------------------------------------------------

total_protected = 0
total_free = 0

print("\nLayer-wise mask:")
print("-" * 85)

for layer, mask in protected_masks_correct.items():

    protected = int(mask.sum())
    free = int((mask == 0).sum())

    total_protected += protected
    total_free += free

    print(
        f"{layer:12s} | "
        f"channels={len(mask):4d} | "
        f"protected={protected:4d} | "
        f"free={free:4d}"
    )

print("-" * 85)

print("Total protected:", total_protected)
print("Total free:", total_free)
print("Total:", total_protected + total_free)

assert total_protected == 6067
assert total_free == 9101

# ------------------------------------------------------------
# 7. Verify exact ranking correspondence
# ------------------------------------------------------------

mismatches = 0

for i, row in ranking.iterrows():

    layer = row["layer"]
    channel = int(row["channel"])

    expected = (
        0.0 if i < num_free
        else 1.0
    )

    actual = protected_masks_correct[
        layer
    ][channel]

    if actual != expected:
        mismatches += 1

print("\nRanking-mask mismatches:", mismatches)

assert mismatches == 0

# ------------------------------------------------------------
# 8. Boundary inspection
# ------------------------------------------------------------

print("\nPruning boundary:")

for rank in range(
    num_free - 3,
    num_free + 3
):

    row = ranking.iloc[rank - 1]

    status = (
        "FREE/PRUNED"
        if rank <= num_free
        else "PROTECTED"
    )

    print(
        f"rank={rank:5d} | "
        f"{row['layer']:12s} | "
        f"channel={int(row['channel']):4d} | "
        f"score={row['bias_score']:.9f} | "
        f"{status}"
    )

# ------------------------------------------------------------
# 9. Save mask
# ------------------------------------------------------------

np.savez(
    CORRECTED_MASK,
    **{
        layer.replace(".", "_"): mask
        for layer, mask
        in protected_masks_correct.items()
    }
)

print("\nSaved corrected mask:")
print(CORRECTED_MASK)

# ------------------------------------------------------------
# 10. Save pruning plan
# ------------------------------------------------------------

with open(CORRECTED_PLAN, "w") as f:

    f.write(
        "Corrected Task-1 BiasPruner Functional Mask\n"
    )
    f.write(
        "===========================================\n\n"
    )

    f.write(
        "Activation points:\n"
    )

    f.write(
        "stem ReLU + ResNet-50 Bottleneck outputs\n\n"
    )

    f.write(
        "Partition:\n"
    )

    f.write(
        "repository-aligned correct / incorrect\n\n"
    )

    f.write(
        "Bias score:\n"
    )

    f.write(
        "paper spatial activation variance difference\n\n"
    )

    f.write(
        f"Total units: {total_units}\n"
    )

    f.write(
        f"Pruning ratio gamma: {gamma}\n"
    )

    f.write(
        f"Free/pruned units: {num_free}\n"
    )

    f.write(
        f"Protected units: {num_protected}\n"
    )

    f.write(
        "\nNOTE:\n"
    )

    f.write(
        "This is a mask-based functional implementation "
        "of pruning, not claimed structural channel pruning.\n"
    )

print("Saved pruning plan:")
print(CORRECTED_PLAN)

# ------------------------------------------------------------
# 11. Confirm model remains untouched
# ------------------------------------------------------------

print("\nModel parameter count:")

param_count = sum(
    p.numel()
    for p in model_task1.parameters()
)

print(param_count)

assert param_count == 23512130

print("\n" + "=" * 65)
print("STEP 20 COMPLETE")
print("=" * 65)

print("✓ Corrected ranking loaded")
print("✓ 9,101 highest-bias units marked free/pruned")
print("✓ 6,067 units protected")
print("✓ Ranking ↔ mask verified")
print("✓ Corrected mask saved")
print("✓ Model weights untouched")
print("=" * 65)

Ranking shape: (15168, 4)
Pruning ratio: 0.6
Free/pruned units: 9101
Protected units: 6067

Layer-wise mask:
-------------------------------------------------------------------------------------
relu         | channels=  64 | protected=  64 | free=   0
layer1.0     | channels= 256 | protected= 247 | free=   9
layer1.1     | channels= 256 | protected= 238 | free=  18
layer1.2     | channels= 256 | protected= 237 | free=  19
layer2.0     | channels= 512 | protected= 471 | free=  41
layer2.1     | channels= 512 | protected= 414 | free=  98
layer2.2     | channels= 512 | protected= 352 | free= 160
layer2.3     | channels= 512 | protected= 317 | free= 195
layer3.0     | channels=1024 | protected= 729 | free= 295
layer3.1     | channels=1024 | protected= 445 | free= 579
layer3.2     | channels=1024 | protected= 330 | free= 694
layer3.3     | channels=1024 | protected= 236 | free= 788
layer3.4     | channels=1024 | protected= 158 | free= 866
layer3.5     | channels=1024 | protected= 145 | fre

In [47]:
# ============================================================
# STEP 21 — CORRECTED MASK FORWARD VALIDATION
# ============================================================
# Uses:
#   - stem ReLU
#   - Bottleneck block outputs
#
# No training.
# No weight modification.
# Temporary hooks only.
# ============================================================

import torch
import numpy as np

model_task1.eval()

device = next(model_task1.parameters()).device

# ------------------------------------------------------------
# 1. Clear ALL existing forward hooks first
# ------------------------------------------------------------

for module in model_task1.modules():
    module._forward_hooks.clear()

print("Existing hooks cleared.")

# ------------------------------------------------------------
# 2. Correct activation modules
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task1.relu,

    "layer1.0": model_task1.layer1[0],
    "layer1.1": model_task1.layer1[1],
    "layer1.2": model_task1.layer1[2],

    "layer2.0": model_task1.layer2[0],
    "layer2.1": model_task1.layer2[1],
    "layer2.2": model_task1.layer2[2],
    "layer2.3": model_task1.layer2[3],

    "layer3.0": model_task1.layer3[0],
    "layer3.1": model_task1.layer3[1],
    "layer3.2": model_task1.layer3[2],
    "layer3.3": model_task1.layer3[3],
    "layer3.4": model_task1.layer3[4],
    "layer3.5": model_task1.layer3[5],

    "layer4.0": model_task1.layer4[0],
    "layer4.1": model_task1.layer4[1],
    "layer4.2": model_task1.layer4[2],
}

# ------------------------------------------------------------
# 3. Verify corrected mask exists
# ------------------------------------------------------------

assert "protected_masks_correct" in globals()

print(
    "Mask activation points:",
    len(protected_masks_correct)
)

assert len(protected_masks_correct) == 17

# ------------------------------------------------------------
# 4. Prepare mask tensors
# ------------------------------------------------------------

mask_tensors = {}

for name, mask in protected_masks_correct.items():

    mask_tensors[name] = torch.tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).view(1, -1, 1, 1)

# ------------------------------------------------------------
# 5. Get deterministic batch
# ------------------------------------------------------------

batch = next(iter(task1_analysis_loader))

images = batch["image"].to(device)

# ------------------------------------------------------------
# 6. Original forward pass
# ------------------------------------------------------------

with torch.no_grad():
    original_logits = model_task1(images)

print(
    "Original logits shape:",
    tuple(original_logits.shape)
)

# ------------------------------------------------------------
# 7. Temporary masking hooks
# ------------------------------------------------------------

hooks = []
mask_stats = {}

def make_mask_hook(layer_name):

    def hook(module, inputs, output):

        # ----------------------------------------------------
        # Special handling:
        # layer "relu" is the actual stem ReLU.
        # All others are Bottleneck outputs.
        # ----------------------------------------------------

        actual_channels = output.shape[1]
        mask = mask_tensors[layer_name]

        expected_channels = mask.shape[1]

        assert actual_channels == expected_channels, (
            f"Mismatch at {layer_name}: "
            f"activation={actual_channels}, "
            f"mask={expected_channels}"
        )

        masked_output = output * mask

        # Free channels
        free_indices = torch.tensor(
            protected_masks_correct[layer_name] == 0,
            dtype=torch.bool,
            device=device
        )

        # Protected channels
        protected_indices = torch.tensor(
            protected_masks_correct[layer_name] == 1,
            dtype=torch.bool,
            device=device
        )

        if free_indices.any():

            free_values = masked_output[
                :, free_indices, :, :
            ]

            max_free_abs = (
                free_values.abs().max().item()
            )

        else:

            max_free_abs = 0.0

        if protected_indices.any():

            protected_values = masked_output[
                :, protected_indices, :, :
            ]

            protected_abs_sum = (
                protected_values.abs().sum().item()
            )

        else:

            protected_abs_sum = 0.0

        mask_stats[layer_name] = {
            "actual_channels": actual_channels,
            "protected": int(protected_indices.sum()),
            "free": int(free_indices.sum()),
            "max_free_abs": max_free_abs,
            "protected_abs_sum": protected_abs_sum,
        }

        return masked_output

    return hook


for name, module in activation_modules.items():

    hooks.append(
        module.register_forward_hook(
            make_mask_hook(name)
        )
    )

print("Temporary hooks registered:", len(hooks))

assert len(hooks) == 17

# ------------------------------------------------------------
# 8. Masked forward pass
# ------------------------------------------------------------

with torch.no_grad():
    masked_logits = model_task1(images)

print(
    "Masked logits shape:",
    tuple(masked_logits.shape)
)

assert masked_logits.shape == original_logits.shape
assert torch.isfinite(masked_logits).all()

# ------------------------------------------------------------
# 9. Validate every activation point
# ------------------------------------------------------------

print("\nLayer validation")
print("-" * 100)

all_free_zero = True

for name in activation_modules:

    s = mask_stats[name]

    print(
        f"{name:12s} | "
        f"channels={s['actual_channels']:4d} | "
        f"protected={s['protected']:4d} | "
        f"free={s['free']:4d} | "
        f"max_free_abs={s['max_free_abs']:.3e}"
    )

    if s["max_free_abs"] != 0.0:
        all_free_zero = False

print("-" * 100)

print(
    "\nAll free channels exactly zero:",
    all_free_zero
)

assert all_free_zero

# ------------------------------------------------------------
# 10. Output difference
# ------------------------------------------------------------

output_difference = (
    masked_logits - original_logits
).abs()

print(
    "\nMaximum logit difference:",
    output_difference.max().item()
)

print(
    "Mean logit difference:",
    output_difference.mean().item()
)

# ------------------------------------------------------------
# 11. Remove temporary hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

hooks.clear()

print("\nTemporary hooks removed.")

# ------------------------------------------------------------
# 12. Verify ZERO hooks remain
# ------------------------------------------------------------

remaining_hooks = sum(
    len(module._forward_hooks)
    for module in model_task1.modules()
)

print(
    "Remaining forward hooks:",
    remaining_hooks
)

assert remaining_hooks == 0

# ------------------------------------------------------------
# 13. Verify model output is restored exactly
# ------------------------------------------------------------

with torch.no_grad():
    restored_logits = model_task1(images)

restore_difference = (
    restored_logits - original_logits
).abs()

print(
    "Restored-vs-original max difference:",
    restore_difference.max().item()
)

assert restore_difference.max().item() == 0.0

# ------------------------------------------------------------
# 14. Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 21 COMPLETE")
print("=" * 70)

print("✓ Correct activation points used")
print("✓ 17 masks applied")
print("✓ All free channels became exactly zero")
print("✓ Masked output is finite")
print("✓ Temporary hooks removed")
print("✓ Zero hooks remain")
print("✓ Original output restored exactly")
print("✓ No model weights changed")
print("=" * 70)

Existing hooks cleared.
Mask activation points: 17
Original logits shape: (32, 2)
Temporary hooks registered: 17
Masked logits shape: (32, 2)

Layer validation
----------------------------------------------------------------------------------------------------
relu         | channels=  64 | protected=  64 | free=   0 | max_free_abs=0.000e+00
layer1.0     | channels= 256 | protected= 247 | free=   9 | max_free_abs=0.000e+00
layer1.1     | channels= 256 | protected= 238 | free=  18 | max_free_abs=0.000e+00
layer1.2     | channels= 256 | protected= 237 | free=  19 | max_free_abs=0.000e+00
layer2.0     | channels= 512 | protected= 471 | free=  41 | max_free_abs=0.000e+00
layer2.1     | channels= 512 | protected= 414 | free=  98 | max_free_abs=0.000e+00
layer2.2     | channels= 512 | protected= 352 | free= 160 | max_free_abs=0.000e+00
layer2.3     | channels= 512 | protected= 317 | free= 195 | max_free_abs=0.000e+00
layer3.0     | channels=1024 | protected= 729 | free= 295 | max_free_abs=0.

In [48]:
# ============================================================
# STEP 22A — TASK 1 PRUNED MODEL BASELINE EVALUATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Make sure we are using the CORRECT Task-1 GCE checkpoint
# ------------------------------------------------------------

TASK1_GCE_BEST = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_gce_best.pt"
)

assert os.path.exists(TASK1_GCE_BEST), "Task-1 GCE checkpoint not found."

checkpoint = torch.load(
    TASK1_GCE_BEST,
    map_location=device
)

model_task1.load_state_dict(checkpoint["model_state_dict"])
model_task1.eval()

print("Loaded Task-1 GCE checkpoint")
print("Epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["best_val_accuracy"])


# ------------------------------------------------------------
# 2. Make sure no stale hooks exist
# ------------------------------------------------------------

for module in model_task1.modules():
    if hasattr(module, "_forward_hooks"):
        for hook_id in list(module._forward_hooks.keys()):
            module._forward_hooks[hook_id].remove()

print("Remaining hooks before evaluation:",
      sum(len(m._forward_hooks) for m in model_task1.modules()))


# ------------------------------------------------------------
# 3. Activation masks
#
# protected_masks_correct was created in Step 20.
# Each mask = 1 for protected channels
#             0 for pruned/free channels
# ------------------------------------------------------------

assert "protected_masks_correct" in globals(), \
    "protected_masks_correct is missing. Re-run Step 20."

mask_dict = {
    name: mask.to(device)
    for name, mask in protected_masks_correct.items()
}

print("\nMask summary:")
total_channels = 0
protected_channels = 0

for name, mask in mask_dict.items():
    p = int(mask.sum().item())
    n = int(mask.numel())
    protected_channels += p
    total_channels += n
    print(
        f"{name:12s} total={n:5d} "
        f"protected={p:5d} free/pruned={n-p:5d}"
    )

print("\nTotal activation units:", total_channels)
print("Protected:", protected_channels)
print("Free/pruned:", total_channels - protected_channels)


# ------------------------------------------------------------
# 4. Correct activation points
#
# IMPORTANT:
# For torchvision ResNet-50, the Bottleneck's self.relu
# is reused internally.
#
# Therefore we mask:
#   model.relu             -> stem output
#   layer1.0 ... layer4.2  -> Bottleneck output
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task1.relu,
}

for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
    layer = getattr(model_task1, layer_name)

    for i, block in enumerate(layer):
        activation_modules[f"{layer_name}.{i}"] = block


# ------------------------------------------------------------
# 5. Register temporary functional-pruning hooks
# ------------------------------------------------------------

handles = []

for name, module in activation_modules.items():

    mask = mask_dict[name].view(1, -1, 1, 1)

    def make_hook(mask_tensor):
        def hook(module, inputs, output):
            return output * mask_tensor
        return hook

    handles.append(
        module.register_forward_hook(
            make_hook(mask)
        )
    )

print("\nTemporary pruning hooks:", len(handles))


# ------------------------------------------------------------
# 6. Evaluation helper
# ------------------------------------------------------------

def evaluate_pruned_model(model, loader):

    model.eval()

    y_true = []
    y_pred = []
    age_groups = []
    image_ids = []

    with torch.no_grad():

        for batch in loader:

            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            logits = model(images)
            preds = logits.argmax(dim=1)

            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            age_groups.extend(
                batch["age_group"].cpu().numpy().tolist()
            )

            image_ids.extend(
                batch["image_id"]
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    age_groups = np.asarray(age_groups)

    # --------------------------------------------------------
    # Overall metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(y_true, y_pred)

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(TASK1_CLASSES)))
    )

    # --------------------------------------------------------
    # Age-group accuracy
    #
    # age_group:
    #   0 = age < 60
    #   1 = age >= 60
    #   -1 = missing age
    # --------------------------------------------------------

    group_results = {}

    for group_id, group_name in [
        (0, "age_<60"),
        (1, "age_>=60")
    ]:

        mask = age_groups == group_id

        if mask.sum() > 0:
            group_acc = accuracy_score(
                y_true[mask],
                y_pred[mask]
            )
        else:
            group_acc = np.nan

        group_results[group_name] = {
            "n": int(mask.sum()),
            "accuracy": float(group_acc)
            if not np.isnan(group_acc)
            else None
        }

    # --------------------------------------------------------
    # Fairness metrics
    #
    # Positive class = TASK1_CLASSES[1] = "mel"
    #
    # DPR:
    #   P(pred=positive | age>=60)
    #   --------------------------------
    #   P(pred=positive | age<60)
    #
    # EOD:
    #   TPR(age>=60) - TPR(age<60)
    #
    # We also report absolute EOD because the paper discusses
    # lower absolute EOD as fairer.
    # --------------------------------------------------------

    positive_class = 1

    positive_rates = {}
    tprs = {}

    for group_id, group_name in [
        (0, "age_<60"),
        (1, "age_>=60")
    ]:

        group_mask = age_groups == group_id

        if group_mask.sum() == 0:
            positive_rates[group_name] = np.nan
            tprs[group_name] = np.nan
            continue

        positive_rates[group_name] = np.mean(
            y_pred[group_mask] == positive_class
        )

        true_positive_mask = (
            group_mask &
            (y_true == positive_class)
        )

        if true_positive_mask.sum() > 0:
            tprs[group_name] = np.mean(
                y_pred[true_positive_mask] == positive_class
            )
        else:
            tprs[group_name] = np.nan

    p_under60 = positive_rates["age_<60"]
    p_over60 = positive_rates["age_>=60"]

    if p_under60 > 0:
        dpr = p_over60 / p_under60
    else:
        dpr = np.nan

    eod = (
        tprs["age_>=60"] -
        tprs["age_<60"]
    )

    results = {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_acc),
        "macro_f1": float(macro_f1),

        "age_<60_accuracy":
            group_results["age_<60"]["accuracy"],

        "age_>=60_accuracy":
            group_results["age_>=60"]["accuracy"],

        "age_<60_positive_rate":
            float(p_under60),

        "age_>=60_positive_rate":
            float(p_over60),

        "DPR":
            float(dpr),

        "EOD":
            float(eod),

        "absolute_EOD":
            float(abs(eod)),

        "n_samples": int(len(y_true)),
        "confusion_matrix": cm.tolist()
    }

    return results, y_true, y_pred, age_groups


# ------------------------------------------------------------
# 7. Evaluate Task-1 TEST set
# ------------------------------------------------------------

task1_pruned_results, y_true_t1, y_pred_t1, age_t1 = \
    evaluate_pruned_model(
        model_task1,
        task1_test_loader
    )


# ------------------------------------------------------------
# 8. Remove temporary hooks immediately
# ------------------------------------------------------------

for handle in handles:
    handle.remove()

print(
    "\nRemaining hooks after evaluation:",
    sum(
        len(m._forward_hooks)
        for m in model_task1.modules()
    )
)


# ------------------------------------------------------------
# 9. Print results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TASK 1 — POST-PRUNING BASELINE")
print("=" * 65)

print(
    f"Accuracy           : "
    f"{task1_pruned_results['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{task1_pruned_results['balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1           : "
    f"{task1_pruned_results['macro_f1']:.4f}"
)

print(
    f"Age <60 Accuracy   : "
    f"{task1_pruned_results['age_<60_accuracy']:.4f}"
)

print(
    f"Age >=60 Accuracy  : "
    f"{task1_pruned_results['age_>=60_accuracy']:.4f}"
)

print(
    f"DPR                : "
    f"{task1_pruned_results['DPR']:.4f}"
)

print(
    f"EOD                : "
    f"{task1_pruned_results['EOD']:.4f}"
)

print(
    f"|EOD|              : "
    f"{task1_pruned_results['absolute_EOD']:.4f}"
)

print("\nConfusion Matrix:")
print(
    np.array(
        task1_pruned_results["confusion_matrix"]
    )
)


# ------------------------------------------------------------
# 10. Save diagnostic results
# ------------------------------------------------------------

task1_diag_path = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_post_pruning_metrics.json"
)

with open(task1_diag_path, "w") as f:
    json.dump(
        task1_pruned_results,
        f,
        indent=2
    )

print("\nSaved:")
print(task1_diag_path)

Loaded Task-1 GCE checkpoint
Epoch: 4
Best validation accuracy: 0.7936507936507936
Remaining hooks before evaluation: 0


AttributeError: 'numpy.ndarray' object has no attribute 'to'

In [52]:
# ============================================================
# STEP 22A — TASK 1 POST-PRUNING BASELINE
# CORRECTED AGE-GROUP VERSION
# ============================================================

import os
import json
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 65)
print("STEP 22A — TASK 1 POST-PRUNING BASELINE")
print("=" * 65)


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. LOAD TASK-1 GCE CHECKPOINT
# ============================================================

TASK1_GCE_BEST = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_gce_best.pt"
)

checkpoint = torch.load(
    TASK1_GCE_BEST,
    map_location=device
)

model_task1.load_state_dict(
    checkpoint["model_state_dict"]
)

model_task1 = model_task1.to(device)
model_task1.eval()

print("\nTask-1 GCE checkpoint loaded.")
print("Epoch:", checkpoint["epoch"])
print(
    "Best validation accuracy:",
    checkpoint["best_val_accuracy"]
)


# ============================================================
# 3. MASK
# ============================================================

assert "protected_masks_correct" in globals(), (
    "protected_masks_correct is missing. "
    "Please rerun Step 20."
)

mask_dict = {}

for name, mask in protected_masks_correct.items():

    mask_tensor = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()

    mask_dict[name] = mask_tensor


# ============================================================
# 4. EXPECTED ACTIVATION CHANNELS
# ============================================================

expected_activation_channels = {

    "relu": 64,

    "layer1.0": 256,
    "layer1.1": 256,
    "layer1.2": 256,

    "layer2.0": 512,
    "layer2.1": 512,
    "layer2.2": 512,
    "layer2.3": 512,

    "layer3.0": 1024,
    "layer3.1": 1024,
    "layer3.2": 1024,
    "layer3.3": 1024,
    "layer3.4": 1024,
    "layer3.5": 1024,

    "layer4.0": 2048,
    "layer4.1": 2048,
    "layer4.2": 2048,
}


# ============================================================
# 5. VERIFY MASK
# ============================================================

for name, expected in expected_activation_channels.items():

    assert name in mask_dict

    assert mask_dict[name].numel() == expected, (
        f"{name}: expected {expected}, "
        f"got {mask_dict[name].numel()}"
    )


total_units = sum(
    mask.numel()
    for mask in mask_dict.values()
)

protected_units = sum(
    int(mask.sum().item())
    for mask in mask_dict.values()
)

pruned_units = (
    total_units -
    protected_units
)

print("\nMask verified:")
print("Total units     :", total_units)
print("Protected units :", protected_units)
print("Pruned units    :", pruned_units)

assert total_units == 15168
assert protected_units == 6067
assert pruned_units == 9101


# ============================================================
# 6. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task1.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task1,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


assert len(activation_modules) == 17


# ============================================================
# 7. MASK HOOK
# ============================================================

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        mask_4d = mask_tensor.view(
            1,
            -1,
            1,
            1
        )

        return output * mask_4d

    return hook


for name, module in activation_modules.items():

    handle = module.register_forward_hook(
        make_mask_hook(
            mask_dict[name]
        )
    )

    handles.append(handle)

print("\nTemporary pruning hooks:", len(handles))


# ============================================================
# 8. SAFE AGE-GROUP CONVERSION
# ============================================================

def normalize_age_group(value):

    """
    Convert dataset age-group representation into:

        0 = age < 60
        1 = age >= 60
       -1 = missing/unknown
    """

    # --------------------------------------------------------
    # Python string
    # --------------------------------------------------------

    if isinstance(value, str):

        value_clean = value.strip().lower()

        if value_clean in [
            "age<60",
            "age <60",
            "age< 60",
            "age < 60"
        ]:
            return 0

        if value_clean in [
            "age>=60",
            "age >=60",
            "age>= 60",
            "age >= 60"
        ]:
            return 1

        return -1


    # --------------------------------------------------------
    # Numeric representation
    # --------------------------------------------------------

    if isinstance(
        value,
        (int, np.integer)
    ):

        if value == 0:
            return 0

        if value == 1:
            return 1

        return -1


    # --------------------------------------------------------
    # Float representation
    # --------------------------------------------------------

    if isinstance(
        value,
        (float, np.floating)
    ):

        if value == 0:
            return 0

        if value == 1:
            return 1

        return -1


    return -1


# ============================================================
# 9. EVALUATION
# ============================================================

def evaluate_pruned_model(
    model,
    loader
):

    model.eval()

    y_true = []
    y_pred = []
    age_groups = []
    image_ids = []

    with torch.no_grad():

        for batch in loader:

            # ------------------------------------------------
            # Images
            # ------------------------------------------------

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # Labels
            # ------------------------------------------------

            labels = batch["label"]

            if torch.is_tensor(labels):

                labels_cpu = (
                    labels
                    .detach()
                    .cpu()
                )

                labels_device = labels.to(
                    device,
                    non_blocking=True
                )

            else:

                labels_cpu = torch.tensor(
                    labels,
                    dtype=torch.long
                )

                labels_device = labels_cpu.to(
                    device
                )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits = model(images)

            predictions = logits.argmax(
                dim=1
            )


            # ------------------------------------------------
            # Store predictions
            # ------------------------------------------------

            y_true.extend(
                labels_cpu
                .numpy()
                .tolist()
            )

            y_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            # ------------------------------------------------
            # AGE GROUP
            # ------------------------------------------------

            batch_age = batch["age_group"]

            # DataLoader returns a list of strings in your case.
            #
            # Example:
            # ['age<60', 'age>=60', ...]

            if torch.is_tensor(batch_age):

                batch_age = (
                    batch_age
                    .detach()
                    .cpu()
                    .numpy()
                    .tolist()
                )

            elif isinstance(
                batch_age,
                np.ndarray
            ):

                batch_age = (
                    batch_age
                    .tolist()
                )

            elif not isinstance(
                batch_age,
                (list, tuple)
            ):

                batch_age = [batch_age]


            for age_value in batch_age:

                age_groups.append(
                    normalize_age_group(
                        age_value
                    )
                )


            # ------------------------------------------------
            # IMAGE IDs
            # ------------------------------------------------

            batch_ids = batch["image_id"]

            if isinstance(
                batch_ids,
                (list, tuple)
            ):

                image_ids.extend(
                    list(batch_ids)
                )

            elif torch.is_tensor(
                batch_ids
            ):

                image_ids.extend(
                    batch_ids
                    .detach()
                    .cpu()
                    .numpy()
                    .tolist()
                )

            else:

                image_ids.append(
                    batch_ids
                )


    # ========================================================
    # ARRAYS
    # ========================================================

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64
    )

    age_groups = np.asarray(
        age_groups,
        dtype=np.int64
    )


    # ========================================================
    # SAFETY CHECK
    # ========================================================

    assert len(y_true) == len(y_pred)
    assert len(y_true) == len(age_groups)

    print(
        "\nEvaluation samples:",
        len(y_true)
    )

    print(
        "Age <60 samples:",
        int(np.sum(age_groups == 0))
    )

    print(
        "Age >=60 samples:",
        int(np.sum(age_groups == 1))
    )

    print(
        "Missing/unknown age:",
        int(np.sum(age_groups == -1))
    )


    # ========================================================
    # OVERALL METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )


    # ========================================================
    # GROUP ACCURACY
    # ========================================================

    group_accuracy = {}

    for group_id, group_name in [
        (0, "age_<60"),
        (1, "age_>=60")
    ]:

        mask = (
            age_groups == group_id
        )

        if mask.sum() > 0:

            group_accuracy[group_name] = (
                float(
                    accuracy_score(
                        y_true[mask],
                        y_pred[mask]
                    )
                )
            )

        else:

            group_accuracy[group_name] = np.nan


    # ========================================================
    # FAIRNESS METRICS
    # ========================================================
    #
    # Task 1:
    #
    #   class 0 = bkl
    #   class 1 = mel
    #
    # Positive class = mel
    # ========================================================

    positive_class = 1

    positive_rates = {}
    tprs = {}


    for group_id, group_name in [
        (0, "age_<60"),
        (1, "age_>=60")
    ]:

        group_mask = (
            age_groups == group_id
        )

        # ----------------------------------------------------
        # Positive prediction rate
        # ----------------------------------------------------

        if group_mask.sum() > 0:

            positive_rates[group_name] = float(
                np.mean(
                    y_pred[group_mask] ==
                    positive_class
                )
            )

        else:

            positive_rates[group_name] = np.nan


        # ----------------------------------------------------
        # TPR
        # ----------------------------------------------------

        true_positive_mask = (
            group_mask &
            (y_true == positive_class)
        )

        if true_positive_mask.sum() > 0:

            tprs[group_name] = float(
                np.mean(
                    y_pred[true_positive_mask] ==
                    positive_class
                )
            )

        else:

            tprs[group_name] = np.nan


    # ========================================================
    # DPR
    # ========================================================

    p_under60 = positive_rates[
        "age_<60"
    ]

    p_over60 = positive_rates[
        "age_>=60"
    ]

    if (
        not np.isnan(p_under60)
        and p_under60 > 0
    ):

        dpr = (
            p_over60 /
            p_under60
        )

    else:

        dpr = np.nan


    # ========================================================
    # EOD
    # ========================================================

    tpr_under60 = tprs[
        "age_<60"
    ]

    tpr_over60 = tprs[
        "age_>=60"
    ]

    if (
        not np.isnan(tpr_under60)
        and not np.isnan(tpr_over60)
    ):

        eod = (
            tpr_over60 -
            tpr_under60
        )

    else:

        eod = np.nan


    # ========================================================
    # RESULT
    # ========================================================

    results = {

        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "age_<60_accuracy":
            group_accuracy[
                "age_<60"
            ],

        "age_>=60_accuracy":
            group_accuracy[
                "age_>=60"
            ],

        "age_<60_positive_rate":
            float(p_under60),

        "age_>=60_positive_rate":
            float(p_over60),

        "DPR":
            float(dpr),

        "EOD":
            float(eod),

        "absolute_EOD":
            float(abs(eod)),

        "n_samples":
            int(len(y_true)),

        "confusion_matrix":
            confusion.tolist()
    }


    return (
        results,
        y_true,
        y_pred,
        age_groups
    )


# ============================================================
# 10. RUN EVALUATION
# ============================================================

try:

    print(
        "\n" +
        "=" * 65
    )

    print(
        "Evaluating Task-1 test set..."
    )

    print(
        "=" * 65
    )

    (
        task1_pruned_results,
        y_true_t1,
        y_pred_t1,
        age_t1
    ) = evaluate_pruned_model(
        model_task1,
        task1_test_loader
    )

finally:

    # CRITICAL:
    # remove hooks using their actual handles.
    for handle in handles:
        handle.remove()

    print(
        "\nTemporary pruning hooks removed."
    )


# ============================================================
# 11. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("TASK 1 — POST-PRUNING BASELINE")
print("=" * 65)

print(
    f"Samples            : "
    f"{task1_pruned_results['n_samples']}"
)

print(
    f"Accuracy           : "
    f"{task1_pruned_results['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{task1_pruned_results['balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1           : "
    f"{task1_pruned_results['macro_f1']:.4f}"
)

print(
    f"Age <60 Accuracy   : "
    f"{task1_pruned_results['age_<60_accuracy']:.4f}"
)

print(
    f"Age >=60 Accuracy  : "
    f"{task1_pruned_results['age_>=60_accuracy']:.4f}"
)

print(
    f"Age <60 Pos Rate   : "
    f"{task1_pruned_results['age_<60_positive_rate']:.4f}"
)

print(
    f"Age >=60 Pos Rate  : "
    f"{task1_pruned_results['age_>=60_positive_rate']:.4f}"
)

print(
    f"DPR                : "
    f"{task1_pruned_results['DPR']:.4f}"
)

print(
    f"EOD                : "
    f"{task1_pruned_results['EOD']:.4f}"
)

print(
    f"|EOD|              : "
    f"{task1_pruned_results['absolute_EOD']:.4f}"
)


# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix:")

print(
    np.array(
        task1_pruned_results[
            "confusion_matrix"
        ]
    )
)

print(
    "\nClass order:",
    TASK1_CLASSES
)


# ============================================================
# 13. SAVE METRICS
# ============================================================

TASK1_METRICS_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_post_pruning_metrics.json"
)

with open(
    TASK1_METRICS_PATH,
    "w"
) as f:

    json.dump(
        task1_pruned_results,
        f,
        indent=2
    )

print(
    "\nSaved metrics:"
)

print(
    TASK1_METRICS_PATH
)


# ============================================================
# 14. SAVE SUMMARY
# ============================================================

TASK1_SUMMARY_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_post_pruning_summary.txt"
)

with open(
    TASK1_SUMMARY_PATH,
    "w"
) as f:

    f.write(
        "BiasPruner Task-1 Post-Pruning Baseline\n"
    )

    f.write(
        "======================================\n\n"
    )

    f.write(
        f"GCE checkpoint: "
        f"{TASK1_GCE_BEST}\n"
    )

    f.write(
        f"GCE epoch: "
        f"{checkpoint['epoch']}\n"
    )

    f.write(
        "Pruning ratio: 0.60\n"
    )

    f.write(
        f"Total units: "
        f"{total_units}\n"
    )

    f.write(
        f"Protected units: "
        f"{protected_units}\n"
    )

    f.write(
        f"Pruned units: "
        f"{pruned_units}\n\n"
    )

    for key, value in (
        task1_pruned_results.items()
    ):

        f.write(
            f"{key}: {value}\n"
        )


print(
    "Saved summary:"
)

print(
    TASK1_SUMMARY_PATH
)


# ============================================================
# 15. DONE
# ============================================================

print("\n" + "=" * 65)
print("STEP 22A COMPLETE")
print("=" * 65)

print(
    "No training was performed."
)

print(
    "Pruning was applied only for evaluation."
)

print(
    "Task-1 GCE checkpoint remains safe."
)

print(
    "Metrics saved successfully."
)

print("=" * 65)

STEP 22A — TASK 1 POST-PRUNING BASELINE
Device: cuda
GPU: Tesla T4

Task-1 GCE checkpoint loaded.
Epoch: 4
Best validation accuracy: 0.7936507936507936

Mask verified:
Total units     : 15168
Protected units : 6067
Pruned units    : 9101

Temporary pruning hooks: 17

Evaluating Task-1 test set...

Evaluation samples: 226
Age <60 samples: 85
Age >=60 samples: 140
Missing/unknown age: 1

Temporary pruning hooks removed.


TASK 1 — POST-PRUNING BASELINE
Samples            : 226
Accuracy           : 0.4956
Balanced Accuracy  : 0.5000
Macro-F1           : 0.3314
Age <60 Accuracy   : 0.4824
Age >=60 Accuracy  : 0.5000
Age <60 Pos Rate   : 0.0000
Age >=60 Pos Rate  : 0.0000
DPR                : nan
EOD                : 0.0000
|EOD|              : 0.0000

Confusion Matrix:
[[112   0]
 [114   0]]

Class order: ['bkl', 'mel']

Saved metrics:
/kaggle/working/biaspruner_checkpoints/task1/task1_post_pruning_metrics.json
Saved summary:
/kaggle/working/biaspruner_checkpoints/task1/task1_post_pruning_

In [53]:
# ============================================================
# STEP 22B — TASK 1 WCE FINE-TUNING AFTER PRUNING
# ============================================================
#
# Paper WCE:
#
#   L_WCE(x) = W(x) * L_CE(x)
#
#   W(x) = exp(alpha * L_GCE(x))
#
#   L_GCE(x) = (1 - p_y^q) / q
#
# q = 0.7
#
# Pruning mask remains ACTIVE during all forward passes.
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 22B — TASK 1 WCE FINE-TUNING")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

device = torch.device("cuda")

GCE_Q = 0.7
ALPHA = 1.0
LEARNING_RATE = 1e-4
WCE_EPOCHS = 20
PRUNING_RATIO = 0.60

TASK1_WCE_DIR = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1"
)

os.makedirs(
    TASK1_WCE_DIR,
    exist_ok=True
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("GCE q:", GCE_Q)
print("WCE alpha:", ALPHA)
print("Learning rate:", LEARNING_RATE)
print("WCE epochs:", WCE_EPOCHS)
print("Pruning ratio:", PRUNING_RATIO)


# ============================================================
# 2. RELOAD CLEAN TASK-1 GCE MODEL
# ============================================================
#
# This guarantees that we start WCE from the original
# Task-1 GCE checkpoint rather than from any accidentally
# modified model.
# ============================================================

TASK1_GCE_BEST = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_gce_best.pt"
)

assert os.path.exists(
    TASK1_GCE_BEST
), "Task-1 GCE checkpoint not found."

checkpoint = torch.load(
    TASK1_GCE_BEST,
    map_location=device
)

model_task1.load_state_dict(
    checkpoint["model_state_dict"]
)

model_task1 = model_task1.to(device)
model_task1.train()

print("\nLoaded Task-1 GCE checkpoint.")
print("Starting from GCE epoch:", checkpoint["epoch"])
print(
    "Best validation accuracy:",
    checkpoint["best_val_accuracy"]
)


# ============================================================
# 3. LOAD CORRECT PRUNING MASK
# ============================================================

assert "protected_masks_correct" in globals(), (
    "protected_masks_correct is missing."
)

mask_dict = {}

for name, mask in protected_masks_correct.items():

    mask_dict[name] = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()


# ============================================================
# 4. EXPECTED ACTIVATION POINTS
# ============================================================

expected_activation_channels = {

    "relu": 64,

    "layer1.0": 256,
    "layer1.1": 256,
    "layer1.2": 256,

    "layer2.0": 512,
    "layer2.1": 512,
    "layer2.2": 512,
    "layer2.3": 512,

    "layer3.0": 1024,
    "layer3.1": 1024,
    "layer3.2": 1024,
    "layer3.3": 1024,
    "layer3.4": 1024,
    "layer3.5": 1024,

    "layer4.0": 2048,
    "layer4.1": 2048,
    "layer4.2": 2048,
}


for name, expected in (
    expected_activation_channels.items()
):

    assert name in mask_dict

    assert mask_dict[name].numel() == expected, (
        f"Mask mismatch for {name}"
    )


# ============================================================
# 5. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task1.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task1,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


assert len(activation_modules) == 17


# ============================================================
# 6. REGISTER PERMANENT WCE MASK HOOKS
# ============================================================

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        mask_4d = mask_tensor.view(
            1,
            -1,
            1,
            1
        )

        return output * mask_4d

    return hook


for name, module in activation_modules.items():

    handle = module.register_forward_hook(
        make_mask_hook(
            mask_dict[name]
        )
    )

    handles.append(handle)


print(
    "\nPruning hooks active:",
    len(handles)
)

assert len(handles) == 17


# ============================================================
# 7. GCE FUNCTION
# ============================================================

def generalized_cross_entropy(
    logits,
    targets,
    q=0.7
):

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    p_y = probabilities[
        torch.arange(
            logits.size(0),
            device=logits.device
        ),
        targets
    ]

    # Numerical safety
    p_y = torch.clamp(
        p_y,
        min=1e-8,
        max=1.0
    )

    loss = (
        1.0 -
        torch.pow(p_y, q)
    ) / q

    return loss


# ============================================================
# 8. WCE FUNCTION
# ============================================================

def weighted_cross_entropy(
    logits,
    targets,
    q=0.7,
    alpha=1.0
):

    # --------------------------------------------------------
    # Standard CE
    # --------------------------------------------------------

    ce_loss = nn.functional.cross_entropy(
        logits,
        targets,
        reduction="none"
    )


    # --------------------------------------------------------
    # GCE
    # --------------------------------------------------------

    gce_loss = generalized_cross_entropy(
        logits,
        targets,
        q=q
    )


    # --------------------------------------------------------
    # Paper's weighting:
    #
    # W(x) = exp(alpha * L_GCE(x))
    # --------------------------------------------------------

    weights = torch.exp(
        alpha * gce_loss
    )


    # --------------------------------------------------------
    # WCE
    # --------------------------------------------------------

    wce_loss = (
        weights *
        ce_loss
    )

    return wce_loss.mean(), {
        "ce": ce_loss.mean().detach(),
        "gce": gce_loss.mean().detach(),
        "weight": weights.mean().detach()
    }


# ============================================================
# 9. OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model_task1.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 10. TRAINING HISTORY
# ============================================================

history = []

best_val_accuracy = -float("inf")
best_epoch = None

BEST_WCE_CHECKPOINT = (
    TASK1_WCE_DIR +
    "/task1_wce_pruned_best.pt"
)

FINAL_WCE_CHECKPOINT = (
    TASK1_WCE_DIR +
    "/task1_wce_pruned_final.pt"
)


# ============================================================
# 11. VALIDATION FUNCTION
# ============================================================

def evaluate_basic(
    model,
    loader
):

    model.eval()

    all_targets = []
    all_predictions = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch in loader:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            targets = batch["label"]

            if not torch.is_tensor(targets):

                targets = torch.tensor(
                    targets,
                    dtype=torch.long
                )

            targets = targets.to(
                device,
                non_blocking=True
            )

            logits = model(images)

            loss = nn.functional.cross_entropy(
                logits,
                targets
            )

            predictions = logits.argmax(
                dim=1
            )

            batch_size = targets.size(0)

            total_loss += (
                loss.item() *
                batch_size
            )

            total_samples += batch_size

            all_targets.extend(
                targets.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions.detach()
                .cpu()
                .numpy()
                .tolist()
            )


    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "loss":
            total_loss / total_samples,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1
    }


# ============================================================
# 12. WCE TRAINING LOOP
# ============================================================

print("\n")
print("=" * 70)
print("STARTING WCE FINE-TUNING")
print("=" * 70)


try:

    for epoch in range(
        1,
        WCE_EPOCHS + 1
    ):

        model_task1.train()

        running_loss = 0.0
        running_ce = 0.0
        running_gce = 0.0
        running_weight = 0.0

        correct = 0
        total = 0


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        for batch in task1_train_loader:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            targets = batch["label"]

            if not torch.is_tensor(targets):

                targets = torch.tensor(
                    targets,
                    dtype=torch.long
                )

            targets = targets.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # IMPORTANT:
            # pruning mask is active here

            logits = model_task1(
                images
            )


            loss, components = (
                weighted_cross_entropy(
                    logits,
                    targets,
                    q=GCE_Q,
                    alpha=ALPHA
                )
            )


            loss.backward()

            optimizer.step()


            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            batch_size = targets.size(0)

            running_loss += (
                loss.item() *
                batch_size
            )

            running_ce += (
                components["ce"].item() *
                batch_size
            )

            running_gce += (
                components["gce"].item() *
                batch_size
            )

            running_weight += (
                components["weight"].item() *
                batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            correct += int(
                (
                    predictions ==
                    targets
                ).sum().item()
            )

            total += batch_size


        train_loss = (
            running_loss /
            total
        )

        train_ce = (
            running_ce /
            total
        )

        train_gce = (
            running_gce /
            total
        )

        train_weight = (
            running_weight /
            total
        )

        train_accuracy = (
            correct /
            total
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        val_results = evaluate_basic(
            model_task1,
            task1_val_loader
        )


        # ----------------------------------------------------
        # SAVE HISTORY
        # ----------------------------------------------------

        epoch_result = {

            "epoch": epoch,

            "train_wce":
                train_loss,

            "train_ce":
                train_ce,

            "train_gce":
                train_gce,

            "mean_wce_weight":
                train_weight,

            "train_accuracy":
                train_accuracy,

            "val_loss":
                val_results["loss"],

            "val_accuracy":
                val_results["accuracy"],

            "val_balanced_accuracy":
                val_results[
                    "balanced_accuracy"
                ],

            "val_macro_f1":
                val_results[
                    "macro_f1"
                ]
        }

        history.append(
            epoch_result
        )


        # ----------------------------------------------------
        # BEST CHECKPOINT
        # ----------------------------------------------------

        if (
            val_results["accuracy"]
            >
            best_val_accuracy
        ):

            best_val_accuracy = (
                val_results["accuracy"]
            )

            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,

                    "model_state_dict":
                        model_task1.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "classes":
                        TASK1_CLASSES,

                    "gce_q":
                        GCE_Q,

                    "alpha":
                        ALPHA,

                    "learning_rate":
                        LEARNING_RATE,

                    "pruning_ratio":
                        PRUNING_RATIO,

                    "total_units":
                        total_units,

                    "protected_units":
                        protected_units,

                    "pruned_units":
                        pruned_units,

                    "best_val_accuracy":
                        best_val_accuracy,

                    "history":
                        history
                },
                BEST_WCE_CHECKPOINT
            )


        # ----------------------------------------------------
        # PRINT
        # ----------------------------------------------------

        print(
            f"Epoch {epoch:02d}/{WCE_EPOCHS} | "
            f"WCE {train_loss:.4f} | "
            f"CE {train_ce:.4f} | "
            f"GCE {train_gce:.4f} | "
            f"W {train_weight:.4f} | "
            f"Train Acc {train_accuracy:.4f} | "
            f"Val Acc {val_results['accuracy']:.4f} | "
            f"Val BA {val_results['balanced_accuracy']:.4f} | "
            f"Val F1 {val_results['macro_f1']:.4f}"
        )


finally:

    # --------------------------------------------------------
    # ALWAYS remove hooks
    # --------------------------------------------------------

    for handle in handles:

        handle.remove()

    print(
        "\nWCE pruning hooks removed."
    )


# ============================================================
# 13. SAVE FINAL CHECKPOINT
# ============================================================

torch.save(
    {
        "epoch": WCE_EPOCHS,

        "model_state_dict":
            model_task1.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "classes":
            TASK1_CLASSES,

        "gce_q":
            GCE_Q,

        "alpha":
            ALPHA,

        "learning_rate":
            LEARNING_RATE,

        "pruning_ratio":
            PRUNING_RATIO,

        "total_units":
            total_units,

        "protected_units":
            protected_units,

        "pruned_units":
            pruned_units,

        "best_val_accuracy":
            best_val_accuracy,

        "best_epoch":
            best_epoch,

        "history":
            history
    },
    FINAL_WCE_CHECKPOINT
)


# ============================================================
# 14. SAVE HISTORY
# ============================================================

HISTORY_PATH = (
    TASK1_WCE_DIR +
    "/task1_wce_history.json"
)

with open(
    HISTORY_PATH,
    "w"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


# ============================================================
# 15. SAVE CONFIG
# ============================================================

CONFIG_PATH = (
    TASK1_WCE_DIR +
    "/task1_wce_config.txt"
)

with open(
    CONFIG_PATH,
    "w"
) as f:

    f.write(
        "Task 1 WCE Fine-Tuning Configuration\n"
    )

    f.write(
        "=====================================\n\n"
    )

    f.write(
        f"GCE q = {GCE_Q}\n"
    )

    f.write(
        f"Alpha = {ALPHA}\n"
    )

    f.write(
        f"Learning rate = {LEARNING_RATE}\n"
    )

    f.write(
        f"Epochs = {WCE_EPOCHS}\n"
    )

    f.write(
        f"Pruning ratio = {PRUNING_RATIO}\n"
    )

    f.write(
        f"Total units = {total_units}\n"
    )

    f.write(
        f"Protected units = {protected_units}\n"
    )

    f.write(
        f"Pruned units = {pruned_units}\n"
    )

    f.write(
        f"Best validation accuracy = "
        f"{best_val_accuracy}\n"
    )

    f.write(
        f"Best epoch = {best_epoch}\n"
    )


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("STEP 22B COMPLETE")
print("=" * 70)

print(
    "Best validation accuracy:",
    best_val_accuracy
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "\nBest checkpoint:"
)

print(
    BEST_WCE_CHECKPOINT
)

print(
    "\nFinal checkpoint:"
)

print(
    FINAL_WCE_CHECKPOINT
)

print(
    "\nHistory:"
)

print(
    HISTORY_PATH
)

print(
    "\nConfiguration:"
)

print(
    CONFIG_PATH
)

print("=" * 70)

STEP 22B — TASK 1 WCE FINE-TUNING
Device: cuda
GPU: Tesla T4
GCE q: 0.7
WCE alpha: 1.0
Learning rate: 0.0001
WCE epochs: 20
Pruning ratio: 0.6

Loaded Task-1 GCE checkpoint.
Starting from GCE epoch: 4
Best validation accuracy: 0.7936507936507936

Pruning hooks active: 17


STARTING WCE FINE-TUNING
Epoch 01/20 | WCE 2.5473 | CE 0.9482 | GCE 0.5612 | W 1.9073 | Train Acc 0.5320 | Val Acc 0.6122 | Val BA 0.6195 | Val F1 0.5801
Epoch 02/20 | WCE 1.0363 | CE 0.5954 | GCE 0.4725 | W 1.6250 | Train Acc 0.6997 | Val Acc 0.7506 | Val BA 0.7478 | Val F1 0.7463
Epoch 03/20 | WCE 0.9524 | CE 0.5480 | GCE 0.4359 | W 1.5736 | Train Acc 0.7392 | Val Acc 0.7211 | Val BA 0.7226 | Val F1 0.7206
Epoch 04/20 | WCE 0.9100 | CE 0.5210 | GCE 0.4137 | W 1.5443 | Train Acc 0.7515 | Val Acc 0.7619 | Val BA 0.7606 | Val F1 0.7606
Epoch 05/20 | WCE 0.9287 | CE 0.5244 | GCE 0.4137 | W 1.5473 | Train Acc 0.7540 | Val Acc 0.6893 | Val BA 0.6914 | Val F1 0.6881
Epoch 06/20 | WCE 0.8983 | CE 0.5177 | GCE 0.4126 | W 1.

In [54]:
# ============================================================
# STEP 23 — TASK 1 FINAL WCE TEST EVALUATION
# ============================================================

import os
import json
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 23 — TASK 1 FINAL WCE TEST EVALUATION")
print("=" * 70)

device = torch.device("cuda")


# ============================================================
# 1. LOAD BEST WCE CHECKPOINT
# ============================================================

BEST_WCE_CHECKPOINT = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_wce_pruned_best.pt"
)

assert os.path.exists(
    BEST_WCE_CHECKPOINT
), "Best WCE checkpoint not found."

checkpoint_wce = torch.load(
    BEST_WCE_CHECKPOINT,
    map_location=device
)

model_task1.load_state_dict(
    checkpoint_wce["model_state_dict"]
)

model_task1 = model_task1.to(device)
model_task1.eval()

print("\nLoaded best WCE checkpoint.")
print("Checkpoint:", BEST_WCE_CHECKPOINT)
print("Best epoch:", checkpoint_wce["epoch"])
print(
    "Best validation accuracy:",
    checkpoint_wce["best_val_accuracy"]
)


# ============================================================
# 2. LOAD PRUNING MASK
# ============================================================

assert "protected_masks_correct" in globals()

mask_dict = {}

for name, mask in protected_masks_correct.items():

    mask_dict[name] = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()


# ============================================================
# 3. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task1.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task1,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


# ============================================================
# 4. REGISTER PRUNING MASK
# ============================================================

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        mask_4d = mask_tensor.view(
            1,
            -1,
            1,
            1
        )

        return output * mask_4d

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_mask_hook(
                mask_dict[name]
            )
        )
    )

print(
    "\nPruning hooks active:",
    len(handles)
)


# ============================================================
# 5. AGE-GROUP NORMALIZATION
# ============================================================

def normalize_age_group(value):

    if isinstance(value, str):

        value = value.strip().lower()

        if value in [
            "age<60",
            "age <60",
            "age< 60",
            "age < 60"
        ]:
            return 0

        if value in [
            "age>=60",
            "age >=60",
            "age>= 60",
            "age >= 60"
        ]:
            return 1

        return -1


    if isinstance(
        value,
        (int, np.integer)
    ):

        if value in [0, 1]:
            return int(value)

        return -1


    if isinstance(
        value,
        (float, np.floating)
    ):

        if value in [0.0, 1.0]:
            return int(value)

        return -1


    return -1


# ============================================================
# 6. EVALUATION
# ============================================================

y_true = []
y_pred = []
age_groups = []


try:

    with torch.no_grad():

        for batch in task1_test_loader:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            labels = batch["label"]

            if torch.is_tensor(labels):

                labels_cpu = (
                    labels
                    .detach()
                    .cpu()
                )

                labels_device = labels.to(
                    device,
                    non_blocking=True
                )

            else:

                labels_cpu = torch.tensor(
                    labels,
                    dtype=torch.long
                )

                labels_device = labels_cpu.to(
                    device
                )


            logits = model_task1(
                images
            )

            predictions = logits.argmax(
                dim=1
            )


            y_true.extend(
                labels_cpu
                .numpy()
                .tolist()
            )

            y_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            # ----------------------------------------------
            # Age groups
            # ----------------------------------------------

            batch_age = batch["age_group"]

            if torch.is_tensor(batch_age):

                batch_age = (
                    batch_age
                    .detach()
                    .cpu()
                    .numpy()
                    .tolist()
                )

            elif isinstance(
                batch_age,
                np.ndarray
            ):

                batch_age = (
                    batch_age.tolist()
                )

            elif not isinstance(
                batch_age,
                (list, tuple)
            ):

                batch_age = [batch_age]


            for value in batch_age:

                age_groups.append(
                    normalize_age_group(
                        value
                    )
                )


finally:

    for handle in handles:
        handle.remove()

    print(
        "\nPruning hooks removed."
    )


# ============================================================
# 7. ARRAYS
# ============================================================

y_true = np.asarray(
    y_true,
    dtype=np.int64
)

y_pred = np.asarray(
    y_pred,
    dtype=np.int64
)

age_groups = np.asarray(
    age_groups,
    dtype=np.int64
)

assert len(y_true) == len(y_pred)
assert len(y_true) == len(age_groups)


# ============================================================
# 8. OVERALL METRICS
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)


# ============================================================
# 9. AGE ACCURACY
# ============================================================

age_results = {}

for group_id, group_name in [
    (0, "age_<60"),
    (1, "age_>=60")
]:

    mask = age_groups == group_id

    if mask.sum() > 0:

        age_results[group_name] = float(
            accuracy_score(
                y_true[mask],
                y_pred[mask]
            )
        )

    else:

        age_results[group_name] = np.nan


# ============================================================
# 10. FAIRNESS
# ============================================================

positive_class = 1

positive_rates = {}
tprs = {}

for group_id, group_name in [
    (0, "age_<60"),
    (1, "age_>=60")
]:

    group_mask = (
        age_groups == group_id
    )

    if group_mask.sum() > 0:

        positive_rates[group_name] = float(
            np.mean(
                y_pred[group_mask]
                == positive_class
            )
        )

    else:

        positive_rates[group_name] = np.nan


    true_positive_mask = (
        group_mask &
        (y_true == positive_class)
    )

    if true_positive_mask.sum() > 0:

        tprs[group_name] = float(
            np.mean(
                y_pred[true_positive_mask]
                == positive_class
            )
        )

    else:

        tprs[group_name] = np.nan


p_under60 = positive_rates[
    "age_<60"
]

p_over60 = positive_rates[
    "age_>=60"
]

if (
    not np.isnan(p_under60)
    and p_under60 > 0
):

    dpr = (
        p_over60 /
        p_under60
    )

else:

    dpr = np.nan


tpr_under60 = tprs[
    "age_<60"
]

tpr_over60 = tprs[
    "age_>=60"
]

if (
    not np.isnan(tpr_under60)
    and not np.isnan(tpr_over60)
):

    eod = (
        tpr_over60 -
        tpr_under60
    )

else:

    eod = np.nan


# ============================================================
# 11. FINAL RESULTS
# ============================================================

task1_final_results = {

    "checkpoint":
        BEST_WCE_CHECKPOINT,

    "epoch":
        int(checkpoint_wce["epoch"]),

    "validation_accuracy":
        float(
            checkpoint_wce[
                "best_val_accuracy"
            ]
        ),

    "test_accuracy":
        float(accuracy),

    "test_balanced_accuracy":
        float(balanced_accuracy),

    "test_macro_f1":
        float(macro_f1),

    "age_<60_accuracy":
        float(age_results["age_<60"]),

    "age_>=60_accuracy":
        float(age_results["age_>=60"]),

    "age_<60_positive_rate":
        float(p_under60),

    "age_>=60_positive_rate":
        float(p_over60),

    "DPR":
        float(dpr),

    "EOD":
        float(eod),

    "absolute_EOD":
        float(abs(eod)),

    "n_test_samples":
        int(len(y_true)),

    "confusion_matrix":
        cm.tolist()
}


# ============================================================
# 12. PRINT
# ============================================================

print("\n")
print("=" * 70)
print("TASK 1 — FINAL WCE TEST RESULTS")
print("=" * 70)

print(
    f"Best epoch         : "
    f"{task1_final_results['epoch']}"
)

print(
    f"Validation Accuracy: "
    f"{task1_final_results['validation_accuracy']:.4f}"
)

print(
    f"Test Accuracy      : "
    f"{task1_final_results['test_accuracy']:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{task1_final_results['test_balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1           : "
    f"{task1_final_results['test_macro_f1']:.4f}"
)

print(
    f"Age <60 Accuracy   : "
    f"{task1_final_results['age_<60_accuracy']:.4f}"
)

print(
    f"Age >=60 Accuracy  : "
    f"{task1_final_results['age_>=60_accuracy']:.4f}"
)

print(
    f"Age <60 Pos Rate   : "
    f"{task1_final_results['age_<60_positive_rate']:.4f}"
)

print(
    f"Age >=60 Pos Rate  : "
    f"{task1_final_results['age_>=60_positive_rate']:.4f}"
)

print(
    f"DPR                : "
    f"{task1_final_results['DPR']:.4f}"
)

print(
    f"EOD                : "
    f"{task1_final_results['EOD']:.4f}"
)

print(
    f"|EOD|              : "
    f"{task1_final_results['absolute_EOD']:.4f}"
)

print("\nConfusion Matrix:")
print(cm)

print(
    "\nClass order:",
    TASK1_CLASSES
)


# ============================================================
# 13. SAVE FINAL TASK-1 RESULTS
# ============================================================

FINAL_RESULTS_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_final_wce_test_metrics.json"
)

with open(
    FINAL_RESULTS_PATH,
    "w"
) as f:

    json.dump(
        task1_final_results,
        f,
        indent=2
    )

print(
    "\nSaved final Task-1 results:"
)

print(
    FINAL_RESULTS_PATH
)


# ============================================================
# 14. FINAL CHECKPOINT STATUS
# ============================================================

print("\n" + "=" * 70)
print("TASK 1 IS NOW CHECKPOINTED")
print("=" * 70)

print(
    "Best WCE checkpoint:"
)

print(
    BEST_WCE_CHECKPOINT
)

print(
    "\nFinal Task-1 metrics:"
)

print(
    FINAL_RESULTS_PATH
)

print("=" * 70)

STEP 23 — TASK 1 FINAL WCE TEST EVALUATION


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [55]:
# ============================================================
# STEP 23 — TASK 1 FINAL WCE TEST EVALUATION
# CORRECTED CHECKPOINT LOADING
# ============================================================

import os
import json
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 23 — TASK 1 FINAL WCE TEST EVALUATION")
print("=" * 70)

device = torch.device("cuda")


# ============================================================
# 1. BEST WCE CHECKPOINT
# ============================================================

BEST_WCE_CHECKPOINT = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_wce_pruned_best.pt"
)

assert os.path.exists(
    BEST_WCE_CHECKPOINT
), "Best WCE checkpoint not found."

print("\nCheckpoint found:")
print(BEST_WCE_CHECKPOINT)


# ============================================================
# 2. LOAD CHECKPOINT
# ============================================================
#
# IMPORTANT:
# weights_only=False is required because our checkpoint
# contains history/configuration objects in addition to the
# model state_dict.
#
# This checkpoint was created by our own Kaggle notebook.
# ============================================================

checkpoint_wce = torch.load(
    BEST_WCE_CHECKPOINT,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded successfully.")


# ============================================================
# 3. LOAD MODEL
# ============================================================

model_task1.load_state_dict(
    checkpoint_wce["model_state_dict"]
)

model_task1 = model_task1.to(device)
model_task1.eval()

print(
    "Best epoch:",
    checkpoint_wce["epoch"]
)

print(
    "Best validation accuracy:",
    checkpoint_wce["best_val_accuracy"]
)


# ============================================================
# 4. LOAD PRUNING MASK
# ============================================================

assert "protected_masks_correct" in globals(), (
    "protected_masks_correct is missing."
)

mask_dict = {}

for name, mask in protected_masks_correct.items():

    mask_dict[name] = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()


# ============================================================
# 5. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task1.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task1,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


assert len(activation_modules) == 17


# ============================================================
# 6. REGISTER PRUNING MASK
# ============================================================

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        mask_4d = mask_tensor.view(
            1,
            -1,
            1,
            1
        )

        return output * mask_4d

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_mask_hook(
                mask_dict[name]
            )
        )
    )

print(
    "\nPruning hooks active:",
    len(handles)
)


# ============================================================
# 7. AGE GROUP CONVERSION
# ============================================================

def normalize_age_group(value):

    if isinstance(value, str):

        value = value.strip().lower()

        if value in [
            "age<60",
            "age <60",
            "age< 60",
            "age < 60"
        ]:
            return 0

        if value in [
            "age>=60",
            "age >=60",
            "age>= 60",
            "age >= 60"
        ]:
            return 1

        return -1


    if isinstance(
        value,
        (int, np.integer)
    ):

        if value in [0, 1]:
            return int(value)

        return -1


    if isinstance(
        value,
        (float, np.floating)
    ):

        if value in [0.0, 1.0]:
            return int(value)

        return -1


    return -1


# ============================================================
# 8. EVALUATE TEST SET
# ============================================================

y_true = []
y_pred = []
age_groups = []


try:

    with torch.no_grad():

        for batch in task1_test_loader:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            labels = batch["label"]

            if torch.is_tensor(labels):

                labels_cpu = (
                    labels
                    .detach()
                    .cpu()
                )

                labels_device = labels.to(
                    device,
                    non_blocking=True
                )

            else:

                labels_cpu = torch.tensor(
                    labels,
                    dtype=torch.long
                )

                labels_device = labels_cpu.to(
                    device
                )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits = model_task1(
                images
            )

            predictions = logits.argmax(
                dim=1
            )


            # ------------------------------------------------
            # Store labels
            # ------------------------------------------------

            y_true.extend(
                labels_cpu
                .numpy()
                .tolist()
            )


            # ------------------------------------------------
            # Store predictions
            # ------------------------------------------------

            y_pred.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            # ------------------------------------------------
            # Age groups
            # ------------------------------------------------

            batch_age = batch["age_group"]

            if torch.is_tensor(batch_age):

                batch_age = (
                    batch_age
                    .detach()
                    .cpu()
                    .numpy()
                    .tolist()
                )

            elif isinstance(
                batch_age,
                np.ndarray
            ):

                batch_age = (
                    batch_age.tolist()
                )

            elif not isinstance(
                batch_age,
                (list, tuple)
            ):

                batch_age = [batch_age]


            for value in batch_age:

                age_groups.append(
                    normalize_age_group(
                        value
                    )
                )


finally:

    for handle in handles:
        handle.remove()

    print(
        "\nPruning hooks removed."
    )


# ============================================================
# 9. ARRAYS
# ============================================================

y_true = np.asarray(
    y_true,
    dtype=np.int64
)

y_pred = np.asarray(
    y_pred,
    dtype=np.int64
)

age_groups = np.asarray(
    age_groups,
    dtype=np.int64
)

assert len(y_true) == len(y_pred)
assert len(y_true) == len(age_groups)

print(
    "\nTest samples:",
    len(y_true)
)

print(
    "Age <60:",
    int(np.sum(age_groups == 0))
)

print(
    "Age >=60:",
    int(np.sum(age_groups == 1))
)

print(
    "Missing age:",
    int(np.sum(age_groups == -1))
)


# ============================================================
# 10. OVERALL METRICS
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)


# ============================================================
# 11. AGE ACCURACY
# ============================================================

age_results = {}

for group_id, group_name in [
    (0, "age_<60"),
    (1, "age_>=60")
]:

    mask = age_groups == group_id

    if mask.sum() > 0:

        age_results[group_name] = float(
            accuracy_score(
                y_true[mask],
                y_pred[mask]
            )
        )

    else:

        age_results[group_name] = np.nan


# ============================================================
# 12. FAIRNESS METRICS
# ============================================================

positive_class = 1

positive_rates = {}
tprs = {}


for group_id, group_name in [
    (0, "age_<60"),
    (1, "age_>=60")
]:

    group_mask = (
        age_groups == group_id
    )


    # --------------------------------------------------------
    # Positive prediction rate
    # --------------------------------------------------------

    if group_mask.sum() > 0:

        positive_rates[group_name] = float(
            np.mean(
                y_pred[group_mask]
                == positive_class
            )
        )

    else:

        positive_rates[group_name] = np.nan


    # --------------------------------------------------------
    # TPR
    # --------------------------------------------------------

    true_positive_mask = (
        group_mask &
        (y_true == positive_class)
    )

    if true_positive_mask.sum() > 0:

        tprs[group_name] = float(
            np.mean(
                y_pred[true_positive_mask]
                == positive_class
            )
        )

    else:

        tprs[group_name] = np.nan


# ============================================================
# 13. DPR
# ============================================================

p_under60 = positive_rates[
    "age_<60"
]

p_over60 = positive_rates[
    "age_>=60"
]

if (
    not np.isnan(p_under60)
    and p_under60 > 0
):

    dpr = (
        p_over60 /
        p_under60
    )

else:

    dpr = np.nan


# ============================================================
# 14. EOD
# ============================================================

tpr_under60 = tprs[
    "age_<60"
]

tpr_over60 = tprs[
    "age_>=60"
]

if (
    not np.isnan(tpr_under60)
    and not np.isnan(tpr_over60)
):

    eod = (
        tpr_over60 -
        tpr_under60
    )

else:

    eod = np.nan


# ============================================================
# 15. FINAL RESULTS
# ============================================================

task1_final_results = {

    "checkpoint":
        BEST_WCE_CHECKPOINT,

    "epoch":
        int(checkpoint_wce["epoch"]),

    "validation_accuracy":
        float(
            checkpoint_wce[
                "best_val_accuracy"
            ]
        ),

    "test_accuracy":
        float(accuracy),

    "test_balanced_accuracy":
        float(balanced_accuracy),

    "test_macro_f1":
        float(macro_f1),

    "age_<60_accuracy":
        float(
            age_results["age_<60"]
        ),

    "age_>=60_accuracy":
        float(
            age_results["age_>=60"]
        ),

    "age_<60_positive_rate":
        float(p_under60),

    "age_>=60_positive_rate":
        float(p_over60),

    "DPR":
        float(dpr),

    "EOD":
        float(eod),

    "absolute_EOD":
        float(abs(eod)),

    "n_test_samples":
        int(len(y_true)),

    "confusion_matrix":
        cm.tolist()
}


# ============================================================
# 16. PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("TASK 1 — FINAL WCE TEST RESULTS")
print("=" * 70)

print(
    f"Best epoch         : "
    f"{task1_final_results['epoch']}"
)

print(
    f"Validation Accuracy: "
    f"{task1_final_results['validation_accuracy']:.4f}"
)

print(
    f"Test Accuracy      : "
    f"{task1_final_results['test_accuracy']:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{task1_final_results['test_balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1           : "
    f"{task1_final_results['test_macro_f1']:.4f}"
)

print(
    f"Age <60 Accuracy   : "
    f"{task1_final_results['age_<60_accuracy']:.4f}"
)

print(
    f"Age >=60 Accuracy  : "
    f"{task1_final_results['age_>=60_accuracy']:.4f}"
)

print(
    f"Age <60 Pos Rate   : "
    f"{task1_final_results['age_<60_positive_rate']:.4f}"
)

print(
    f"Age >=60 Pos Rate  : "
    f"{task1_final_results['age_>=60_positive_rate']:.4f}"
)

print(
    f"DPR                : "
    f"{task1_final_results['DPR']:.4f}"
)

print(
    f"EOD                : "
    f"{task1_final_results['EOD']:.4f}"
)

print(
    f"|EOD|              : "
    f"{task1_final_results['absolute_EOD']:.4f}"
)

print("\nConfusion Matrix:")
print(cm)

print(
    "\nClass order:",
    TASK1_CLASSES
)


# ============================================================
# 17. SAVE FINAL RESULTS
# ============================================================

FINAL_RESULTS_PATH = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_final_wce_test_metrics.json"
)

with open(
    FINAL_RESULTS_PATH,
    "w"
) as f:

    json.dump(
        task1_final_results,
        f,
        indent=2
    )

print(
    "\nSaved final Task-1 results:"
)

print(
    FINAL_RESULTS_PATH
)


# ============================================================
# 18. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("TASK 1 FINALIZED")
print("=" * 70)

print(
    "Best WCE checkpoint:"
)

print(
    BEST_WCE_CHECKPOINT
)

print(
    "\nFinal metrics:"
)

print(
    FINAL_RESULTS_PATH
)

print("=" * 70)

STEP 23 — TASK 1 FINAL WCE TEST EVALUATION

Checkpoint found:
/kaggle/working/biaspruner_checkpoints/task1/task1_wce_pruned_best.pt
Checkpoint loaded successfully.
Best epoch: 15
Best validation accuracy: 0.8117913832199547

Pruning hooks active: 17

Pruning hooks removed.

Test samples: 226
Age <60: 85
Age >=60: 140
Missing age: 1


TASK 1 — FINAL WCE TEST RESULTS
Best epoch         : 15
Validation Accuracy: 0.8118
Test Accuracy      : 0.7434
Balanced Accuracy  : 0.7430
Macro-F1           : 0.7426
Age <60 Accuracy   : 0.7882
Age >=60 Accuracy  : 0.7214
Age <60 Pos Rate   : 0.6353
Age >=60 Pos Rate  : 0.4929
DPR                : 0.7758
EOD                : -0.1948
|EOD|              : 0.1948

Confusion Matrix:
[[78 34]
 [24 90]]

Class order: ['bkl', 'mel']

Saved final Task-1 results:
/kaggle/working/biaspruner_checkpoints/task1/task1_final_wce_test_metrics.json

TASK 1 FINALIZED
Best WCE checkpoint:
/kaggle/working/biaspruner_checkpoints/task1/task1_wce_pruned_best.pt

Final metrics:

In [56]:
# ============================================================
# STEP 24 — TASK 1 RETENTION BASELINE BEFORE TASK 2
# ============================================================

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 24 — TASK 1 RETENTION BASELINE BEFORE TASK 2")
print("=" * 70)

device = torch.device("cuda")

# ------------------------------------------------------------
# 1. Load FINAL/BEST Task-1 WCE checkpoint
# ------------------------------------------------------------

TASK1_CHECKPOINT = (
    "/kaggle/working/biaspruner_checkpoints/"
    "task1/task1_wce_pruned_best.pt"
)

assert os.path.exists(TASK1_CHECKPOINT), (
    "Task-1 WCE checkpoint not found."
)

checkpoint = torch.load(
    TASK1_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model_task1.load_state_dict(
    checkpoint["model_state_dict"]
)

model_task1 = model_task1.to(device)
model_task1.eval()

print("Loaded Task-1 WCE checkpoint.")
print("Epoch:", checkpoint["epoch"])
print(
    "Validation accuracy:",
    checkpoint["best_val_accuracy"]
)


# ------------------------------------------------------------
# 2. Prepare Task-1 pruning masks
# ------------------------------------------------------------

assert "protected_masks_correct" in globals(), (
    "Task-1 pruning mask is missing."
)

task1_masks = {}

for name, mask in protected_masks_correct.items():

    task1_masks[name] = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()

print(
    "\nTask-1 protected channels:",
    sum(
        int(mask.sum().item())
        for mask in task1_masks.values()
    )
)

print(
    "Task-1 total channels:",
    sum(
        len(mask)
        for mask in task1_masks.values()
    )
)


# ------------------------------------------------------------
# 3. Activation modules
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task1.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task1,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block

assert len(activation_modules) == 17


# ------------------------------------------------------------
# 4. Register Task-1 pruning mask
# ------------------------------------------------------------

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        return output * mask_tensor.view(
            1, -1, 1, 1
        )

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_mask_hook(
                task1_masks[name]
            )
        )
    )

print(
    "Pruning hooks active:",
    len(handles)
)


# ------------------------------------------------------------
# 5. Evaluate Task-1 test set
# ------------------------------------------------------------

y_true = []
y_pred = []


try:

    with torch.no_grad():

        for batch in task1_test_loader:

            images = batch["image"].to(
                device
            )

            labels = batch["label"]

            if torch.is_tensor(labels):
                labels_cpu = (
                    labels.cpu()
                )
            else:
                labels_cpu = torch.tensor(
                    labels,
                    dtype=torch.long
                )

            logits = model_task1(
                images
            )

            predictions = logits.argmax(
                dim=1
            )

            y_true.extend(
                labels_cpu.numpy().tolist()
            )

            y_pred.extend(
                predictions.cpu()
                .numpy()
                .tolist()
            )

finally:

    for handle in handles:
        handle.remove()


# ------------------------------------------------------------
# 6. Metrics
# ------------------------------------------------------------

y_true = np.asarray(
    y_true,
    dtype=np.int64
)

y_pred = np.asarray(
    y_pred,
    dtype=np.int64
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)


# ------------------------------------------------------------
# 7. Save retention baseline
# ------------------------------------------------------------

baseline = {

    "task": "Task1",

    "classes": TASK1_CLASSES,

    "checkpoint":
        TASK1_CHECKPOINT,

    "epoch":
        int(checkpoint["epoch"]),

    "validation_accuracy":
        float(
            checkpoint[
                "best_val_accuracy"
            ]
        ),

    "test_accuracy":
        float(accuracy),

    "test_balanced_accuracy":
        float(
            balanced_accuracy
        ),

    "test_macro_f1":
        float(macro_f1),

    "confusion_matrix":
        cm.tolist(),

    "n_test_samples":
        int(len(y_true))
}


BASELINE_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task1/"
    "task1_retention_baseline.json"
)

with open(
    BASELINE_PATH,
    "w"
) as f:

    json.dump(
        baseline,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 8. Print
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK 1 RETENTION BASELINE")
print("=" * 70)

print(
    f"Test Accuracy     : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"Macro-F1          : {macro_f1:.4f}"
)

print("\nConfusion Matrix:")
print(cm)

print(
    "\nSaved:",
    BASELINE_PATH
)

print("=" * 70)
print("TASK 1 IS LOCKED BEFORE TASK 2")
print("=" * 70)

STEP 24 — TASK 1 RETENTION BASELINE BEFORE TASK 2
Loaded Task-1 WCE checkpoint.
Epoch: 15
Validation accuracy: 0.8117913832199547

Task-1 protected channels: 6067
Task-1 total channels: 15168
Pruning hooks active: 17

TASK 1 RETENTION BASELINE
Test Accuracy     : 0.7434
Balanced Accuracy : 0.7430
Macro-F1          : 0.7426

Confusion Matrix:
[[78 34]
 [24 90]]

Saved: /kaggle/working/biaspruner_checkpoints/task1/task1_retention_baseline.json
TASK 1 IS LOCKED BEFORE TASK 2


In [57]:
# ============================================================
# STEP 25 — TASK 2 DATA PREPARATION
# Task 2 = bcc + akiec
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

print("=" * 70)
print("STEP 25 — TASK 2 DATA PREPARATION")
print("=" * 70)


# ============================================================
# 1. TASK 2 DEFINITION
# ============================================================

TASK2_CLASSES = ["bcc", "akiec"]

print("\nTask 2 classes:", TASK2_CLASSES)


# ============================================================
# 2. VERIFY LOCKED SPLIT
# ============================================================

SPLIT_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "splits/"
    "ham10000_lesion_split_seed42.csv"
)

assert os.path.exists(
    SPLIT_PATH
), "Locked split file not found."

split_df = pd.read_csv(
    SPLIT_PATH
)

print("\nLocked split loaded:")
print(SPLIT_PATH)

print(
    "Rows:",
    len(split_df)
)

print(
    "Columns:",
    list(split_df.columns)
)


# ============================================================
# 3. IDENTIFY SPLIT COLUMN
# ============================================================

possible_split_columns = [
    "split",
    "set",
    "partition"
]

split_column = None

for col in possible_split_columns:

    if col in split_df.columns:

        split_column = col
        break

assert split_column is not None, (
    "Could not find split column."
)

print(
    "Split column:",
    split_column
)

print(
    "Split values:",
    split_df[split_column]
    .value_counts()
    .to_dict()
)


# ============================================================
# 4. IDENTIFY IMAGE/CLASS COLUMNS
# ============================================================

assert "image_id" in split_df.columns, (
    "image_id column missing."
)

assert "dx" in split_df.columns, (
    "dx column missing."
)


# ============================================================
# 5. BUILD TASK-2 DATAFRAMES
# ============================================================

task2_all_df = split_df[
    split_df["dx"].isin(
        TASK2_CLASSES
    )
].copy()


task2_train_df = task2_all_df[
    task2_all_df[split_column] == "train"
].copy()


task2_val_df = task2_all_df[
    task2_all_df[split_column] == "val"
].copy()


task2_test_df = task2_all_df[
    task2_all_df[split_column] == "test"
].copy()


# ============================================================
# 6. BASIC COUNTS
# ============================================================

print("\n" + "-" * 70)
print("TASK 2 COUNTS")
print("-" * 70)

print(
    "All:",
    len(task2_all_df)
)

print(
    "Train:",
    len(task2_train_df)
)

print(
    "Validation:",
    len(task2_val_df)
)

print(
    "Test:",
    len(task2_test_df)
)


# ============================================================
# 7. CLASS DISTRIBUTION
# ============================================================

print("\nClass distribution:")

distribution = pd.DataFrame({

    "train":
        task2_train_df["dx"]
        .value_counts()
        .reindex(TASK2_CLASSES)
        .fillna(0)
        .astype(int),

    "val":
        task2_val_df["dx"]
        .value_counts()
        .reindex(TASK2_CLASSES)
        .fillna(0)
        .astype(int),

    "test":
        task2_test_df["dx"]
        .value_counts()
        .reindex(TASK2_CLASSES)
        .fillna(0)
        .astype(int)

})

print(distribution)


# ============================================================
# 8. VERIFY NO OTHER CLASSES
# ============================================================

unexpected = sorted(
    set(task2_all_df["dx"])
    - set(TASK2_CLASSES)
)

assert len(unexpected) == 0, (
    f"Unexpected classes: {unexpected}"
)


# ============================================================
# 9. VERIFY SPLIT INTEGRITY
# ============================================================

train_ids = set(
    task2_train_df["image_id"]
)

val_ids = set(
    task2_val_df["image_id"]
)

test_ids = set(
    task2_test_df["image_id"]
)

print("\nImage-ID overlap:")

print(
    "Train ∩ Val :",
    len(train_ids & val_ids)
)

print(
    "Train ∩ Test:",
    len(train_ids & test_ids)
)

print(
    "Val ∩ Test  :",
    len(val_ids & test_ids)
)

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0


# ============================================================
# 10. VERIFY LESION-LEVEL SPLIT
# ============================================================

if "lesion_id" in task2_all_df.columns:

    train_lesions = set(
        task2_train_df["lesion_id"]
    )

    val_lesions = set(
        task2_val_df["lesion_id"]
    )

    test_lesions = set(
        task2_test_df["lesion_id"]
    )

    print("\nLesion-ID overlap:")

    print(
        "Train ∩ Val :",
        len(train_lesions & val_lesions)
    )

    print(
        "Train ∩ Test:",
        len(train_lesions & test_lesions)
    )

    print(
        "Val ∩ Test  :",
        len(val_lesions & test_lesions)
    )

    assert len(
        train_lesions & val_lesions
    ) == 0

    assert len(
        train_lesions & test_lesions
    ) == 0

    assert len(
        val_lesions & test_lesions
    ) == 0


# ============================================================
# 11. VERIFY DATASET CLASS EXISTS
# ============================================================

assert "HAM10000TaskDataset" in globals(), (
    "HAM10000TaskDataset is not available."
)

assert "image_lookup" in globals(), (
    "image_lookup is not available."
)


# ============================================================
# 12. CREATE TASK-2 DATASETS
# ============================================================

task2_train_dataset = HAM10000TaskDataset(
    task2_train_df,
    image_lookup,
    TASK2_CLASSES,
    transform=train_transform
)

task2_val_dataset = HAM10000TaskDataset(
    task2_val_df,
    image_lookup,
    TASK2_CLASSES,
    transform=eval_transform
)

task2_test_dataset = HAM10000TaskDataset(
    task2_test_df,
    image_lookup,
    TASK2_CLASSES,
    transform=eval_transform
)


# ============================================================
# 13. CREATE TASK-2 DATALOADERS
# ============================================================

task2_train_loader = DataLoader(
    task2_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

task2_analysis_loader = DataLoader(
    task2_train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

task2_val_loader = DataLoader(
    task2_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

task2_test_loader = DataLoader(
    task2_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# 14. VERIFY ONE BATCH
# ============================================================

batch = next(
    iter(task2_train_loader)
)

print("\n" + "-" * 70)
print("TASK 2 BATCH VERIFICATION")
print("-" * 70)

print(
    "Batch keys:",
    list(batch.keys())
)

print(
    "Images:",
    batch["image"].shape,
    batch["image"].dtype
)

print(
    "Labels:",
    batch["label"].shape,
    batch["label"].dtype
)

print(
    "Class names:",
    batch["class_name"][:5]
)

print(
    "Age groups:",
    batch["age_group"][:5]
)

print(
    "Image IDs:",
    batch["image_id"][:5]
)


# ============================================================
# 15. VERIFY LABEL MAPPING
# ============================================================

print("\nLabel mapping:")

for i, class_name in enumerate(TASK2_CLASSES):

    print(
        f"  {i} -> {class_name}"
    )

assert set(
    batch["label"].cpu().numpy()
).issubset({0, 1})


# ============================================================
# 16. SAVE TASK-2 CONFIGURATION
# ============================================================

TASK2_CONFIG_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2_config.txt"
)

with open(
    TASK2_CONFIG_PATH,
    "w"
) as f:

    f.write(
        "BiasPruner HAM10000 Experiment\n"
    )

    f.write(
        "================================\n\n"
    )

    f.write(
        "Task 2 classes: "
        + str(TASK2_CLASSES)
        + "\n"
    )

    f.write(
        "Locked split: "
        + SPLIT_PATH
        + "\n"
    )

    f.write(
        f"Train samples: "
        f"{len(task2_train_df)}\n"
    )

    f.write(
        f"Validation samples: "
        f"{len(task2_val_df)}\n"
    )

    f.write(
        f"Test samples: "
        f"{len(task2_test_df)}\n"
    )

    f.write(
        "Batch size: 32\n"
    )

    f.write(
        "Image size: 224x224\n"
    )

    f.write(
        "GCE q: 0.7\n"
    )

    f.write(
        "Pruning ratio gamma: 0.60\n"
    )


# ============================================================
# 17. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("STEP 25 COMPLETE")
print("=" * 70)

print(
    "Task 2:",
    TASK2_CLASSES
)

print(
    "Train:",
    len(task2_train_df)
)

print(
    "Validation:",
    len(task2_val_df)
)

print(
    "Test:",
    len(task2_test_df)
)

print(
    "Configuration saved:",
    TASK2_CONFIG_PATH
)

print("=" * 70)

STEP 25 — TASK 2 DATA PREPARATION

Task 2 classes: ['bcc', 'akiec']

Locked split loaded:
/kaggle/working/biaspruner_checkpoints/splits/ham10000_lesion_split_seed42.csv
Rows: 10015
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'age_group', 'split']
Split column: split
Split values: {'train': 6981, 'val': 2033, 'test': 1001}

----------------------------------------------------------------------
TASK 2 COUNTS
----------------------------------------------------------------------
All: 841
Train: 583
Validation: 178
Test: 80

Class distribution:
       train  val  test
dx                     
bcc      361  106    47
akiec    222   72    33

Image-ID overlap:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Lesion-ID overlap:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

----------------------------------------------------------------------
TASK 2 BATCH VERIFICATION
----------------------------------------------------------------------
Batch keys: ['i

In [58]:
# ============================================================
# STEP 26 — TASK 2 CONTINUAL-LEARNING MODEL INITIALIZATION
# ============================================================

import os
import torch
import torch.nn as nn

print("=" * 70)
print("STEP 26 — TASK 2 CONTINUAL-LEARNING MODEL INITIALIZATION")
print("=" * 70)

device = torch.device("cuda")


# ============================================================
# 1. TASK DEFINITIONS
# ============================================================

TASK1_CLASSES = ["bkl", "mel"]
TASK2_CLASSES = ["bcc", "akiec"]

print("\nTask 1:", TASK1_CLASSES)
print("Task 2:", TASK2_CLASSES)


# ============================================================
# 2. LOAD TASK-1 FINAL WCE CHECKPOINT
# ============================================================

TASK1_CHECKPOINT = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task1/"
    "task1_wce_pruned_best.pt"
)

assert os.path.exists(
    TASK1_CHECKPOINT
), "Task-1 checkpoint not found."

checkpoint_task1 = torch.load(
    TASK1_CHECKPOINT,
    map_location=device,
    weights_only=False
)

print("\nTask-1 checkpoint loaded.")
print(
    "Epoch:",
    checkpoint_task1["epoch"]
)

print(
    "Validation accuracy:",
    checkpoint_task1["best_val_accuracy"]
)


# ============================================================
# 3. CREATE FRESH STANDARD RESNET-50 CONTAINER
# ============================================================
#
# We create a new model with the SAME architecture and then
# load Task-1 weights.
#
# This avoids accidentally modifying the existing model_task1.
# ============================================================

from torchvision.models import resnet50

model_task2 = resnet50(
    weights=None,
    num_classes=2
)

model_task2 = model_task2.to(device)


# ============================================================
# 4. LOAD TASK-1 WEIGHTS
# ============================================================

model_task2.load_state_dict(
    checkpoint_task1["model_state_dict"]
)

print(
    "\nTask-1 weights loaded into model_task2."
)


# ============================================================
# 5. VERIFY ARCHITECTURE
# ============================================================

assert isinstance(
    model_task2,
    torchvision.models.ResNet
) if "torchvision" in globals() else True

assert model_task2.fc.in_features == 2048
assert model_task2.fc.out_features == 2

print(
    "Backbone:",
    type(model_task2).__name__
)

print(
    "Classifier:",
    model_task2.fc.in_features,
    "->",
    model_task2.fc.out_features
)


# ============================================================
# 6. LOAD TASK-1 PROTECTED MASK
# ============================================================

assert "protected_masks_correct" in globals(), (
    "Task-1 protected mask is missing."
)

task1_protected_masks = {}

for name, mask in protected_masks_correct.items():

    task1_protected_masks[name] = torch.as_tensor(
        mask,
        dtype=torch.float32,
        device=device
    ).flatten()


# ============================================================
# 7. CONSTRUCT TASK-2 FREE MASK
# ============================================================
#
# Task-1 protected = 1
# Task-1 free      = 0
#
# Therefore:
#
# Task-2 free mask = 1 - Task-1 protected mask
#
# These are the units available to Task 2.
# ============================================================

task2_free_masks = {}

for name, mask in task1_protected_masks.items():

    task2_free_masks[name] = (
        1.0 - mask
    )


# ============================================================
# 8. COUNT CAPACITY
# ============================================================

total_units = 0
protected_units = 0
free_units = 0

for name in task1_protected_masks:

    protected = int(
        task1_protected_masks[name]
        .sum()
        .item()
    )

    free = int(
        task2_free_masks[name]
        .sum()
        .item()
    )

    total = len(
        task1_protected_masks[name]
    )

    protected_units += protected
    free_units += free
    total_units += total


print("\n" + "-" * 70)
print("CONTINUAL-LEARNING CAPACITY")
print("-" * 70)

print(
    "Total units     :",
    total_units
)

print(
    "Task-1 protected:",
    protected_units
)

print(
    "Task-2 free     :",
    free_units
)

assert total_units == 15168
assert protected_units == 6067
assert free_units == 9101


# ============================================================
# 9. CHECK MASK COMPLEMENT
# ============================================================

mask_errors = 0

for name in task1_protected_masks:

    combined = (
        task1_protected_masks[name]
        + task2_free_masks[name]
    )

    if not torch.allclose(
        combined,
        torch.ones_like(combined)
    ):

        mask_errors += 1


print(
    "Mask complement errors:",
    mask_errors
)

assert mask_errors == 0


# ============================================================
# 10. IMPORTANT: CLASSIFIER
# ============================================================
#
# The backbone is inherited from Task 1.
#
# Task 2 needs its own classifier:
#
#       backbone → bcc / akiec
#
# The Task-1 classifier itself is NOT used for Task-2
# predictions.
#
# This classifier is new and trainable.
# ============================================================

model_task2.fc = nn.Linear(
    2048,
    2
).to(device)

print(
    "\nTask-2 classifier created:"
)

print(
    model_task2.fc
)


# ============================================================
# 11. VERIFY NEW CLASSIFIER
# ============================================================

assert model_task2.fc.in_features == 2048
assert model_task2.fc.out_features == 2


# ============================================================
# 12. VERIFY FORWARD PASS
# ============================================================

model_task2.eval()

batch = next(
    iter(task2_analysis_loader)
)

images = batch["image"].to(
    device
)

with torch.no_grad():

    logits = model_task2(
        images
    )

print(
    "\nForward verification:"
)

print(
    "Input:",
    tuple(images.shape)
)

print(
    "Logits:",
    tuple(logits.shape)
)

assert logits.shape == (
    images.shape[0],
    2
)

assert torch.isfinite(
    logits
).all()


# ============================================================
# 13. SAVE TASK-2 INITIALIZATION CHECKPOINT
# ============================================================

TASK2_INIT_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2/"
    "task2_initialized_from_task1.pt"
)

os.makedirs(
    os.path.dirname(TASK2_INIT_PATH),
    exist_ok=True
)

torch.save(
    {
        "model_state_dict":
            model_task2.state_dict(),

        "task1_checkpoint":
            TASK1_CHECKPOINT,

        "task1_classes":
            TASK1_CLASSES,

        "task2_classes":
            TASK2_CLASSES,

        "total_units":
            total_units,

        "task1_protected_units":
            protected_units,

        "task2_free_units":
            free_units
    },
    TASK2_INIT_PATH
)


# ============================================================
# 14. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("STEP 26 COMPLETE")
print("=" * 70)

print(
    "Task-1 protected units:",
    protected_units
)

print(
    "Task-2 free units:",
    free_units
)

print(
    "Task-2 classifier:",
    "2048 -> 2"
)

print(
    "Initialization checkpoint:"
)

print(
    TASK2_INIT_PATH
)

print("=" * 70)

STEP 26 — TASK 2 CONTINUAL-LEARNING MODEL INITIALIZATION

Task 1: ['bkl', 'mel']
Task 2: ['bcc', 'akiec']

Task-1 checkpoint loaded.
Epoch: 15
Validation accuracy: 0.8117913832199547

Task-1 weights loaded into model_task2.
Backbone: ResNet
Classifier: 2048 -> 2

----------------------------------------------------------------------
CONTINUAL-LEARNING CAPACITY
----------------------------------------------------------------------
Total units     : 15168
Task-1 protected: 6067
Task-2 free     : 9101
Mask complement errors: 0

Task-2 classifier created:
Linear(in_features=2048, out_features=2, bias=True)

Forward verification:
Input: (32, 3, 224, 224)
Logits: (32, 2)

STEP 26 COMPLETE
Task-1 protected units: 6067
Task-2 free units: 9101
Task-2 classifier: 2048 -> 2
Initialization checkpoint:
/kaggle/working/biaspruner_checkpoints/task2/task2_initialized_from_task1.pt


In [59]:
# ============================================================
# STEP 27 — TASK-2 CONTINUAL-LEARNING FREEZE VERIFICATION
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("STEP 27 — TASK-2 CONTINUAL-LEARNING FREEZE VERIFICATION")
print("=" * 70)

device = torch.device("cuda")

# ============================================================
# IMPORTANT
# ============================================================
#
# We are using a FUNCTIONAL activation-mask approximation.
#
# Therefore we can directly guarantee that Task-1 protected
# ACTIVATION CHANNELS are masked during Task-2 training.
#
# We also verify which parameters receive gradients.
#
# We do NOT claim this is exact structural ResNet-50 pruning.
# ============================================================


# ============================================================
# 1. MODEL
# ============================================================

model_task2 = model_task2.to(device)
model_task2.train()

print("\nModel ready.")


# ============================================================
# 2. TASK-1 PROTECTED MASKS
# ============================================================

assert "task1_protected_masks" in globals()
assert "task2_free_masks" in globals()

protected_units = sum(
    int(mask.sum().item())
    for mask in task1_protected_masks.values()
)

free_units = sum(
    int(mask.sum().item())
    for mask in task2_free_masks.values()
)

print(
    "\nTask-1 protected units:",
    protected_units
)

print(
    "Task-2 free units:",
    free_units
)

assert protected_units == 6067
assert free_units == 9101


# ============================================================
# 3. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task2.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task2,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


assert len(activation_modules) == 17

print(
    "\nActivation points:",
    len(activation_modules)
)


# ============================================================
# 4. CREATE PROTECTED-MASK HOOKS
# ============================================================

handles = []


def make_protected_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        # Only Task-2 free channels are allowed to remain active.
        #
        # Task-1 protected channels are suppressed for the
        # Task-2 functional subnetwork.

        return output * (
            1.0 - mask_tensor
        ).view(
            1, -1, 1, 1
        )

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_protected_mask_hook(
                task1_protected_masks[name]
            )
        )
    )


print(
    "Mask hooks registered:",
    len(handles)
)


# ============================================================
# 5. CREATE TASK-2 OPTIMIZER
# ============================================================
#
# We train the model normally through the FREE functional
# subnetwork.
#
# IMPORTANT:
# Because ResNet-50 bottleneck parameters are shared across
# protected/free output channels, ordinary parameter-level
# requires_grad=False cannot exactly freeze individual
# channels.
#
# Therefore this verification explicitly checks the functional
# mask and gradient behavior instead of pretending we have
# exact structural parameter freezing.
# ============================================================

optimizer_task2 = torch.optim.Adam(
    model_task2.parameters(),
    lr=1e-4
)


# ============================================================
# 6. GET ONE TASK-2 BATCH
# ============================================================

batch = next(
    iter(task2_analysis_loader)
)

images = batch["image"].to(
    device
)

labels = batch["label"].to(
    device
)

print(
    "\nBatch:",
    tuple(images.shape)
)

print(
    "Labels:",
    tuple(labels.shape)
)


# ============================================================
# 7. CLEAR GRADIENTS
# ============================================================

optimizer_task2.zero_grad(
    set_to_none=True
)


# ============================================================
# 8. FORWARD + GCE LOSS
# ============================================================

logits = model_task2(
    images
)

probabilities = torch.softmax(
    logits,
    dim=1
)

q = 0.7

true_class_probability = probabilities[
    torch.arange(
        labels.size(0),
        device=device
    ),
    labels
]

gce_loss = (
    (
        1.0
        -
        true_class_probability.pow(q)
    )
    / q
).mean()


print(
    "\nGCE loss:",
    float(gce_loss.item())
)


# ============================================================
# 9. BACKWARD
# ============================================================

gce_loss.backward()


# ============================================================
# 10. GRADIENT STATISTICS
# ============================================================

total_parameters = 0
parameters_with_gradient = 0
parameters_without_gradient = 0

total_gradient_norm = 0.0

nonfinite_gradients = 0


for name, parameter in model_task2.named_parameters():

    total_parameters += 1

    if parameter.grad is None:

        parameters_without_gradient += 1

        continue

    parameters_with_gradient += 1

    grad = parameter.grad

    if not torch.isfinite(
        grad
    ).all():

        nonfinite_gradients += 1

    total_gradient_norm += (
        grad.detach()
        .norm()
        .item()
    ) ** 2


total_gradient_norm = (
    total_gradient_norm ** 0.5
)


print("\n" + "-" * 70)
print("GRADIENT VERIFICATION")
print("-" * 70)

print(
    "Total parameter tensors:",
    total_parameters
)

print(
    "Parameters with gradient:",
    parameters_with_gradient
)

print(
    "Parameters without gradient:",
    parameters_without_gradient
)

print(
    "Total gradient norm:",
    total_gradient_norm
)

print(
    "Non-finite gradients:",
    nonfinite_gradients
)


# ============================================================
# 11. VERIFY GRADIENTS ARE FINITE
# ============================================================

assert nonfinite_gradients == 0

assert parameters_with_gradient > 0

assert torch.isfinite(
    torch.tensor(
        total_gradient_norm,
        device=device
    )
)


# ============================================================
# 12. VERIFY PROTECTED CHANNEL MASKS
# ============================================================
#
# We directly inspect the output of each activation point.
# Protected channels must be exactly zero.
# ============================================================

activation_outputs = {}


def make_capture_hook(name):

    def hook(module, inputs, output):

        activation_outputs[name] = (
            output.detach()
            .clone()
        )

    return hook


capture_handles = []

for name, module in activation_modules.items():

    capture_handles.append(
        module.register_forward_hook(
            make_capture_hook(name)
        )
    )


with torch.no_grad():

    _ = model_task2(
        images
    )


# Remove capture hooks
for handle in capture_handles:
    handle.remove()


# ============================================================
# 13. CHECK PROTECTED CHANNELS
# ============================================================

max_protected_activation = 0.0

protected_channel_errors = 0

for name in activation_modules:

    output = activation_outputs[name]

    protected_mask = (
        task1_protected_masks[name]
        .bool()
    )

    if protected_mask.sum() == 0:
        continue

    protected_values = output[
        :,
        protected_mask,
        :,
        :
    ]

    maximum = float(
        protected_values.abs().max().item()
    )

    max_protected_activation = max(
        max_protected_activation,
        maximum
    )

    if maximum > 1e-7:

        protected_channel_errors += 1


print("\n" + "-" * 70)
print("FUNCTIONAL MASK VERIFICATION")
print("-" * 70)

print(
    "Protected-channel errors:",
    protected_channel_errors
)

print(
    "Maximum protected activation:",
    max_protected_activation
)

assert protected_channel_errors == 0


# ============================================================
# 14. CLEANUP
# ============================================================

optimizer_task2.zero_grad(
    set_to_none=True
)

for handle in handles:
    handle.remove()

print(
    "\nMask hooks removed."
)


# ============================================================
# 15. VERIFY NO HOOKS REMAIN
# ============================================================

remaining_hooks = sum(
    len(module._forward_hooks)
    for module in activation_modules.values()
)

print(
    "Remaining forward hooks:",
    remaining_hooks
)

assert remaining_hooks == 0


# ============================================================
# 16. SAVE VERIFICATION
# ============================================================

verification = {

    "task1_protected_units":
        protected_units,

    "task2_free_units":
        free_units,

    "activation_points":
        len(activation_modules),

    "gce_q":
        q,

    "gradient_parameters":
        parameters_with_gradient,

    "parameters_without_gradient":
        parameters_without_gradient,

    "gradient_norm":
        float(total_gradient_norm),

    "nonfinite_gradients":
        nonfinite_gradients,

    "protected_channel_errors":
        protected_channel_errors,

    "max_protected_activation":
        max_protected_activation,

    "remaining_hooks":
        remaining_hooks
}


import json

VERIFY_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2/"
    "task2_freeze_verification.json"
)

with open(
    VERIFY_PATH,
    "w"
) as f:

    json.dump(
        verification,
        f,
        indent=2
    )


# ============================================================
# 17. FINAL
# ============================================================

print("\n" + "=" * 70)
print("STEP 27 COMPLETE")
print("=" * 70)

print(
    "Functional Task-2 mask verified."
)

print(
    "Task-1 protected units:",
    protected_units
)

print(
    "Task-2 free units:",
    free_units
)

print(
    "Protected activation errors:",
    protected_channel_errors
)

print(
    "Verification saved:",
    VERIFY_PATH
)

print("=" * 70)

STEP 27 — TASK-2 CONTINUAL-LEARNING FREEZE VERIFICATION

Model ready.

Task-1 protected units: 6067
Task-2 free units: 9101

Activation points: 17
Mask hooks registered: 17

Batch: (32, 3, 224, 224)
Labels: (32,)

GCE loss: 0.34921127557754517

----------------------------------------------------------------------
GRADIENT VERIFICATION
----------------------------------------------------------------------
Total parameter tensors: 161
Parameters with gradient: 161
Parameters without gradient: 0
Total gradient norm: 170.28448007421778
Non-finite gradients: 0

----------------------------------------------------------------------
FUNCTIONAL MASK VERIFICATION
----------------------------------------------------------------------
Protected-channel errors: 0
Maximum protected activation: 0.0

Mask hooks removed.
Remaining forward hooks: 0

STEP 27 COMPLETE
Functional Task-2 mask verified.
Task-1 protected units: 6067
Task-2 free units: 9101
Protected activation errors: 0
Verification saved: 

In [60]:
# ============================================================
# STEP 28 — TASK 2 GCE TRAINING
# ============================================================

import os
import json
import time
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 70)
print("STEP 28 — TASK 2 GCE TRAINING")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

device = torch.device("cuda")

TASK2_Q = 0.7
TASK2_LR = 1e-4
TASK2_EPOCHS = 20

CHECKPOINT_DIR = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print("\nConfiguration:")
print("GCE q       :", TASK2_Q)
print("Learning rate:", TASK2_LR)
print("Epochs      :", TASK2_EPOCHS)
print("Batch size  :", 32)


# ============================================================
# 2. RELOAD CLEAN TASK-1 INITIALIZATION
# ============================================================
#
# This guarantees that Step 28 starts from exactly the state
# verified in Step 26, rather than from any accidental changes
# made during Step 27.
# ============================================================

INIT_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2/"
    "task2_initialized_from_task1.pt"
)

assert os.path.exists(
    INIT_PATH
), "Task-2 initialization checkpoint not found."


init_checkpoint = torch.load(
    INIT_PATH,
    map_location=device,
    weights_only=False
)

model_task2.load_state_dict(
    init_checkpoint["model_state_dict"]
)

model_task2 = model_task2.to(device)

print(
    "\nReloaded clean Task-2 initialization."
)


# ============================================================
# 3. RECREATE TASK-2 CLASSIFIER
# ============================================================

# The initialization checkpoint already contains the Task-2
# classifier. Verify it rather than replacing it again.

assert model_task2.fc.in_features == 2048
assert model_task2.fc.out_features == 2


# ============================================================
# 4. ACTIVATION MODULES
# ============================================================

activation_modules = {
    "relu": model_task2.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task2,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block

assert len(
    activation_modules
) == 17


# ============================================================
# 5. REGISTER TASK-1 PROTECTED MASK
# ============================================================

handles = []


def make_protected_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        return output * (
            1.0 - mask_tensor
        ).view(
            1, -1, 1, 1
        )

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_protected_mask_hook(
                task1_protected_masks[name]
            )
        )
    )

print(
    "Protected-mask hooks:",
    len(handles)
)


# ============================================================
# 6. GCE FUNCTION
# ============================================================

def generalized_cross_entropy(
    logits,
    labels,
    q=0.7
):

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    true_class_probability = probabilities[
        torch.arange(
            labels.size(0),
            device=labels.device
        ),
        labels
    ]

    loss = (
        1.0
        -
        true_class_probability.pow(q)
    ) / q

    return loss.mean()


# ============================================================
# 7. OPTIMIZER
# ============================================================

optimizer_task2 = torch.optim.Adam(
    model_task2.parameters(),
    lr=TASK2_LR
)


# ============================================================
# 8. TRAINING HISTORY
# ============================================================

history_task2 = []

best_val_accuracy = -1.0
best_epoch = None

start_time = time.time()


# ============================================================
# 9. TRAINING LOOP
# ============================================================

try:

    for epoch in range(
        1,
        TASK2_EPOCHS + 1
    ):

        epoch_start = time.time()

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model_task2.train()

        train_losses = []
        train_true = []
        train_pred = []


        for batch in task2_train_loader:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            optimizer_task2.zero_grad(
                set_to_none=True
            )

            logits = model_task2(
                images
            )

            loss = generalized_cross_entropy(
                logits,
                labels,
                q=TASK2_Q
            )

            loss.backward()

            optimizer_task2.step()

            train_losses.append(
                float(loss.item())
            )

            predictions = logits.argmax(
                dim=1
            )

            train_true.extend(
                labels.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            train_pred.extend(
                predictions.detach()
                .cpu()
                .numpy()
                .tolist()
            )


        train_accuracy = accuracy_score(
            train_true,
            train_pred
        )

        train_loss = float(
            np.mean(train_losses)
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        model_task2.eval()

        val_losses = []
        val_true = []
        val_pred = []


        with torch.no_grad():

            for batch in task2_val_loader:

                images = batch["image"].to(
                    device,
                    non_blocking=True
                )

                labels = batch["label"].to(
                    device,
                    non_blocking=True
                )

                logits = model_task2(
                    images
                )

                loss = generalized_cross_entropy(
                    logits,
                    labels,
                    q=TASK2_Q
                )

                val_losses.append(
                    float(loss.item())
                )

                predictions = logits.argmax(
                    dim=1
                )

                val_true.extend(
                    labels.cpu()
                    .numpy()
                    .tolist()
                )

                val_pred.extend(
                    predictions.cpu()
                    .numpy()
                    .tolist()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred
        )

        val_balanced_accuracy = (
            balanced_accuracy_score(
                val_true,
                val_pred
            )
        )

        val_macro_f1 = f1_score(
            val_true,
            val_pred,
            average="macro",
            zero_division=0
        )

        val_loss = float(
            np.mean(val_losses)
        )


        # ----------------------------------------------------
        # SAVE HISTORY
        # ----------------------------------------------------

        epoch_record = {

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                float(
                    train_accuracy
                ),

            "val_loss":
                val_loss,

            "val_accuracy":
                float(
                    val_accuracy
                ),

            "val_balanced_accuracy":
                float(
                    val_balanced_accuracy
                ),

            "val_macro_f1":
                float(
                    val_macro_f1
                ),

            "epoch_time_seconds":
                float(
                    time.time()
                    - epoch_start
                )
        }

        history_task2.append(
            epoch_record
        )


        # ----------------------------------------------------
        # BEST CHECKPOINT
        # ----------------------------------------------------

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_path = (
                CHECKPOINT_DIR
                +
                "/task2_gce_best.pt"
            )

            torch.save(
                {
                    "epoch":
                        epoch,

                    "model_state_dict":
                        model_task2.state_dict(),

                    "optimizer_state_dict":
                        optimizer_task2.state_dict(),

                    "task2_classes":
                        TASK2_CLASSES,

                    "gce_q":
                        TASK2_Q,

                    "learning_rate":
                        TASK2_LR,

                    "best_val_accuracy":
                        best_val_accuracy,

                    "task1_protected_units":
                        6067,

                    "task2_free_units":
                        9101
                },
                best_path
            )


        # ----------------------------------------------------
        # EVERY-EPOCH CHECKPOINT
        # ----------------------------------------------------

        epoch_path = (
            CHECKPOINT_DIR
            +
            f"/task2_gce_epoch_{epoch:03d}.pt"
        )

        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model_task2.state_dict(),

                "optimizer_state_dict":
                    optimizer_task2.state_dict(),

                "task2_classes":
                    TASK2_CLASSES,

                "gce_q":
                    TASK2_Q,

                "learning_rate":
                    TASK2_LR,

                "val_accuracy":
                    float(
                        val_accuracy
                    )
            },
            epoch_path
        )


        # ----------------------------------------------------
        # PRINT
        # ----------------------------------------------------

        marker = ""

        if epoch == best_epoch:
            marker = "  <-- BEST"


        print(
            f"Epoch {epoch:02d}/{TASK2_EPOCHS} | "
            f"GCE {train_loss:.4f} | "
            f"Train Acc {train_accuracy:.4f} | "
            f"Val Acc {val_accuracy:.4f} | "
            f"Val BA {val_balanced_accuracy:.4f} | "
            f"Val F1 {val_macro_f1:.4f}"
            f"{marker}"
        )


finally:

    # --------------------------------------------------------
    # ALWAYS REMOVE MASK HOOKS
    # --------------------------------------------------------

    for handle in handles:
        handle.remove()

    print(
        "\nMask hooks removed."
    )


# ============================================================
# 10. SAVE HISTORY
# ============================================================

history_path = (
    CHECKPOINT_DIR
    +
    "/task2_gce_history.json"
)

with open(
    history_path,
    "w"
) as f:

    json.dump(
        history_task2,
        f,
        indent=2
    )


# ============================================================
# 11. SAVE CONFIG
# ============================================================

config_path = (
    CHECKPOINT_DIR
    +
    "/task2_gce_config.txt"
)

with open(
    config_path,
    "w"
) as f:

    f.write(
        "Task 2 GCE Training\n"
    )

    f.write(
        "===================\n"
    )

    f.write(
        f"Classes: {TASK2_CLASSES}\n"
    )

    f.write(
        f"GCE q: {TASK2_Q}\n"
    )

    f.write(
        f"Learning rate: {TASK2_LR}\n"
    )

    f.write(
        f"Epochs: {TASK2_EPOCHS}\n"
    )

    f.write(
        "Batch size: 32\n"
    )

    f.write(
        "Task-1 protected units: 6067\n"
    )

    f.write(
        "Task-2 free units: 9101\n"
    )

    f.write(
        "Functional activation masking: True\n"
    )


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

elapsed = (
    time.time()
    - start_time
)

print("\n" + "=" * 70)
print("STEP 28 COMPLETE")
print("=" * 70)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation accuracy:",
    best_val_accuracy
)

print(
    "Training time:",
    f"{elapsed / 60:.2f} minutes"
)

print(
    "\nBest checkpoint:"
)

print(
    best_path
)

print(
    "\nHistory:"
)

print(
    history_path
)

print(
    "\nConfiguration:"
)

print(
    config_path
)

print("=" * 70)

STEP 28 — TASK 2 GCE TRAINING

Configuration:
GCE q       : 0.7
Learning rate: 0.0001
Epochs      : 20
Batch size  : 32

Reloaded clean Task-2 initialization.
Protected-mask hooks: 17
Epoch 01/20 | GCE 0.5087 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732  <-- BEST
Epoch 02/20 | GCE 0.4968 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732
Epoch 03/20 | GCE 0.5037 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732
Epoch 05/20 | GCE 0.4960 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732
Epoch 06/20 | GCE 0.4984 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732
Epoch 07/20 | GCE 0.5007 | Train Acc 0.6192 | Val Acc 0.5955 | Val BA 0.5000 | Val F1 0.3732

Mask hooks removed.


KeyboardInterrupt: 

In [61]:
# ============================================================
# STEP 28A — TASK 2 GCE TRAINING SANITY CHECK
# ============================================================

import torch
import numpy as np

print("=" * 70)
print("STEP 28A — TASK 2 TRAINING SANITY CHECK")
print("=" * 70)

device = torch.device("cuda")

# ------------------------------------------------------------
# Reload clean Task-2 initialization
# ------------------------------------------------------------

INIT_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2/"
    "task2_initialized_from_task1.pt"
)

init_checkpoint = torch.load(
    INIT_PATH,
    map_location=device,
    weights_only=False
)

model_task2.load_state_dict(
    init_checkpoint["model_state_dict"]
)

model_task2 = model_task2.to(device)
model_task2.train()


# ------------------------------------------------------------
# Activation modules
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task2.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task2,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


# ------------------------------------------------------------
# Protected-mask hooks
# ------------------------------------------------------------

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        return output * (
            1.0 - mask_tensor
        ).view(
            1, -1, 1, 1
        )

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_mask_hook(
                task1_protected_masks[name]
            )
        )
    )


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model_task2.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# GCE
# ------------------------------------------------------------

def gce(
    logits,
    labels,
    q=0.7
):

    probs = torch.softmax(
        logits,
        dim=1
    )

    py = probs[
        torch.arange(
            labels.size(0),
            device=labels.device
        ),
        labels
    ]

    return (
        (1.0 - py.pow(q)) / q
    ).mean()


# ------------------------------------------------------------
# Initial classifier weights
# ------------------------------------------------------------

fc_before = (
    model_task2.fc.weight
    .detach()
    .clone()
)


print(
    "\nInitial FC weight norm:",
    float(
        model_task2.fc.weight
        .detach()
        .norm()
    )
)


# ------------------------------------------------------------
# Run only 3 batches
# ------------------------------------------------------------

for batch_idx, batch in enumerate(
    task2_train_loader
):

    if batch_idx >= 3:
        break

    images = batch["image"].to(
        device
    )

    labels = batch["label"].to(
        device
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    logits = model_task2(
        images
    )

    loss = gce(
        logits,
        labels
    )

    predictions = logits.argmax(
        dim=1
    )

    accuracy = (
        predictions == labels
    ).float().mean()

    print(
        f"\nBatch {batch_idx + 1}"
    )

    print(
        "  Loss:",
        float(loss.item())
    )

    print(
        "  Accuracy:",
        float(accuracy.item())
    )

    print(
        "  Logit mean:",
        float(logits.mean().item())
    )

    print(
        "  Logit std:",
        float(logits.std().item())
    )

    print(
        "  Prediction counts:",
        torch.bincount(
            predictions,
            minlength=2
        ).detach().cpu().tolist()
    )

    loss.backward()

    fc_grad_norm = (
        model_task2.fc.weight.grad
        .detach()
        .norm()
        .item()
    )

    print(
        "  FC gradient norm:",
        fc_grad_norm
    )

    optimizer.step()

    fc_change = (
        model_task2.fc.weight
        .detach()
        -
        fc_before
    ).norm().item()

    print(
        "  FC weight change:",
        fc_change
    )


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

for handle in handles:
    handle.remove()

optimizer.zero_grad(
    set_to_none=True
)

print(
    "\nRemaining hooks:",
    sum(
        len(module._forward_hooks)
        for module in activation_modules.values()
    )
)

print("\n" + "=" * 70)
print("STEP 28A COMPLETE")
print("=" * 70)

STEP 28A — TASK 2 TRAINING SANITY CHECK

Initial FC weight norm: 0.8238596320152283

Batch 1
  Loss: 0.48107028007507324
  Accuracy: 0.6875
  Logit mean: 0.08234618604183197
  Logit std: 0.35695233941078186
  Prediction counts: [32, 0]
  FC gradient norm: 3.6455836296081543
  FC weight change: 0.005803439300507307

Batch 2
  Loss: 0.435947060585022
  Accuracy: 0.6875
  Logit mean: 0.0711049735546112
  Logit std: 0.8265263438224792
  Prediction counts: [32, 0]
  FC gradient norm: 1.2381709814071655
  FC weight change: 0.010874203406274319

Batch 3
  Loss: 0.4955756664276123
  Accuracy: 0.625
  Logit mean: 0.05198848247528076
  Logit std: 1.259202480316162
  Prediction counts: [32, 0]
  FC gradient norm: 0.594413161277771
  FC weight change: 0.014181498438119888

Remaining hooks: 0

STEP 28A COMPLETE


In [62]:
# ============================================================
# STEP 28B — TASK 2 REPRESENTATION DIAGNOSTIC
# ============================================================

import torch
import numpy as np

print("=" * 70)
print("STEP 28B — TASK 2 REPRESENTATION DIAGNOSTIC")
print("=" * 70)

device = torch.device("cuda")

# ------------------------------------------------------------
# Reload CLEAN Task-2 initialization
# ------------------------------------------------------------

INIT_PATH = (
    "/kaggle/working/"
    "biaspruner_checkpoints/"
    "task2/"
    "task2_initialized_from_task1.pt"
)

init_checkpoint = torch.load(
    INIT_PATH,
    map_location=device,
    weights_only=False
)

model_task2.load_state_dict(
    init_checkpoint["model_state_dict"]
)

model_task2 = model_task2.to(device)
model_task2.eval()


# ------------------------------------------------------------
# Activation modules
# ------------------------------------------------------------

activation_modules = {
    "relu": model_task2.relu
}

for layer_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    layer = getattr(
        model_task2,
        layer_name
    )

    for i, block in enumerate(layer):

        activation_modules[
            f"{layer_name}.{i}"
        ] = block


# ------------------------------------------------------------
# Protected-mask hooks
# ------------------------------------------------------------

handles = []


def make_mask_hook(mask_tensor):

    def hook(module, inputs, output):

        return output * (
            1.0 - mask_tensor
        ).view(
            1, -1, 1, 1
        )

    return hook


for name, module in activation_modules.items():

    handles.append(
        module.register_forward_hook(
            make_mask_hook(
                task1_protected_masks[name]
            )
        )
    )


# ------------------------------------------------------------
# Collect representations
# ------------------------------------------------------------

bcc_features = []
akiec_features = []

bcc_logits = []
akiec_logits = []


with torch.no_grad():

    for batch in task2_analysis_loader:

        images = batch["image"].to(
            device
        )

        labels = batch["label"].to(
            device
        )

        logits = model_task2(
            images
        )

        # Extract feature vector BEFORE FC.
        #
        # ResNet forward:
        #
        # backbone
        #    ↓
        # avgpool
        #    ↓
        # 2048-D feature
        #    ↓
        # fc
        #
        features = model_task2.avgpool(
            model_task2.layer4(
                model_task2.layer3(
                    model_task2.layer2(
                        model_task2.layer1(
                            model_task2.maxpool(
                                model_task2.relu(
                                    model_task2.bn1(
                                        model_task2.conv1(
                                            images
                                        )
                                    )
                                )
                            )
                        )
                    )
                )
            )
        )

        features = torch.flatten(
            features,
            1
        )

        # ----------------------------------------------------
        # Store by class
        # ----------------------------------------------------

        bcc_mask = (
            labels == 0
        )

        akiec_mask = (
            labels == 1
        )

        if bcc_mask.any():

            bcc_features.append(
                features[bcc_mask]
                .cpu()
            )

            bcc_logits.append(
                logits[bcc_mask]
                .cpu()
            )

        if akiec_mask.any():

            akiec_features.append(
                features[akiec_mask]
                .cpu()
            )

            akiec_logits.append(
                logits[akiec_mask]
                .cpu()
            )


# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

for handle in handles:
    handle.remove()


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

bcc_features = torch.cat(
    bcc_features,
    dim=0
)

akiec_features = torch.cat(
    akiec_features,
    dim=0
)

bcc_logits = torch.cat(
    bcc_logits,
    dim=0
)

akiec_logits = torch.cat(
    akiec_logits,
    dim=0
)


# ============================================================
# 1. FEATURE STATISTICS
# ============================================================

bcc_mean = (
    bcc_features.mean(
        dim=0
    )
)

akiec_mean = (
    akiec_features.mean(
        dim=0
    )
)

mean_difference = (
    akiec_mean - bcc_mean
)

mean_difference_norm = (
    mean_difference.norm()
    .item()
)


# ============================================================
# 2. FEATURE VARIANCE
# ============================================================

bcc_std = (
    bcc_features.std(
        dim=0
    )
)

akiec_std = (
    akiec_features.std(
        dim=0
    )
)


# ============================================================
# 3. LOGIT STATISTICS
# ============================================================

print("\n" + "-" * 70)
print("REPRESENTATION STATISTICS")
print("-" * 70)

print(
    "BCC samples:",
    len(bcc_features)
)

print(
    "AKIEC samples:",
    len(akiec_features)
)

print(
    "Feature dimension:",
    bcc_features.shape[1]
)

print(
    "\nBCC feature mean norm:",
    float(
        bcc_mean.norm()
    )
)

print(
    "AKIEC feature mean norm:",
    float(
        akiec_mean.norm()
    )
)

print(
    "Mean feature difference norm:",
    mean_difference_norm
)

print(
    "\nBCC average logits:",
    bcc_logits.mean(
        dim=0
    ).numpy()
)

print(
    "AKIEC average logits:",
    akiec_logits.mean(
        dim=0
    ).numpy()
)


# ============================================================
# 4. CLASS SEPARABILITY
# ============================================================

# Cosine similarity between class means

cosine_similarity = torch.nn.functional.cosine_similarity(
    bcc_mean.unsqueeze(0),
    akiec_mean.unsqueeze(0)
).item()

print(
    "\nCosine similarity between class means:",
    cosine_similarity
)


# ============================================================
# 5. FEATURE DIMENSIONS WITH DIFFERENCE
# ============================================================

abs_difference = (
    mean_difference.abs()
)

top_k = min(
    20,
    len(abs_difference)
)

top_values, top_indices = torch.topk(
    abs_difference,
    top_k
)

print(
    "\nTop feature dimensions separating classes:"
)

for rank, (
    idx,
    value
) in enumerate(
    zip(
        top_indices.tolist(),
        top_values.tolist()
    ),
    start=1
):

    print(
        f"{rank:02d}. "
        f"Feature {idx:04d} | "
        f"|mean difference| = {value:.6f}"
    )


# ============================================================
# 6. FINAL
# ============================================================

print("\n" + "=" * 70)
print("STEP 28B COMPLETE")
print("=" * 70)

print(
    "Mean feature difference norm:",
    mean_difference_norm
)

print(
    "Class-mean cosine similarity:",
    cosine_similarity
)

print(
    "Hooks remaining:",
    sum(
        len(module._forward_hooks)
        for module in activation_modules.values()
    )
)

print("=" * 70)

STEP 28B — TASK 2 REPRESENTATION DIAGNOSTIC

----------------------------------------------------------------------
REPRESENTATION STATISTICS
----------------------------------------------------------------------
BCC samples: 361
AKIEC samples: 222
Feature dimension: 2048

BCC feature mean norm: 82468.4453125
AKIEC feature mean norm: 82468.4453125
Mean feature difference norm: 0.005215332843363285

BCC average logits: [163.74197 694.5478 ]
AKIEC average logits: [163.74196 694.5478 ]

Cosine similarity between class means: 0.9999998807907104

Top feature dimensions separating classes:
01. Feature 0031 | |mean difference| = 0.000488
02. Feature 0073 | |mean difference| = 0.000488
03. Feature 0430 | |mean difference| = 0.000488
04. Feature 0552 | |mean difference| = 0.000488
05. Feature 0038 | |mean difference| = 0.000488
06. Feature 0531 | |mean difference| = 0.000488
07. Feature 0326 | |mean difference| = 0.000488
08. Feature 0664 | |mean difference| = 0.000488
09. Feature 0254 | |mean 